In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import json
import pandas as pd
import torch

drive.mount('/content/drive')

PROJECT_ROOT = Path(
    '/content/drive/MyDrive/NLP04_Toxic_Comment_Classification'
)

assert PROJECT_ROOT.is_dir(), "Project folder not found. Check the extracted folder path."


def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as file:
        for chunk in iter(lambda: file.read(1024**2), b''):
            digest.update(chunk)
    return digest.hexdigest()


# Verify the transferred project structure

required_folders = [
    '00_config',
    '01_data',
    '02_notebooks',
    '03_src',
    '04_models',
    '05_references',
    '06_reports'
]

for folder in required_folders:
    print(folder, ":", (PROJECT_ROOT / folder).is_dir())


# Verify experiment records

registry = pd.read_csv(
    PROJECT_ROOT / '00_config/experiment_registry.csv'
)

decisions = pd.read_csv(
    PROJECT_ROOT / '00_config/project_decisions.csv'
)

assert registry['run_id'].is_unique
assert decisions['decision_id'].is_unique

print("\nRegistered experiments:", len(registry))
print("Methodological decisions:", len(decisions))

display(registry[
    ['run_id', 'status', 'validation_macro_ap']
])


# Verify development datasets and evaluation module

distilbert_dir = (
    PROJECT_ROOT / '02_notebooks/04_distilbert/results'
)

selection = json.loads(
    (distilbert_dir / 'configs/distilbert_final_selection.json')
    .read_text()
)

checks = {
    'Training dataset': (
        PROJECT_ROOT / '01_data/processed/train_representative.parquet',
        selection['training_dataset_sha256']
    ),
    'Validation dataset': (
        PROJECT_ROOT / '01_data/processed/validation.parquet',
        selection['validation_dataset_sha256']
    ),
    'Evaluation module': (
        PROJECT_ROOT / '03_src/evaluation_utils.py',
        selection['evaluation_module_sha256']
    )
}

print("\nDataset and evaluation integrity")

for name, (path, expected_hash) in checks.items():
    assert path.is_file(), f"Missing: {path}"
    assert sha256(path) == expected_hash, f"Mismatch: {path}"
    print(name, ": Passed")


# Verify all retained DistilBERT artifacts

for run_id in [
    'T01_BCE_S42',
    'T02_WBCE_S42',
    'T03_FOCAL_S42'
]:

    manifest = pd.read_csv(
        distilbert_dir / run_id / 'tables/artifact_manifest.csv'
    )

    for _, artifact in manifest.iterrows():

        path = PROJECT_ROOT / artifact['Path']

        assert path.is_file(), f"Missing: {path}"

        assert path.stat().st_size == int(
            artifact['Size Bytes']
        ), f"Size mismatch: {path}"

        assert sha256(path) == artifact['SHA256'], (
            f"Hash mismatch: {path}"
        )

    print(f"{run_id}: {len(manifest)} artifacts verified")


# Verify selected classical model artifacts

classical_selection = json.loads(
    (
        PROJECT_ROOT
        / '02_notebooks/03_classical_ml'
        / '06_classical_ml_comparison/results/configs'
        / 'classical_final_selection.json'
    ).read_text()
)

for selected in classical_selection['selected_models'].values():

    for artifact in ['model', 'config', 'threshold', 'predictions']:

        path = PROJECT_ROOT / selected[f'{artifact}_path']

        assert path.is_file()
        assert sha256(path) == selected[f'{artifact}_sha256']

    print(selected['run_id'], ": Verified")


# Check existing RoBERTa progress

roberta_dir = PROJECT_ROOT / '02_notebooks/05_roberta'

print("\nRoBERTa notebook exists:", roberta_dir.exists())

if roberta_dir.exists():

    for path in roberta_dir.rglob('*.json'):
        print(path.relative_to(roberta_dir))


# Check GPU availability

print("\nGPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# Check transferred project storage

size = sum(
    path.stat().st_size
    for path in PROJECT_ROOT.rglob('*')
    if path.is_file()
) / 1024**3

print("\nProject storage (GiB):", round(size, 4))

print("\nSelected research artifacts verified successfully.")

In [ ]:
# Verify remaining assets after Drive migration

models_dir = PROJECT_ROOT / '04_models'
notebooks_dir = PROJECT_ROOT / '02_notebooks'

print("Shared DistilBERT initialization:")

initialization_dir = (
    models_dir / 'distilbert/shared_initialization_s42'
)

print("Directory exists:", initialization_dir.is_dir())

print("\nRoBERTa notebook files:")

roberta_dir = notebooks_dir / '05_roberta'

for path in sorted(roberta_dir.rglob('*')):
    if path.is_file():
        print(path.relative_to(roberta_dir))

print("\nRoBERTa model files:")

roberta_models = models_dir / 'roberta'

if roberta_models.exists():
    for path in sorted(roberta_models.rglob('*')):
        if path.is_file():
            print(path.relative_to(roberta_models))
else:
    print("No saved RoBERTa models found.")


# Verify remaining prepared datasets

manifest = pd.read_csv(
    PROJECT_ROOT / '01_data/splits/prepared_dataset_manifest.csv'
)

print("\nPrepared dataset integrity:")

for _, row in manifest.iterrows():

    path = PROJECT_ROOT / row['Dataset Path']

    assert path.is_file(), f"Missing: {path}"

    assert sha256(path) == row['SHA256'], (
        f"Fingerprint mismatch: {path}"
    )

    print(row['Partition'], ": Passed")


# Verify all registered model paths

print("\nRegistered model availability:")

for _, row in registry.iterrows():

    path = PROJECT_ROOT / row['model_path']

    assert path.exists(), f"Missing model: {path}"

    print(row['run_id'], ": Available")

print("\nNo files modified.")
print("No training performed.")
print("No figures generated.")

In [ ]:
# Import libraries and mount Google Drive

from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

import os
import sys
import json
import time
import random
import hashlib
import math
import gc
import shutil
import tempfile
import platform
import psutil

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import tokenizers
import pyarrow as pa
import pyarrow.parquet as pq

import matplotlib.pyplot as plt
import seaborn as sns

drive.mount('/content/drive')


# Initialize project paths

PROJECT_ROOT = Path(
    '/content/drive/MyDrive/NLP04_Toxic_Comment_Classification'
)

assert PROJECT_ROOT.is_dir(), "Project directory not found."

CONFIG_DIR = PROJECT_ROOT / '00_config'
PROCESSED_DIR = PROJECT_ROOT / '01_data/processed'
SPLITS_DIR = PROJECT_ROOT / '01_data/splits'
SRC_DIR = PROJECT_ROOT / '03_src'

NOTEBOOK_DIR = PROJECT_ROOT / '02_notebooks/05_roberta'
RESULTS_DIR = NOTEBOOK_DIR / 'results'

TABLES_DIR = RESULTS_DIR / 'tables'
FIGURES_DIR = RESULTS_DIR / 'figures'
LOGS_DIR = RESULTS_DIR / 'logs'
RUN_CONFIGS_DIR = RESULTS_DIR / 'configs'
PREDICTIONS_DIR = RESULTS_DIR / 'predictions'

MODELS_DIR = PROJECT_ROOT / '04_models/roberta'
TOKEN_CACHE_DIR = PROJECT_ROOT / '01_data/tokenized/roberta'

REGISTRY_PATH = CONFIG_DIR / 'experiment_registry.csv'
DECISION_LOG_PATH = CONFIG_DIR / 'project_decisions.csv'

for folder in [
    TABLES_DIR,
    FIGURES_DIR,
    LOGS_DIR,
    RUN_CONFIGS_DIR,
    PREDICTIONS_DIR,
    MODELS_DIR,
    TOKEN_CACHE_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project directories initialized.")


# Define reusable utility functions

def read_json(path):
    return json.loads(Path(path).read_text())


def save_json_atomic(path, content):

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary = path.with_suffix(path.suffix + '.tmp')

    temporary.write_text(
        json.dumps(content, indent=4, allow_nan=False)
    )

    temporary.replace(path)


def save_csv_atomic(path, dataframe):

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary = path.with_suffix(path.suffix + '.tmp')

    dataframe.to_csv(temporary, index=False)

    temporary.replace(path)


def calculate_sha256(path):

    digest = hashlib.sha256()

    with open(path, 'rb') as file:
        for chunk in iter(lambda: file.read(1024**2), b''):
            digest.update(chunk)

    return digest.hexdigest()


def set_random_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# Restore the original project configuration

config = read_json(CONFIG_DIR / 'project_config.json')

LABELS = config['labels']
NUM_LABELS = len(LABELS)
SEED = config['seed']

MODEL_NAME = 'RoBERTa'

transformer_config = config['transformer']
model_config = transformer_config['models'][MODEL_NAME]

MODEL_CHECKPOINT = model_config['checkpoint']
MODEL_REVISION = model_config['revision']
MAX_LENGTH = transformer_config['initial_max_length']

TRAINING_SAMPLES = 59997
VALIDATION_SAMPLES = 31911

assert LABELS == [
    'toxic',
    'severe_toxic',
    'obscene',
    'threat',
    'insult',
    'identity_hate'
]

assert NUM_LABELS == 6
assert SEED == 42
assert MAX_LENGTH == 256

assert transformer_config['padding_strategy'] == 'dynamic'
assert transformer_config['truncation'] is True
assert transformer_config['truncation_side'] == 'right'

set_random_seed(SEED)

print("Original project configuration verified.")


# Verify GPU functionality

assert torch.cuda.is_available(), (
    "GPU unavailable. Enable the GPU runtime."
)

DEVICE = torch.device('cuda')
CUDA_AVAILABLE = True

test_tensor = torch.ones((2, 2), device=DEVICE)

assert torch.allclose(
    test_tensor @ test_tensor,
    torch.full((2, 2), 2.0, device=DEVICE)
)

del test_tensor

torch.cuda.empty_cache()

free_memory, total_memory = torch.cuda.mem_get_info()

gpu_info = {
    'GPU': torch.cuda.get_device_name(0),
    'CUDA Version': torch.version.cuda,
    'Compute Capability': list(
        torch.cuda.get_device_capability(0)
    ),
    'Total GPU Memory (GiB)': total_memory / 1024**3,
    'Free GPU Memory (GiB)': free_memory / 1024**3,
    'Available RAM (GiB)':
        psutil.virtual_memory().available / 1024**3,
    'Local Disk Free (GiB)':
        psutil.disk_usage('/content').free / 1024**3,
    'CUDA Operation Verified': True
}

print("\nRuntime environment")

display(pd.DataFrame(
    list(gpu_info.items()),
    columns=['Property', 'Value']
))


# Record library versions

import sklearn
import scipy

environment_versions = {
    'python': platform.python_version(),
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'scipy': scipy.__version__,
    'scikit_learn': sklearn.__version__,
    'torch': str(torch.__version__),
    'transformers': transformers.__version__,
    'tokenizers': tokenizers.__version__,
    'pyarrow': pa.__version__
}

print("\nEnvironment versions")

display(pd.DataFrame(
    list(environment_versions.items()),
    columns=['Package', 'Version']
))


# Restore completed experiment records

registry = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
    keep_default_na=False
)

decisions = pd.read_csv(
    DECISION_LOG_PATH,
    dtype=str,
    keep_default_na=False
)

assert registry['run_id'].is_unique
assert registry['status'].eq('Completed').all()
assert decisions['decision_id'].is_unique

registry_original_hash = calculate_sha256(REGISTRY_PATH)
decision_log_original_hash = calculate_sha256(DECISION_LOG_PATH)

DISTILBERT_RUN_IDS = [
    'T01_BCE_S42',
    'T02_WBCE_S42',
    'T03_FOCAL_S42'
]

ROBERTA_RUN_IDS = [
    'T04_BCE_S42',
    'T05_WBCE_S42',
    'T06_FOCAL_S42'
]

previous_registry = registry.loc[
    ~registry['run_id'].isin(ROBERTA_RUN_IDS)
].copy().reset_index(drop=True)

assert len(previous_registry) == 14

assert set(DISTILBERT_RUN_IDS).issubset(
    previous_registry['run_id']
)

assert len(decisions) >= 60

print("\nPrevious experiment records verified.")


# Restore DistilBERT completion and selection records

DISTILBERT_RESULTS_DIR = (
    PROJECT_ROOT / '02_notebooks/04_distilbert/results'
)

distilbert_completion = read_json(
    DISTILBERT_RESULTS_DIR
    / 'logs/distilbert_completion_summary.json'
)

distilbert_selection = read_json(
    DISTILBERT_RESULTS_DIR
    / 'configs/distilbert_final_selection.json'
)

assert distilbert_completion['status'] == 'Completed'

assert distilbert_completion['retained_artifacts_verified']

assert distilbert_selection['controlled_settings_verified']

assert distilbert_selection['labels'] == LABELS

assert not distilbert_selection['official_test_used']

assert not distilbert_selection['final_project_model_frozen']


# Verify the transferred DistilBERT completion inventory

for artifact in distilbert_completion['artifact_inventory']:

    path = PROJECT_ROOT / artifact['path']

    assert path.is_file(), f"Missing: {path}"

    assert path.stat().st_size == artifact['size_bytes'], (
        f"Size mismatch: {path}"
    )

    assert calculate_sha256(path) == artifact['sha256'], (
        f"Fingerprint mismatch: {path}"
    )

print("DistilBERT completion inventory verified.")


# Restore selected classical baselines

CLASSICAL_SELECTION_PATH = (
    PROJECT_ROOT
    / '02_notebooks/03_classical_ml'
    / '06_classical_ml_comparison/results/configs'
    / 'classical_final_selection.json'
)

classical_selection = read_json(
    CLASSICAL_SELECTION_PATH
)

assert classical_selection['labels'] == LABELS

assert not classical_selection['official_test_used']

EXPECTED_TRAINING_HASH = (
    classical_selection['training_dataset_sha256']
)

EXPECTED_VALIDATION_HASH = (
    classical_selection['validation_dataset_sha256']
)

assert distilbert_selection['training_dataset_sha256'] == (
    EXPECTED_TRAINING_HASH
)

assert distilbert_selection['validation_dataset_sha256'] == (
    EXPECTED_VALIDATION_HASH
)


# Verify the unchanged development datasets

TRAIN_PATH = PROCESSED_DIR / 'train_representative.parquet'
VALIDATION_PATH = PROCESSED_DIR / 'validation.parquet'

assert calculate_sha256(TRAIN_PATH) == (
    EXPECTED_TRAINING_HASH
)

assert calculate_sha256(VALIDATION_PATH) == (
    EXPECTED_VALIDATION_HASH
)

assert pq.ParquetFile(TRAIN_PATH).metadata.num_rows == (
    TRAINING_SAMPLES
)

assert pq.ParquetFile(VALIDATION_PATH).metadata.num_rows == (
    VALIDATION_SAMPLES
)

print("Development dataset fingerprints verified.")


# Restore shared evaluation functions

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import evaluation_utils

from evaluation_utils import (
    evaluate_multilabel,
    optimize_thresholds,
    apply_thresholds
)

evaluation_hash = calculate_sha256(
    SRC_DIR / 'evaluation_utils.py'
)

assert evaluation_hash == (
    classical_selection['evaluation_module_sha256']
)

assert evaluation_hash == (
    distilbert_selection['evaluation_module_sha256']
)

assert Path(evaluation_utils.__file__).resolve() == (
    SRC_DIR / 'evaluation_utils.py'
).resolve()

print("Shared evaluation module verified.")


# Restore the previous training policy

distilbert_pipeline = read_json(
    DISTILBERT_RESULTS_DIR
    / 'configs/distilbert_training_pipeline.json'
)

assert distilbert_pipeline[
    'training_configuration_finalized'
]

training_policy = (
    distilbert_pipeline['training_policy'].copy()
)

training_policy.update({
    'training_configuration_finalized': False,
    'full_model_smoke_test_completed': False,
    'checkpoint_resume_verified': False
})

assert training_policy['effective_batch_size'] == 32
assert training_policy['maximum_epochs'] == 3
assert training_policy['learning_rate'] == 2e-5
assert training_policy['weight_decay'] == 0.01
assert training_policy['warmup_ratio'] == 0.10
assert training_policy['early_stopping_patience'] == 2
assert training_policy['mixed_precision'] == 'fp16'

print("\nProvisional RoBERTa training configuration")

display(pd.DataFrame(
    list(training_policy.items()),
    columns=['Parameter', 'Value']
))


# Restore reference model performance

reference_records = []

for selected in classical_selection[
    'selected_models'
].values():

    reference_records.append({
        'Run ID': selected['run_id'],
        'Model': selected['model'],
        'Macro AP': selected['validation_macro_ap'],
        'Optimized Macro F1':
            selected['validation_optimized_macro_f1']
    })

for run_id in DISTILBERT_RUN_IDS:

    summary = read_json(
        DISTILBERT_RESULTS_DIR
        / run_id / 'logs/run_summary.json'
    )

    reference_records.append({
        'Run ID': run_id,
        'Model': 'DistilBERT',
        'Macro AP': summary['validation_macro_ap'],
        'Optimized Macro F1':
            summary['validation_optimized_macro_f1']
    })

reference_models_df = pd.DataFrame(
    reference_records
).sort_values(
    'Macro AP',
    ascending=False
).reset_index(drop=True)

print("\nReference model performance")

display(reference_models_df.round(6))


# Define the controlled RoBERTa experiments

experiment_plan_df = pd.DataFrame({
    'Run ID': ROBERTA_RUN_IDS,
    'Model': MODEL_NAME,
    'Loss': ['BCE', 'WBCE', 'FOCAL'],
    'Training Samples': TRAINING_SAMPLES,
    'Validation Samples': VALIDATION_SAMPLES,
    'Maximum Length': MAX_LENGTH,
    'Seed': SEED
})

experiment_plan_df['Status'] = [
    'Completed'
    if run_id in registry['run_id'].values
    else 'Resume Available'
    if (MODELS_DIR / run_id / 'latest_checkpoint.pt').is_file()
    else 'Planned'
    for run_id in ROBERTA_RUN_IDS
]

print("\nRoBERTa experiment plan")

display(experiment_plan_df)


# Define the storage retention policy

storage_policy = {
    'retain_shared_initialization': True,
    'retain_selected_best_model': True,
    'retain_superseded_models': False,
    'retain_active_resume_checkpoint': True,
    'retain_completed_optimizer_checkpoint': False,
    'retain_validation_predictions': True,
    'retain_thresholds_and_metrics': True,
    'retain_epoch_history': True,
    'figure_format': 'png',
    'figure_dpi': 300,
    'avoid_duplicate_figures': True,
    'verify_before_pruning': True,
    'account_quota_requires_manual_check': True
}

print("\nStorage retention policy")

display(pd.DataFrame(
    list(storage_policy.items()),
    columns=['Parameter', 'Value']
))


# Measure active project storage

project_size = sum(
    path.stat().st_size
    for path in PROJECT_ROOT.rglob('*')
    if path.is_file()
) / 1024**3

print(
    "\nActive project storage (GiB):",
    round(project_size, 4)
)


# Configure visualization settings

sns.set_theme(style='whitegrid')

LABEL_COLORS = {
    'toxic': '#4C78A8',
    'severe_toxic': '#F58518',
    'obscene': '#54A24B',
    'threat': '#E45756',
    'insult': '#B279A2',
    'identity_hate': '#72B7B2'
}

LOSS_COLORS = {
    'BCE': '#4C78A8',
    'WBCE': '#F58518',
    'FOCAL': '#54A24B'
}


# Save consolidated initialization metadata

INITIALIZATION_PATH = (
    LOGS_DIR / 'roberta_initialization.json'
)

initialization_summary = {
    'notebook': '05_roberta',
    'model': MODEL_NAME,
    'checkpoint': MODEL_CHECKPOINT,
    'revision': MODEL_REVISION,
    'labels': LABELS,
    'seed': SEED,
    'maximum_length': MAX_LENGTH,
    'training_samples': TRAINING_SAMPLES,
    'validation_samples': VALIDATION_SAMPLES,
    'training_dataset_sha256': EXPECTED_TRAINING_HASH,
    'validation_dataset_sha256': EXPECTED_VALIDATION_HASH,
    'evaluation_module_sha256': evaluation_hash,
    'previous_completed_experiments': len(previous_registry),
    'distilbert_candidate':
        distilbert_selection['selected_run_id'],
    'training_policy': training_policy.copy(),
    'storage_policy': storage_policy,
    'environment_versions': environment_versions,
    'gpu_environment': gpu_info,
    'active_project_storage_gib': project_size,
    'migration_development_fingerprints_verified': True,
    'distilbert_completion_inventory_verified': True,
    'official_test_used': False,
    'model_training_performed': False
}

if not INITIALIZATION_PATH.exists():

    save_json_atomic(
        INITIALIZATION_PATH,
        initialization_summary
    )

else:

    saved_initialization = read_json(
        INITIALIZATION_PATH
    )

    for key in [
        'checkpoint',
        'revision',
        'labels',
        'seed',
        'maximum_length',
        'training_dataset_sha256',
        'validation_dataset_sha256',
        'evaluation_module_sha256'
    ]:

        assert saved_initialization[key] == (
            initialization_summary[key]
        ), f"Initialization mismatch: {key}"

    print("\nExisting initialization configuration verified.")


# Verify the original research records remain unchanged

assert calculate_sha256(REGISTRY_PATH) == (
    registry_original_hash
)

assert calculate_sha256(DECISION_LOG_PATH) == (
    decision_log_original_hash
)


# Display final initialization summary

print("\nRoBERTa initialization summary")

print("Model:", MODEL_NAME)
print("Checkpoint:", MODEL_CHECKPOINT)
print("Pinned revision:", MODEL_REVISION)
print("Configured vocabulary size:", model_config['vocabulary_size'])
print("Maximum sequence length:", MAX_LENGTH)
print("Seed:", SEED)

print("\nTraining samples:", TRAINING_SAMPLES)
print("Validation samples:", VALIDATION_SAMPLES)

print("\nPrevious completed experiments:", len(previous_registry))
print("Total registered experiments:", len(registry))
print("Methodological decisions:", len(decisions))
print("Planned RoBERTa configurations:", len(ROBERTA_RUN_IDS))

print("\nPrevious research records: Preserved")
print("DistilBERT completion inventory: Verified")
print("Development datasets: Verified")
print("Shared evaluation module: Verified")
print("Storage-efficient retention policy: Defined")
print("Initialization metadata: Saved")

print("\nNew model weights downloaded: No")
print("Tokenization performed: No")
print("Model training performed: No")
print("Additional figures generated: 0")
print("Official test dataset: Not loaded")

gc.collect()
torch.cuda.empty_cache()

print("\nCell 1 completed successfully.")

In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

# Load the original development datasets

train_df = pd.read_parquet(TRAIN_PATH)
val_df = pd.read_parquet(VALIDATION_PATH)

assert len(train_df) == TRAINING_SAMPLES
assert len(val_df) == VALIDATION_SAMPLES

required_columns = [
    'id', 'comment_text', 'group_id', *LABELS
]

for partition, dataframe in [
    ('train_representative', train_df),
    ('validation', val_df)
]:

    assert set(required_columns).issubset(dataframe.columns)
    assert not dataframe[required_columns].isna().any().any()
    assert dataframe['id'].is_unique
    assert dataframe[LABELS].isin([0, 1]).all().all()
    assert dataframe['comment_text'].map(
        lambda text: isinstance(text, str)
    ).all()

    assert dataframe['comment_text'].str.strip().ne('').all()

    manifest = pd.read_csv(
        SPLITS_DIR / f'{partition}_ids.csv',
        dtype={'id': str, 'group_id': str}
    )

    np.testing.assert_array_equal(
        dataframe['id'].astype(str).to_numpy(),
        manifest['id'].to_numpy()
    )

    np.testing.assert_array_equal(
        dataframe['group_id'].astype(str).to_numpy(),
        manifest['group_id'].to_numpy()
    )

    np.testing.assert_array_equal(
        manifest['row_index'].to_numpy(),
        np.arange(len(dataframe))
    )

    print(f"{partition}: Integrity and alignment verified.")


# Verify train-validation separation

assert not (
    set(train_df['id']) & set(val_df['id'])
)

assert not (
    set(train_df['group_id']) & set(val_df['group_id'])
)

def normalize_for_verification(text):
    return ' '.join(text.lower().split())

train_normalized = set(
    train_df['comment_text'].map(normalize_for_verification)
)

val_normalized = set(
    val_df['comment_text'].map(normalize_for_verification)
)

assert not (train_normalized & val_normalized)

del train_normalized, val_normalized

print("Train-validation leakage checks: Passed")


# Prepare texts and binary targets

train_texts = train_df['comment_text'].tolist()
val_texts = val_df['comment_text'].tolist()

train_comment_ids = train_df['id'].astype(str).to_numpy()
val_comment_ids = val_df['id'].astype(str).to_numpy()

y_train = train_df[LABELS].to_numpy(
    dtype=np.int8, copy=True
)

y_val = val_df[LABELS].to_numpy(
    dtype=np.int8, copy=True
)

assert y_train.shape == (TRAINING_SAMPLES, NUM_LABELS)
assert y_val.shape == (VALIDATION_SAMPLES, NUM_LABELS)

train_positives = y_train.sum(axis=0).astype(np.int64)
train_negatives = TRAINING_SAMPLES - train_positives

val_positives = y_val.sum(axis=0).astype(np.int64)
val_negatives = VALIDATION_SAMPLES - val_positives


# Verify label counts against the completed DistilBERT stage

distilbert_data_summary = read_json(
    DISTILBERT_RESULTS_DIR
    / 'logs/distilbert_dataset_preparation.json'
)

np.testing.assert_array_equal(
    train_positives,
    distilbert_data_summary['training_positives']
)

np.testing.assert_array_equal(
    val_positives,
    distilbert_data_summary['validation_positives']
)

label_support_df = pd.DataFrame({
    'Label': LABELS,
    'Training Positives': train_positives,
    'Validation Positives': val_positives
})

print("\nLabel support verification")

display(label_support_df)


# Load the pinned RoBERTa tokenizer

print("\nLoading RoBERTa tokenizer...", flush=True)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_CHECKPOINT,
    revision=MODEL_REVISION,
    use_fast=True
)

assert tokenizer.is_fast
assert tokenizer.vocab_size == model_config['vocabulary_size']

tokenizer.truncation_side = 'right'
tokenizer.padding_side = 'right'

assert tokenizer.pad_token_id is not None
assert tokenizer.bos_token_id is not None
assert tokenizer.eos_token_id is not None

assert tokenizer.num_special_tokens_to_add(
    pair=False
) == model_config['special_tokens']

assert MAX_LENGTH <= model_config['maximum_supported_length']

PAD_MULTIPLE = training_policy['pad_to_multiple_of']

print("\nTokenizer configuration")

display(pd.DataFrame([
    ('Tokenizer', tokenizer.__class__.__name__),
    ('Fast tokenizer', tokenizer.is_fast),
    ('Vocabulary size', tokenizer.vocab_size),
    ('Maximum length', MAX_LENGTH),
    ('Padding token ID', tokenizer.pad_token_id),
    ('BOS token ID', tokenizer.bos_token_id),
    ('EOS token ID', tokenizer.eos_token_id),
    ('Padding multiple', PAD_MULTIPLE)
], columns=['Parameter', 'Value']))


# Define the multilabel dataset

class ToxicCommentDataset(Dataset):

    def __init__(self, input_ids, labels):

        self.input_ids = input_ids
        self.labels = np.asarray(labels, dtype=np.float32)

        assert self.labels.shape == (
            len(self.input_ids), NUM_LABELS
        )

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, index):

        ids = self.input_ids[index]

        return {
            'input_ids': ids,
            'attention_mask': [1] * len(ids),
            'labels': self.labels[index],
            'row_index': index
        }


# Tokenize and cache each partition

def prepare_tokenized_partition(
    partition,
    texts,
    comment_ids,
    labels,
    dataset_hash
):

    specification = {
        'schema_version': 1,
        'partition': partition,
        'dataset_sha256': dataset_hash,
        'checkpoint': MODEL_CHECKPOINT,
        'revision': MODEL_REVISION,
        'maximum_length': MAX_LENGTH,
        'truncation': True,
        'truncation_side': 'right',
        'padding': False,
        'add_special_tokens': True,
        'rows': len(texts),
        'labels': LABELS,
        'transformers_version': transformers.__version__,
        'tokenizers_version': tokenizers.__version__,
        'pyarrow_version': pa.__version__
    }

    cache_key = hashlib.sha256(
        json.dumps(
            specification, sort_keys=True
        ).encode('utf-8')
    ).hexdigest()

    cache_path = (
        TOKEN_CACHE_DIR
        / f'{partition}_{cache_key[:16]}.parquet'
    )

    manifest_path = cache_path.with_suffix('.json')

    cache_reused = False
    tokenization_seconds = 0.0

    print(f"\nPreparing {partition}...", flush=True)

    if cache_path.exists() or manifest_path.exists():

        assert cache_path.is_file() and manifest_path.is_file(), (
            "Incomplete token cache. Inspection required."
        )

        manifest = read_json(manifest_path)

        assert manifest['specification'] == specification

        assert calculate_sha256(cache_path) == (
            manifest['cache_sha256']
        )

        cache_reused = True

        print("Verified existing cache reused.")

    else:

        start_time = time.perf_counter()

        input_ids = []
        batch_size = 1024

        for start in range(0, len(texts), batch_size):

            encoded = tokenizer(
                texts[start:start + batch_size],
                add_special_tokens=True,
                truncation=True,
                max_length=MAX_LENGTH,
                padding=False,
                return_attention_mask=False,
                return_token_type_ids=False
            )

            input_ids.extend(encoded['input_ids'])

            completed = min(
                start + batch_size, len(texts)
            )

            if (
                (start // batch_size + 1) % 10 == 0
                or completed == len(texts)
            ):

                print(
                    f"Tokenized {completed:,}/{len(texts):,}",
                    flush=True
                )

        tokenization_seconds = (
            time.perf_counter() - start_time
        )

        assert len(input_ids) == len(texts)

        table = pa.table({
            'id': pa.array(
                comment_ids.tolist(),
                type=pa.string()
            ),
            'row_index': pa.array(
                np.arange(len(texts)),
                type=pa.int64()
            ),
            'input_ids': pa.array(
                input_ids,
                type=pa.list_(pa.int32())
            )
        })

        temporary_path = cache_path.with_suffix(
            '.parquet.tmp'
        )

        pq.write_table(
            table,
            temporary_path,
            compression='zstd'
        )

        temporary_path.replace(cache_path)

        manifest = {
            'specification': specification,
            'cache_key': cache_key,
            'cache_sha256': calculate_sha256(cache_path),
            'rows': len(texts),
            'tokenization_seconds': tokenization_seconds
        }

        save_json_atomic(manifest_path, manifest)

        del table, input_ids

    # Verify saved token sequences and comment alignment

    table = pq.read_table(cache_path)

    assert table.num_rows == len(texts)

    np.testing.assert_array_equal(
        np.asarray(table['id'].to_pylist(), dtype=str),
        comment_ids.astype(str)
    )

    np.testing.assert_array_equal(
        table['row_index'].to_numpy(),
        np.arange(len(texts))
    )

    input_ids = table['input_ids'].to_pylist()

    del table

    lengths = np.asarray(
        [len(ids) for ids in input_ids],
        dtype=np.int32
    )

    assert lengths.min() >= 2
    assert lengths.max() <= MAX_LENGTH

    assert all(
        ids[0] == tokenizer.bos_token_id
        and ids[-1] == tokenizer.eos_token_id
        for ids in input_ids
    )

    assert all(
        min(ids) >= 0 and max(ids) < len(tokenizer)
        for ids in input_ids
    )

    # Independently verify selected cached comments

    rng = np.random.default_rng(SEED)

    indices = np.unique(np.concatenate([
        [
            0,
            len(texts) - 1,
            int(np.argmin(lengths)),
            int(np.argmax(lengths))
        ],
        rng.choice(
            len(texts), size=32, replace=False
        )
    ]))

    expected_tokens = tokenizer(
        [texts[int(index)] for index in indices],
        add_special_tokens=True,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
        return_attention_mask=False,
        return_token_type_ids=False
    )['input_ids']

    for index, expected in zip(indices, expected_tokens):

        assert input_ids[int(index)] == expected

    dataset = ToxicCommentDataset(input_ids, labels)

    np.testing.assert_array_equal(
        dataset.labels,
        labels
    )

    record = {
        'Partition': partition,
        'Rows': len(dataset),
        'Cache Path': str(
            cache_path.relative_to(PROJECT_ROOT)
        ),
        'Manifest Path': str(
            manifest_path.relative_to(PROJECT_ROOT)
        ),
        'Cache SHA256': manifest['cache_sha256'],
        'Cache Size (MiB)':
            cache_path.stat().st_size / 1024**2,
        'Cache Reused': cache_reused,
        'Tokenization Time (s)': tokenization_seconds,
        'Token Verification Samples': len(indices),
        'Verification': 'Passed'
    }

    print(f"{partition}: Token cache verified.")

    return dataset, lengths, record


train_dataset, train_token_lengths, train_cache_record = (
    prepare_tokenized_partition(
        'train_representative',
        train_texts,
        train_comment_ids,
        y_train,
        EXPECTED_TRAINING_HASH
    )
)

val_dataset, val_token_lengths, val_cache_record = (
    prepare_tokenized_partition(
        'validation',
        val_texts,
        val_comment_ids,
        y_val,
        EXPECTED_VALIDATION_HASH
    )
)


# Define dynamic batch padding

def dynamic_collate_fn(samples):

    longest = max(
        len(sample['input_ids'])
        for sample in samples
    )

    padded_length = (
        math.ceil(longest / PAD_MULTIPLE)
        * PAD_MULTIPLE
    )

    assert padded_length <= MAX_LENGTH

    input_ids = np.full(
        (len(samples), padded_length),
        tokenizer.pad_token_id,
        dtype=np.int64
    )

    attention_mask = np.zeros(
        (len(samples), padded_length),
        dtype=np.int64
    )

    for index, sample in enumerate(samples):

        length = len(sample['input_ids'])

        input_ids[index, :length] = sample['input_ids']
        attention_mask[index, :length] = 1

    return {
        'input_ids': torch.from_numpy(input_ids),
        'attention_mask': torch.from_numpy(attention_mask),
        'labels': torch.from_numpy(
            np.stack([
                sample['labels'] for sample in samples
            ]).astype(np.float32)
        ),
        'row_index': torch.tensor([
            sample['row_index'] for sample in samples
        ], dtype=torch.long)
    }


# Verify short, long and mixed sequence batches

padding_records = []

for partition, dataset, lengths in [
    ('Training', train_dataset, train_token_lengths),
    ('Validation', val_dataset, val_token_lengths)
]:

    ordered = np.argsort(lengths)

    test_batches = {
        'Short': ordered[:8],
        'Long': ordered[-8:],
        'Mixed': ordered[
            np.linspace(
                0, len(ordered) - 1, 8, dtype=int
            )
        ]
    }

    for name, indices in test_batches.items():

        samples = [
            dataset[int(index)] for index in indices
        ]

        batch = dynamic_collate_fn(samples)

        ids = batch['input_ids']
        mask = batch['attention_mask']

        assert ids.shape == mask.shape
        assert ids.dtype == torch.int64
        assert mask.dtype == torch.int64

        assert ids.shape[1] % PAD_MULTIPLE == 0
        assert ids.shape[1] <= MAX_LENGTH

        assert batch['labels'].shape == (8, NUM_LABELS)
        assert batch['labels'].dtype == torch.float32

        assert torch.all(
            ids[mask == 0] == tokenizer.pad_token_id
        )

        for index, sample in enumerate(samples):

            length = len(sample['input_ids'])

            assert mask[index, :length].eq(1).all()
            assert mask[index, length:].eq(0).all()

            assert ids[index, :length].tolist() == (
                sample['input_ids']
            )

            np.testing.assert_array_equal(
                batch['labels'][index].numpy(),
                sample['labels']
            )

            assert batch['row_index'][index].item() == (
                sample['row_index']
            )

        padding_records.append({
            'Partition': partition,
            'Batch Type': name,
            'Longest Sequence': max(
                len(sample['input_ids'])
                for sample in samples
            ),
            'Padded Length': ids.shape[1],
            'Verification': 'Passed'
        })

padding_verification_df = pd.DataFrame(padding_records)

assert len(padding_verification_df) == 6

print("\nDynamic padding verification")

display(padding_verification_df)


# Summarize tokenized sequence lengths

length_records = []

for partition, lengths in [
    ('Training', train_token_lengths),
    ('Validation', val_token_lengths)
]:

    length_records.append({
        'Partition': partition,
        'Comments': len(lengths),
        'Minimum Tokens': int(lengths.min()),
        'Mean Tokens': float(lengths.mean()),
        'Median Tokens': float(np.median(lengths)),
        'Maximum Tokens': int(lengths.max()),
        'Total Tokens': int(lengths.sum()),
        'At Maximum Length (%)': float(
            (lengths == MAX_LENGTH).mean() * 100
        )
    })

length_summary_df = pd.DataFrame(length_records)

print("\nTokenized sequence length summary")

display(length_summary_df.round(6))

token_cache_manifest_df = pd.DataFrame([
    train_cache_record,
    val_cache_record
])

print("\nToken cache verification")

display(token_cache_manifest_df)


# Verify original research records remain unchanged

assert calculate_sha256(TRAIN_PATH) == (
    EXPECTED_TRAINING_HASH
)

assert calculate_sha256(VALIDATION_PATH) == (
    EXPECTED_VALIDATION_HASH
)

assert calculate_sha256(REGISTRY_PATH) == (
    registry_original_hash
)

assert calculate_sha256(DECISION_LOG_PATH) == (
    decision_log_original_hash
)


# Save one consolidated tokenization record

tokenization_summary = {
    'model': MODEL_NAME,
    'checkpoint': MODEL_CHECKPOINT,
    'revision': MODEL_REVISION,
    'labels': LABELS,
    'vocabulary_size': tokenizer.vocab_size,
    'maximum_length': MAX_LENGTH,
    'padding_strategy': 'dynamic',
    'padding_multiple': PAD_MULTIPLE,
    'training_samples': len(train_dataset),
    'validation_samples': len(val_dataset),
    'training_positives': train_positives.tolist(),
    'validation_positives': val_positives.tolist(),
    'training_dataset_sha256': EXPECTED_TRAINING_HASH,
    'validation_dataset_sha256': EXPECTED_VALIDATION_HASH,
    'training_cache': train_cache_record,
    'validation_cache': val_cache_record,
    'sequence_length_summary': length_records,
    'dynamic_padding_verification': padding_records,
    'dataset_integrity_verified': True,
    'train_validation_overlap': 0,
    'distilbert_label_support_matched': True,
    'raw_comment_text_preserved': True,
    'permanent_padding': False,
    'official_test_used': False,
    'model_training_performed': False
}

save_json_atomic(
    LOGS_DIR / 'roberta_tokenization_summary.json',
    tokenization_summary
)


# Display completion summary

print("\nRoBERTa tokenization summary")

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Vocabulary size:", tokenizer.vocab_size)
print("Maximum sequence length:", MAX_LENGTH)

print(
    "Total cache size (MiB):",
    round(
        train_cache_record['Cache Size (MiB)']
        + val_cache_record['Cache Size (MiB)'],
        4
    )
)

print("\nDataset alignment: Verified")
print("DistilBERT label support: Matched")
print("Train-validation overlap: 0")
print("Token cache integrity: Verified")
print("Dynamic padding checks: 6/6 passed")
print("Permanent padding: No")

print("\nOriginal datasets: Unchanged")
print("Experiment registry: Unchanged")
print("Decision log: Unchanged")
print("Additional figures generated: 0")
print("Model training performed: No")
print("Official test dataset: Not loaded")

gc.collect()

print("\nCell 2 completed successfully.")

In [ ]:
# Verify the original training targets

assert y_train.shape == (TRAINING_SAMPLES, NUM_LABELS)
assert y_val.shape == (VALIDATION_SAMPLES, NUM_LABELS)

assert np.isin(y_train, [0, 1]).all()
assert np.isin(y_val, [0, 1]).all()

train_positives = y_train.sum(axis=0).astype(np.int64)
train_negatives = TRAINING_SAMPLES - train_positives

assert (train_positives > 0).all()
assert (train_negatives > 0).all()


# Compute the same class weights used for DistilBERT

imbalance_ratios = train_negatives / train_positives

positive_weights = np.sqrt(
    imbalance_ratios
).astype(np.float32)

POS_WEIGHT = torch.tensor(
    positive_weights,
    dtype=torch.float32,
    device=DEVICE
)

assert POS_WEIGHT.shape == (NUM_LABELS,)
assert torch.isfinite(POS_WEIGHT).all()
assert (POS_WEIGHT > 0).all()

class_weights_df = pd.DataFrame({
    'Label': LABELS,
    'Positive Samples': train_positives,
    'Negative Samples': train_negatives,
    'Positive Prevalence (%)':
        train_positives / TRAINING_SAMPLES * 100,
    'Imbalance Ratio': imbalance_ratios,
    'Positive Weight': positive_weights
})

print("Training-derived class weights")

display(class_weights_df.round(6))


# Define the controlled loss configurations

FOCAL_GAMMA = 2.0
FOCAL_ALPHA_FACTOR = 1.0

LOSS_CONFIGS = {
    'T04_BCE_S42': {
        'loss_name': 'BCE',
        'positive_weighting': False,
        'focal_gamma': None,
        'focal_alpha_factor': None
    },
    'T05_WBCE_S42': {
        'loss_name': 'WBCE',
        'positive_weighting': True,
        'focal_gamma': None,
        'focal_alpha_factor': None
    },
    'T06_FOCAL_S42': {
        'loss_name': 'FOCAL',
        'positive_weighting': False,
        'focal_gamma': FOCAL_GAMMA,
        'focal_alpha_factor': FOCAL_ALPHA_FACTOR
    }
}


# Verify matching DistilBERT loss configurations

matched_runs = {
    'T04_BCE_S42': 'T01_BCE_S42',
    'T05_WBCE_S42': 'T02_WBCE_S42',
    'T06_FOCAL_S42': 'T03_FOCAL_S42'
}

for roberta_run, distilbert_run in matched_runs.items():

    previous_config = read_json(
        DISTILBERT_RESULTS_DIR
        / distilbert_run
        / 'config.json'
    )

    previous_loss = previous_config[
        'run_signature'
    ]['loss_configuration']

    for key, value in LOSS_CONFIGS[roberta_run].items():

        assert previous_loss[key] == value, (
            f"Loss configuration mismatch: "
            f"{roberta_run}, {key}"
        )

    print(
        f"{roberta_run}: Matched {distilbert_run}"
    )

loss_configuration_df = pd.DataFrame([
    {
        'Run ID': run_id,
        **loss_config
    }
    for run_id, loss_config in LOSS_CONFIGS.items()
])

print("\nRoBERTa loss configurations")

display(loss_configuration_df)


# Define numerically stable multilabel loss functions

def compute_multilabel_loss(
    logits,
    targets,
    loss_name,
    reduction='mean',
    focal_gamma=FOCAL_GAMMA
):

    if logits.shape != targets.shape:
        raise ValueError("Logits and targets must have equal shapes.")

    if logits.ndim != 2 or logits.shape[1] != NUM_LABELS:
        raise ValueError("Expected a batch of six-label predictions.")

    # Compute loss in FP32 for numerical stability

    logits = logits.float()
    targets = targets.float()

    if loss_name == 'BCE':

        loss = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction='none'
        )

    elif loss_name == 'WBCE':

        loss = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            pos_weight=POS_WEIGHT.to(logits.device),
            reduction='none'
        )

    elif loss_name == 'FOCAL':

        if focal_gamma < 0:
            raise ValueError("Focal gamma must be nonnegative.")

        bce = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction='none'
        )

        # Equivalent to 1 - exp(-BCE), with better stability

        focal_factor = (
            -torch.expm1(-bce)
        ).pow(focal_gamma)

        loss = (
            FOCAL_ALPHA_FACTOR
            * focal_factor
            * bce
        )

    else:

        raise ValueError(
            f"Unsupported loss function: {loss_name}"
        )

    if reduction == 'none':
        return loss

    if reduction == 'mean':
        return loss.mean()

    if reduction == 'sum':
        return loss.sum()

    raise ValueError(
        f"Unsupported reduction: {reduction}"
    )


# Define controlled test inputs

test_logits = torch.tensor([
    [-3.0, 2.0, -1.0, 0.0, 4.0, -2.0],
    [2.0, -3.0, 1.0, -4.0, 0.0, 3.0]
], dtype=torch.float32, device=DEVICE)

test_targets = torch.tensor([
    [0, 1, 0, 1, 1, 0],
    [1, 0, 1, 0, 0, 1]
], dtype=torch.float32, device=DEVICE)

verification_records = []


def verify_result(name, condition):

    assert bool(condition), f"Verification failed: {name}"

    verification_records.append({
        'Check': name,
        'Verification': 'Passed'
    })


# Verify BCE against an independent mathematical calculation

probabilities = torch.sigmoid(
    test_logits.double()
)

reference_bce = -(
    test_targets.double() * torch.log(probabilities)
    + (1 - test_targets.double())
    * torch.log1p(-probabilities)
)

actual_bce = compute_multilabel_loss(
    test_logits,
    test_targets,
    'BCE',
    reduction='none'
)

verify_result(
    'BCE mathematical equivalence',
    torch.allclose(
        actual_bce.double(),
        reference_bce,
        atol=1e-6,
        rtol=1e-6
    )
)


# Verify positive-only weighting

reference_wbce = reference_bce * (
    1 + test_targets.double()
    * (POS_WEIGHT.double() - 1)
)

actual_wbce = compute_multilabel_loss(
    test_logits,
    test_targets,
    'WBCE',
    reduction='none'
)

verify_result(
    'Weighted BCE mathematical equivalence',
    torch.allclose(
        actual_wbce.double(),
        reference_wbce,
        atol=1e-6,
        rtol=1e-6
    )
)

verify_result(
    'Weighted BCE leaves negative examples unchanged',
    torch.allclose(
        actual_wbce[test_targets == 0],
        actual_bce[test_targets == 0]
    )
)

verify_result(
    'Weighted BCE correctly scales positive examples',
    torch.allclose(
        actual_wbce[test_targets == 1],
        (
            actual_bce * POS_WEIGHT
        )[test_targets == 1],
        atol=1e-6,
        rtol=1e-6
    )
)


# Verify focal loss against its mathematical definition

p_t = (
    probabilities * test_targets.double()
    + (1 - probabilities)
    * (1 - test_targets.double())
)

reference_focal = (
    (1 - p_t).pow(FOCAL_GAMMA)
    * reference_bce
)

actual_focal = compute_multilabel_loss(
    test_logits,
    test_targets,
    'FOCAL',
    reduction='none'
)

verify_result(
    'Focal loss mathematical equivalence',
    torch.allclose(
        actual_focal.double(),
        reference_focal,
        atol=1e-6,
        rtol=1e-6
    )
)

focal_gamma_zero = compute_multilabel_loss(
    test_logits,
    test_targets,
    'FOCAL',
    reduction='none',
    focal_gamma=0.0
)

verify_result(
    'Focal loss equals BCE when gamma is zero',
    torch.allclose(
        focal_gamma_zero,
        actual_bce
    )
)


# Verify that focal loss downweights easy predictions

easy_logits = torch.full(
    (2, NUM_LABELS),
    8.0,
    device=DEVICE
)

hard_logits = -easy_logits

positive_targets = torch.ones_like(
    easy_logits
)

easy_bce = compute_multilabel_loss(
    easy_logits,
    positive_targets,
    'BCE'
)

hard_bce = compute_multilabel_loss(
    hard_logits,
    positive_targets,
    'BCE'
)

easy_focal = compute_multilabel_loss(
    easy_logits,
    positive_targets,
    'FOCAL'
)

hard_focal = compute_multilabel_loss(
    hard_logits,
    positive_targets,
    'FOCAL'
)

verify_result(
    'Focal loss emphasizes difficult predictions',
    easy_focal / easy_bce < hard_focal / hard_bce
)


# Verify reduction behavior

for loss_name in ['BCE', 'WBCE', 'FOCAL']:

    elementwise = compute_multilabel_loss(
        test_logits,
        test_targets,
        loss_name,
        reduction='none'
    )

    mean_loss = compute_multilabel_loss(
        test_logits,
        test_targets,
        loss_name,
        reduction='mean'
    )

    sum_loss = compute_multilabel_loss(
        test_logits,
        test_targets,
        loss_name,
        reduction='sum'
    )

    verify_result(
        f'{loss_name} reduction consistency',
        torch.allclose(
            mean_loss, elementwise.mean()
        )
        and torch.allclose(
            sum_loss, elementwise.sum()
        )
    )


# Verify numerical stability and gradients

gradient_records = []

extreme_logits = torch.tensor([
    [-100, -50, -10, 10, 50, 100],
    [100, 50, 10, -10, -50, -100]
], dtype=torch.float32, device=DEVICE)

extreme_targets = torch.tensor([
    [1, 0, 1, 0, 1, 0],
    [0, 1, 0, 1, 0, 1]
], dtype=torch.float32, device=DEVICE)

for loss_name in ['BCE', 'WBCE', 'FOCAL']:

    for input_dtype in [torch.float32, torch.float16]:

        for case_name, source_logits, targets in [
            ('Normal logits', test_logits, test_targets),
            ('Extreme logits', extreme_logits, extreme_targets)
        ]:

            logits = (
                source_logits.detach()
                .clone()
                .to(input_dtype)
                .requires_grad_(True)
            )

            loss = compute_multilabel_loss(
                logits,
                targets,
                loss_name
            )

            assert loss.dtype == torch.float32
            assert torch.isfinite(loss)
            assert loss.item() >= 0

            loss.backward()

            assert logits.grad is not None
            assert torch.isfinite(logits.grad).all()
            assert logits.grad.abs().sum().item() > 0

            gradient_records.append({
                'Loss': loss_name,
                'Input Dtype': str(input_dtype),
                'Test Case': case_name,
                'Loss Value': float(loss.item()),
                'Gradient Norm': float(
                    logits.grad.float().norm().item()
                ),
                'Finite Loss': True,
                'Finite Gradients': True,
                'Verification': 'Passed'
            })

            del logits, loss

gradient_verification_df = pd.DataFrame(
    gradient_records
)

loss_verification_df = pd.DataFrame(
    verification_records
)

assert len(loss_verification_df) == 10
assert len(gradient_verification_df) == 12

print("\nLoss mathematics verification")

display(loss_verification_df)

print("\nGradient and numerical stability verification")

display(gradient_verification_df.round(6))


# Prepare consolidated research evidence

loss_configuration = {
    'model': MODEL_NAME,
    'labels': LABELS,
    'training_samples': TRAINING_SAMPLES,
    'training_dataset_sha256': EXPECTED_TRAINING_HASH,
    'loss_configurations': LOSS_CONFIGS,
    'matched_distilbert_runs': matched_runs,
    'positive_weight_formula':
        'sqrt(negative_samples / positive_samples)',
    'weight_source': 'representative_training_subset',
    'training_positives': train_positives.tolist(),
    'training_negatives': train_negatives.tolist(),
    'positive_weights': {
        label: float(weight)
        for label, weight in zip(
            LABELS, positive_weights
        )
    },
    'focal_gamma': FOCAL_GAMMA,
    'focal_alpha_factor': FOCAL_ALPHA_FACTOR,
    'loss_computation_dtype': 'float32',
    'default_reduction': 'mean',
    'mathematical_checks': verification_records,
    'gradient_checks': gradient_records,
    'distilbert_loss_configurations_matched': True,
    'validation_used_for_class_weights': False,
    'official_test_used': False,
    'model_training_performed': False
}

LOSS_CONFIG_PATH = (
    RUN_CONFIGS_DIR
    / 'roberta_loss_configuration.json'
)

save_json_atomic(
    LOSS_CONFIG_PATH,
    loss_configuration
)

assert read_json(LOSS_CONFIG_PATH) == loss_configuration

LOSS_CONFIG_HASH = calculate_sha256(
    LOSS_CONFIG_PATH
)


# Verify the original research records remain unchanged

assert calculate_sha256(TRAIN_PATH) == (
    EXPECTED_TRAINING_HASH
)

assert calculate_sha256(VALIDATION_PATH) == (
    EXPECTED_VALIDATION_HASH
)

assert calculate_sha256(REGISTRY_PATH) == (
    registry_original_hash
)

assert calculate_sha256(DECISION_LOG_PATH) == (
    decision_log_original_hash
)


# Display the final completion summary

print("\nRoBERTa loss configuration summary")

print("Loss configurations:", len(LOSS_CONFIGS))
print("Class weighting: Square-root imbalance ratio")
print("Class weight source: Training subset only")
print("Focal gamma:", FOCAL_GAMMA)
print("Focal alpha factor:", FOCAL_ALPHA_FACTOR)

print(
    "\nMathematical checks:",
    f"{len(loss_verification_df)}/10 passed"
)

print(
    "Gradient checks:",
    f"{len(gradient_verification_df)}/12 passed"
)

print("DistilBERT loss configurations: Matched")
print("Loss computation precision: FP32")
print("Loss configuration: Saved")

print("\nExperiment registry: Unchanged")
print("Decision log: Unchanged")
print("Additional figures generated: 0")
print("Model training performed: No")
print("Official test dataset: Not loaded")

gc.collect()
torch.cuda.empty_cache()

print("\nCell 3 completed successfully.")

In [ ]:
from transformers import AutoModelForSequenceClassification

# Verify prerequisites

assert 'tokenizer' in globals()
assert 'compute_multilabel_loss' in globals()
assert 'LOSS_CONFIG_HASH' in globals()

assert calculate_sha256(LOSS_CONFIG_PATH) == LOSS_CONFIG_HASH

assert MODEL_NAME == 'RoBERTa'
assert NUM_LABELS == 6
assert SEED == 42


# Define shared initialization paths

INITIAL_MODEL_DIR = (
    MODELS_DIR / 'shared_initialization_s42'
)

INITIAL_MODEL_MANIFEST = (
    INITIAL_MODEL_DIR / 'initialization_manifest.json'
)

LOCAL_INITIAL_MODEL_DIR = Path(
    tempfile.mkdtemp(
        prefix='roberta_initialization_',
        dir='/content'
    )
)

initialization_specification = {
    'checkpoint': MODEL_CHECKPOINT,
    'revision': MODEL_REVISION,
    'seed': SEED,
    'num_labels': NUM_LABELS,
    'labels': LABELS,
    'problem_type': 'multi_label_classification',
    'parameter_dtype': 'float32'
}


# Define a deterministic model-state fingerprint

def model_state_sha256(model):

    digest = hashlib.sha256()

    for name, tensor in sorted(
        model.state_dict().items()
    ):

        tensor = tensor.detach().cpu().contiguous()

        digest.update(name.encode('utf-8'))

        digest.update(
            str(tensor.dtype).encode('utf-8')
        )

        digest.update(
            str(tuple(tensor.shape)).encode('utf-8')
        )

        digest.update(
            tensor.numpy().tobytes()
        )

    return digest.hexdigest()


# Verify model configuration

def verify_model_configuration(model):

    assert model.config.model_type == 'roberta'

    assert model.config.num_labels == NUM_LABELS

    assert model.config.problem_type == (
        'multi_label_classification'
    )

    assert model.config.vocab_size == (
        model_config['vocabulary_size']
    )

    assert model.config.id2label == {
        index: label
        for index, label in enumerate(LABELS)
    }

    assert model.config.label2id == {
        label: index
        for index, label in enumerate(LABELS)
    }

    assert model.classifier.out_proj.out_features == (
        NUM_LABELS
    )

    assert all(
        parameter.dtype == torch.float32
        for parameter in model.parameters()
    )

    assert all(
        parameter.requires_grad
        for parameter in model.parameters()
    )


# Verify model directory fingerprints

def verify_model_files(model_dir, manifest):

    model_dir = Path(model_dir)

    for filename, expected_hash in manifest['files'].items():

        path = model_dir / filename

        assert path.is_file(), (
            f"Missing model artifact: {path}"
        )

        assert calculate_sha256(path) == expected_hash, (
            f"Model artifact fingerprint mismatch: {path}"
        )

    return True


# Initialize or reuse the shared RoBERTa model

initialization_reused = False

if INITIAL_MODEL_DIR.exists():

    assert INITIAL_MODEL_MANIFEST.is_file(), (
        "Existing initialization directory is incomplete. "
        "Inspect it before continuing."
    )

    initial_manifest = read_json(
        INITIAL_MODEL_MANIFEST
    )

    assert initial_manifest['specification'] == (
        initialization_specification
    )

    verify_model_files(
        INITIAL_MODEL_DIR,
        initial_manifest
    )

    print("Existing shared initialization verified.")

    shutil.copytree(
        INITIAL_MODEL_DIR,
        LOCAL_INITIAL_MODEL_DIR,
        dirs_exist_ok=True
    )

    verify_model_files(
        LOCAL_INITIAL_MODEL_DIR,
        initial_manifest
    )

    initialization_reused = True

else:

    print(
        "Downloading and initializing RoBERTa-base...",
        flush=True
    )

    set_random_seed(SEED)

    initial_model = (
        AutoModelForSequenceClassification
        .from_pretrained(
            MODEL_CHECKPOINT,
            revision=MODEL_REVISION,
            num_labels=NUM_LABELS,
            problem_type='multi_label_classification',
            id2label={
                index: label
                for index, label in enumerate(LABELS)
            },
            label2id={
                label: index
                for index, label in enumerate(LABELS)
            },
            dtype=torch.float32
        )
    )

    initial_model = initial_model.cpu()

    verify_model_configuration(initial_model)

    initial_state_hash = model_state_sha256(
        initial_model
    )

    total_parameters = sum(
        parameter.numel()
        for parameter in initial_model.parameters()
    )

    trainable_parameters = sum(
        parameter.numel()
        for parameter in initial_model.parameters()
        if parameter.requires_grad
    )

    assert total_parameters == trainable_parameters

    # Save one local initialization copy

    initial_model.save_pretrained(
        LOCAL_INITIAL_MODEL_DIR
    )

    tokenizer.save_pretrained(
        LOCAL_INITIAL_MODEL_DIR
    )

    initial_manifest = {
        'specification': initialization_specification,
        'created_at': datetime.now(
            timezone.utc
        ).isoformat(),
        'total_parameters': total_parameters,
        'trainable_parameters': trainable_parameters,
        'model_state_sha256': initial_state_hash,
        'attention_implementation': getattr(
            initial_model.config,
            '_attn_implementation',
            None
        ),
        'torch_version': str(torch.__version__),
        'transformers_version': transformers.__version__,
        'files': {
            path.name: calculate_sha256(path)
            for path in LOCAL_INITIAL_MODEL_DIR.iterdir()
            if path.is_file()
        }
    }

    save_json_atomic(
        LOCAL_INITIAL_MODEL_DIR
        / 'initialization_manifest.json',
        initial_manifest
    )

    verify_model_files(
        LOCAL_INITIAL_MODEL_DIR,
        initial_manifest
    )

    del initial_model

    gc.collect()

    # Persist initialization using a verified staging directory

    staging_dir = MODELS_DIR / (
        'shared_initialization_s42_staging_'
        + datetime.now(timezone.utc).strftime(
            '%Y%m%d_%H%M%S_%f'
        )
    )

    assert not staging_dir.exists()
    assert not INITIAL_MODEL_DIR.exists()

    shutil.copytree(
        LOCAL_INITIAL_MODEL_DIR,
        staging_dir
    )

    verify_model_files(
        staging_dir,
        initial_manifest
    )

    assert read_json(
        staging_dir / 'initialization_manifest.json'
    ) == initial_manifest

    staging_dir.rename(INITIAL_MODEL_DIR)

    verify_model_files(
        INITIAL_MODEL_DIR,
        initial_manifest
    )

    print("Shared initialization saved and verified.")


# Reload the saved initialization independently

print(
    "\nVerifying shared model reload...",
    flush=True
)

set_random_seed(SEED)

verification_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        LOCAL_INITIAL_MODEL_DIR,
        local_files_only=True,
        dtype=torch.float32
    )
)

verification_model = verification_model.cpu()

verify_model_configuration(
    verification_model
)

INITIAL_STATE_HASH = model_state_sha256(
    verification_model
)

assert INITIAL_STATE_HASH == (
    initial_manifest['model_state_sha256']
)

total_parameters = sum(
    parameter.numel()
    for parameter in verification_model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in verification_model.parameters()
    if parameter.requires_grad
)

assert total_parameters == (
    initial_manifest['total_parameters']
)

assert trainable_parameters == (
    initial_manifest['trainable_parameters']
)

print("Reloaded model state: Exact fingerprint match.")


# Verify the saved tokenizer

verification_tokenizer = AutoTokenizer.from_pretrained(
    LOCAL_INITIAL_MODEL_DIR,
    local_files_only=True,
    use_fast=True
)

assert verification_tokenizer.vocab_size == (
    tokenizer.vocab_size
)

assert verification_tokenizer.pad_token_id == (
    tokenizer.pad_token_id
)

assert verification_tokenizer.bos_token_id == (
    tokenizer.bos_token_id
)

assert verification_tokenizer.eos_token_id == (
    tokenizer.eos_token_id
)

sample_indices = [
    0,
    TRAINING_SAMPLES // 2,
    TRAINING_SAMPLES - 1
]

sample_texts = [
    train_texts[index]
    for index in sample_indices
]

original_tokens = tokenizer(
    sample_texts,
    truncation=True,
    max_length=MAX_LENGTH,
    padding=False
)['input_ids']

reloaded_tokens = verification_tokenizer(
    sample_texts,
    truncation=True,
    max_length=MAX_LENGTH,
    padding=False
)['input_ids']

assert original_tokens == reloaded_tokens

print("Reloaded tokenizer: Verified.")


# Verify the output shape and independent sigmoid scores

verification_model.eval()

sample_batch = dynamic_collate_fn([
    train_dataset[index]
    for index in sample_indices
])

with torch.inference_mode():

    outputs = verification_model(
        input_ids=sample_batch['input_ids'],
        attention_mask=sample_batch['attention_mask']
    )

    logits = outputs.logits

    probabilities = torch.sigmoid(
        logits.float()
    )

assert logits.shape == (
    len(sample_indices), NUM_LABELS
)

assert torch.isfinite(logits).all()
assert torch.isfinite(probabilities).all()

assert (
    (probabilities >= 0)
    & (probabilities <= 1)
).all()

print("\nInitial model forward-pass verification")

display(pd.DataFrame(
    probabilities.cpu().numpy(),
    columns=LABELS
).round(6))

print("Output shape:", tuple(logits.shape))
print("Independent sigmoid outputs: Verified")

del outputs, logits, probabilities
del verification_model, verification_tokenizer
del sample_batch

gc.collect()
torch.cuda.empty_cache()


# Define the model loader for independent experiments

def load_fresh_model():

    set_random_seed(SEED)

    verify_model_files(
        LOCAL_INITIAL_MODEL_DIR,
        initial_manifest
    )

    model = (
        AutoModelForSequenceClassification
        .from_pretrained(
            LOCAL_INITIAL_MODEL_DIR,
            local_files_only=True,
            dtype=torch.float32
        )
    )

    verify_model_configuration(model)

    assert model_state_sha256(model) == (
        INITIAL_STATE_HASH
    ), "Initial model weights differ."

    return model.to(DEVICE)


# Summarize the architecture

model_size_bytes = sum(
    path.stat().st_size
    for path in INITIAL_MODEL_DIR.rglob('*')
    if path.is_file()
)

parameter_summary_df = pd.DataFrame([
    ('Model', MODEL_NAME),
    ('Checkpoint', MODEL_CHECKPOINT),
    ('Hidden size', 768),
    ('Number of encoder layers', 12),
    ('Attention heads', 12),
    ('Classification outputs', NUM_LABELS),
    ('Total parameters', total_parameters),
    ('Trainable parameters', trainable_parameters),
    ('Frozen parameters',
     total_parameters - trainable_parameters),
    ('Parameter dtype', 'float32'),
    ('Shared initialization size (MiB)',
     model_size_bytes / 1024**2),
    ('Initial weights verified', True)
], columns=['Parameter', 'Value'])

print("\nRoBERTa architecture summary")

display(parameter_summary_df)


# Confirm identical initialization for every planned run

initialization_records = []

for run_id in ROBERTA_RUN_IDS:

    initialization_records.append({
        'Run ID': run_id,
        'Seed': SEED,
        'Initial Model SHA256': INITIAL_STATE_HASH,
        'Classification Outputs': NUM_LABELS,
        'Initialization Source': str(
            INITIAL_MODEL_DIR.relative_to(PROJECT_ROOT)
        )
    })

print("\nPlanned shared initialization")

display(pd.DataFrame(initialization_records))


# Verify previous research records remain unchanged

assert calculate_sha256(REGISTRY_PATH) == (
    registry_original_hash
)

assert calculate_sha256(DECISION_LOG_PATH) == (
    decision_log_original_hash
)

assert calculate_sha256(LOSS_CONFIG_PATH) == (
    LOSS_CONFIG_HASH
)

assert read_json(INITIAL_MODEL_MANIFEST) == (
    initial_manifest
)


# Display final completion summary

print("\nRoBERTa initialization summary")

print("Total parameters:", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")
print("Classification labels:", NUM_LABELS)

print("\nInitialization reused:", initialization_reused)

print(
    "Shared initialization size (MiB):",
    round(model_size_bytes / 1024**2, 4)
)

print("Initial model SHA256:", INITIAL_STATE_HASH)

print("\nShared model initialization: Saved")
print("Model reload fingerprint: Exact match")
print("Tokenizer reload: Verified")
print("Forward-pass verification: Passed")
print("Fresh model loader: Ready")

print("\nExperiment registry: Unchanged")
print("Decision log: Unchanged")
print("Additional figures generated: 0")
print("Model training performed: No")
print("Official test dataset: Not loaded")

print("\nCell 4 completed successfully.")

In [ ]:
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import Subset, DataLoader

# Restore the controlled training parameters

TRAIN_BATCH_SIZE = training_policy['physical_batch_size']
EVAL_BATCH_SIZE = training_policy['validation_batch_size']

EFFECTIVE_BATCH_SIZE = training_policy['effective_batch_size']
GRAD_ACCUMULATION_STEPS = training_policy['gradient_accumulation_steps']

LEARNING_RATE = training_policy['learning_rate']
WEIGHT_DECAY = training_policy['weight_decay']
WARMUP_RATIO = training_policy['warmup_ratio']

MAX_EPOCHS = training_policy['maximum_epochs']
EARLY_STOPPING_PATIENCE = training_policy['early_stopping_patience']

MAX_GRAD_NORM = training_policy['maximum_gradient_norm']
AMP_INITIAL_SCALE = training_policy['amp_initial_scale']

USE_FP16 = True

CHECKPOINT_INTERVAL = 500
LOG_INTERVAL = 100

assert (
    TRAIN_BATCH_SIZE * GRAD_ACCUMULATION_STEPS
    == EFFECTIVE_BATCH_SIZE
)

BATCHES_PER_EPOCH = math.ceil(
    TRAINING_SAMPLES / TRAIN_BATCH_SIZE
)

UPDATES_PER_EPOCH = math.ceil(
    TRAINING_SAMPLES / EFFECTIVE_BATCH_SIZE
)

TOTAL_TRAINING_STEPS = (
    UPDATES_PER_EPOCH * MAX_EPOCHS
)

WARMUP_STEPS = math.ceil(
    TOTAL_TRAINING_STEPS * WARMUP_RATIO
)

FINAL_ACCUMULATION_SIZE = (
    TRAINING_SAMPLES % EFFECTIVE_BATCH_SIZE
    or EFFECTIVE_BATCH_SIZE
)

assert UPDATES_PER_EPOCH == 1875
assert TOTAL_TRAINING_STEPS == 5625
assert WARMUP_STEPS == 563
assert FINAL_ACCUMULATION_SIZE == 29

LOCAL_CHECKPOINT_DIR = (
    Path('/content/roberta_runtime_checkpoints')
)

LOCAL_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Training step accounting")

display(pd.DataFrame([
    ('Training comments', TRAINING_SAMPLES),
    ('Physical batch size', TRAIN_BATCH_SIZE),
    ('Gradient accumulation', GRAD_ACCUMULATION_STEPS),
    ('Effective batch size', EFFECTIVE_BATCH_SIZE),
    ('Batches per epoch', BATCHES_PER_EPOCH),
    ('Updates per epoch', UPDATES_PER_EPOCH),
    ('Maximum training steps', TOTAL_TRAINING_STEPS),
    ('Warmup steps', WARMUP_STEPS),
    ('Final accumulation group', FINAL_ACCUMULATION_SIZE)
], columns=['Parameter', 'Value']))


# Define independent training components

def create_training_components(
    total_steps=TOTAL_TRAINING_STEPS,
    warmup_steps=WARMUP_STEPS
):

    model = load_fresh_model()

    decay_parameters = []
    no_decay_parameters = []

    for name, parameter in model.named_parameters():

        if not parameter.requires_grad:
            continue

        if parameter.ndim <= 1 or name.endswith('.bias'):
            no_decay_parameters.append(parameter)

        else:
            decay_parameters.append(parameter)

    assert (
        len(decay_parameters)
        + len(no_decay_parameters)
        == len(list(model.parameters()))
    )

    optimizer = torch.optim.AdamW(
        [
            {
                'params': decay_parameters,
                'weight_decay': WEIGHT_DECAY
            },
            {
                'params': no_decay_parameters,
                'weight_decay': 0.0
            }
        ],
        lr=LEARNING_RATE,
        betas=(0.9, 0.999),
        eps=1e-8
    )

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    scaler = torch.amp.GradScaler(
        'cuda',
        enabled=USE_FP16,
        init_scale=AMP_INITIAL_SCALE
    )

    optimizer.zero_grad(set_to_none=True)

    return model, optimizer, scheduler, scaler


# Define reproducible epoch ordering

def get_epoch_indices(
    epoch,
    sample_count=TRAINING_SAMPLES
):

    assert epoch >= 1

    rng = np.random.default_rng(
        SEED + epoch - 1
    )

    return rng.permutation(sample_count)


def create_epoch_loader(
    dataset,
    epoch,
    start_row=0
):

    indices = get_epoch_indices(
        epoch,
        len(dataset)
    )

    assert 0 <= start_row <= len(dataset)

    remaining_indices = (
        indices[start_row:].tolist()
    )

    generator = torch.Generator()

    generator.manual_seed(
        SEED + epoch - 1
    )

    return DataLoader(
        Subset(dataset, remaining_indices),
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=dynamic_collate_fn,
        generator=generator
    )


# Define initial training progress

def new_training_progress():

    return {
        'epoch': 1,
        'next_row': 0,
        'epoch_loss_sum': 0.0,
        'epoch_examples': 0,
        'attempted_updates': 0,
        'successful_updates': 0,
        'skipped_updates': 0,
        'best_macro_ap': None,
        'best_epoch': None,
        'best_checkpoint_path': None,
        'best_validation_path': None,
        'best_validation_sha256': None,
        'epochs_without_improvement': 0,
        'history': [],
        'training_time_seconds': 0.0,
        'training_finished': False,
        'stopping_reason': None
    }


# Capture and restore random-number-generator states

def capture_rng_state():

    return {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
        'torch_cuda': torch.cuda.get_rng_state_all()
    }


def restore_rng_state(state):

    random.setstate(state['python'])

    np.random.set_state(state['numpy'])

    torch.set_rng_state(
        state['torch_cpu'].cpu()
    )

    torch.cuda.set_rng_state_all([
        item.cpu()
        for item in state['torch_cuda']
    ])


# Define immutable experiment signatures

def get_run_signature(run_id):

    assert run_id in LOSS_CONFIGS

    return {
        'signature_version': 1,
        'run_id': run_id,
        'model': MODEL_NAME,
        'checkpoint': MODEL_CHECKPOINT,
        'revision': MODEL_REVISION,

        'loss_configuration': LOSS_CONFIGS[run_id].copy(),
        'loss_configuration_sha256': LOSS_CONFIG_HASH,

        'initial_model_sha256': INITIAL_STATE_HASH,

        'training_dataset_sha256': EXPECTED_TRAINING_HASH,
        'validation_dataset_sha256': EXPECTED_VALIDATION_HASH,

        'training_token_cache_sha256':
            train_cache_record['Cache SHA256'],

        'validation_token_cache_sha256':
            val_cache_record['Cache SHA256'],

        'evaluation_module_sha256': evaluation_hash,

        'labels': LABELS,
        'seed': SEED,

        'physical_batch_size': TRAIN_BATCH_SIZE,
        'validation_batch_size': EVAL_BATCH_SIZE,
        'effective_batch_size': EFFECTIVE_BATCH_SIZE,

        'gradient_accumulation_steps':
            GRAD_ACCUMULATION_STEPS,

        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'optimizer_betas': [0.9, 0.999],
        'optimizer_epsilon': 1e-8,

        'maximum_gradient_norm': MAX_GRAD_NORM,
        'maximum_epochs': MAX_EPOCHS,

        'early_stopping_patience':
            EARLY_STOPPING_PATIENCE,

        'warmup_steps': WARMUP_STEPS,
        'total_training_steps': TOTAL_TRAINING_STEPS,

        'fp16': USE_FP16,
        'amp_initial_scale': AMP_INITIAL_SCALE,

        'maximum_length': MAX_LENGTH,
        'padding_multiple': PAD_MULTIPLE,

        'local_checkpoint_interval_updates':
            CHECKPOINT_INTERVAL,

        'persistent_checkpoint_frequency':
            'epoch_boundary',

        'gpu_name': torch.cuda.get_device_name(0),
        'torch_version': str(torch.__version__),
        'transformers_version': transformers.__version__,
        'numpy_version': np.__version__,
        'tokenizers_version': tokenizers.__version__,

        'attention_implementation':
            initial_manifest.get(
                'attention_implementation'
            ),

        'deterministic_algorithms':
            torch.are_deterministic_algorithms_enabled()
    }


# Safely write an exact-resume checkpoint

def save_resume_checkpoint(
    path,
    model,
    optimizer,
    scheduler,
    scaler,
    progress,
    run_signature
):

    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    # Checkpoints are only valid after an accumulation group

    assert all(
        parameter.grad is None
        for parameter in model.parameters()
    )

    torch.cuda.synchronize()

    checkpoint = {
        'schema_version': 1,
        'run_signature': run_signature,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'scaler_state': scaler.state_dict(),
        'progress': progress,
        'rng_state': capture_rng_state()
    }

    is_drive_path = path.resolve().is_relative_to(
        PROJECT_ROOT.resolve()
    )

    local_temp = Path(
        tempfile.mkstemp(
            suffix='.pt',
            prefix='roberta_checkpoint_',
            dir='/content'
        )[1]
    )

    # Close the descriptor created by mkstemp before writing
    # The temporary file remains owned by this function

    staged_path = None

    try:

        torch.save(checkpoint, local_temp)

        local_size = local_temp.stat().st_size

        assert local_size > 0

        expected_hash = calculate_sha256(local_temp)

        if is_drive_path:

            staged_path = path.with_name(
                path.name + '.staging'
            )

            assert not staged_path.exists(), (
                f"Unresolved staging file: {staged_path}"
            )

            shutil.copyfile(
                local_temp,
                staged_path
            )

            assert staged_path.stat().st_size == local_size

            assert calculate_sha256(staged_path) == (
                expected_hash
            )

            staged_path.replace(path)

        else:

            local_temp.replace(path)

        assert path.stat().st_size == local_size

        assert calculate_sha256(path) == expected_hash

        return {
            'path': str(path),
            'size_bytes': local_size,
            'sha256': expected_hash
        }

    finally:

        if local_temp.exists():
            local_temp.unlink()

        del checkpoint


# Restore a trusted project checkpoint

def load_resume_checkpoint(
    path,
    model,
    optimizer,
    scheduler,
    scaler,
    run_signature
):

    path = Path(path)

    assert path.is_file()

    checkpoint = torch.load(
        path,
        map_location='cpu',
        weights_only=False
    )

    assert checkpoint['schema_version'] == 1

    assert checkpoint['run_signature'] == (
        run_signature
    ), "Checkpoint configuration mismatch."

    model.load_state_dict(
        checkpoint['model_state'],
        strict=True
    )

    optimizer.load_state_dict(
        checkpoint['optimizer_state']
    )

    scheduler.load_state_dict(
        checkpoint['scheduler_state']
    )

    scaler.load_state_dict(
        checkpoint['scaler_state']
    )

    progress = checkpoint['progress']

    optimizer.zero_grad(set_to_none=True)

    # Restore RNG after all model and optimizer initialization

    restore_rng_state(
        checkpoint['rng_state']
    )

    del checkpoint

    gc.collect()

    return progress


# Define one training epoch with correct gradient accumulation

def train_one_epoch(
    model,
    optimizer,
    scheduler,
    scaler,
    progress,
    loss_name,
    dataset=None,
    checkpoint_callback=None,
    stop_after_updates=None,
    verbose=True
):

    if dataset is None:
        dataset = train_dataset

    sample_count = len(dataset)

    epoch = progress['epoch']
    start_row = progress['next_row']

    assert progress['epoch_examples'] == start_row

    assert (
        start_row % EFFECTIVE_BATCH_SIZE == 0
        or start_row == sample_count
    )

    if start_row == sample_count:

        return {
            'epoch_completed': True,
            'training_loss': (
                progress['epoch_loss_sum']
                / (sample_count * NUM_LABELS)
            )
        }

    model.train()

    optimizer.zero_grad(set_to_none=True)

    loader = create_epoch_loader(
        dataset,
        epoch,
        start_row
    )

    loader_iterator = iter(loader)

    updates_this_call = 0

    while progress['next_row'] < sample_count:

        torch.cuda.synchronize()

        group_start_time = time.perf_counter()

        remaining_rows = (
            sample_count - progress['next_row']
        )

        # Normalize by the actual group size, including the last group

        group_size = min(
            EFFECTIVE_BATCH_SIZE,
            remaining_rows
        )

        microbatches = math.ceil(
            group_size / TRAIN_BATCH_SIZE
        )

        processed_in_group = 0
        group_loss_sum = 0.0

        for _ in range(microbatches):

            batch = next(loader_iterator)

            input_ids = batch['input_ids'].to(
                DEVICE,
                non_blocking=True
            )

            attention_mask = batch['attention_mask'].to(
                DEVICE,
                non_blocking=True
            )

            targets = batch['labels'].to(
                DEVICE,
                non_blocking=True
            )

            actual_batch_size = targets.shape[0]

            with torch.autocast(
                device_type='cuda',
                dtype=torch.float16,
                enabled=USE_FP16
            ):

                logits = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                ).logits

            loss_sum = compute_multilabel_loss(
                logits,
                targets,
                loss_name,
                reduction='sum'
            )

            assert torch.isfinite(loss_sum), (
                "Non-finite training loss."
            )

            normalized_loss = (
                loss_sum / (group_size * NUM_LABELS)
            )

            scaler.scale(
                normalized_loss
            ).backward()

            processed_in_group += actual_batch_size

            group_loss_sum += float(
                loss_sum.detach().item()
            )

            del batch, input_ids, attention_mask
            del targets, logits, loss_sum, normalized_loss

        assert processed_in_group == group_size

        # Clip unscaled gradients before the optimizer step

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            MAX_GRAD_NORM
        )

        previous_scale = scaler.get_scale()

        scaler.step(optimizer)

        scaler.update()

        updated_scale = scaler.get_scale()

        skipped_update = (
            updated_scale < previous_scale
        )

        progress['attempted_updates'] += 1

        if skipped_update:

            progress['skipped_updates'] += 1

        else:

            scheduler.step()

            progress['successful_updates'] += 1

        optimizer.zero_grad(set_to_none=True)

        progress['next_row'] += group_size
        progress['epoch_examples'] += group_size
        progress['epoch_loss_sum'] += group_loss_sum

        updates_this_call += 1

        torch.cuda.synchronize()

        progress['training_time_seconds'] += (
            time.perf_counter() - group_start_time
        )

        current_update = progress['attempted_updates']

        if verbose and (
            current_update % LOG_INTERVAL == 0
        ):

            running_loss = (
                progress['epoch_loss_sum']
                / (
                    progress['epoch_examples']
                    * NUM_LABELS
                )
            )

            print(
                f"Epoch {epoch} | "
                f"Rows {progress['next_row']:,}/{sample_count:,} | "
                f"Update {current_update:,} | "
                f"Loss {running_loss:.6f}",
                flush=True
            )

        # Frequent checkpoints stay on the local runtime

        if (
            checkpoint_callback is not None
            and current_update % CHECKPOINT_INTERVAL == 0
            and progress['next_row'] < sample_count
        ):

            checkpoint_callback(
                model,
                optimizer,
                scheduler,
                scaler,
                progress
            )

        if (
            stop_after_updates is not None
            and updates_this_call >= stop_after_updates
        ):

            break

    epoch_completed = (
        progress['next_row'] == sample_count
    )

    assert (
        progress['attempted_updates']
        == progress['successful_updates']
        + progress['skipped_updates']
    )

    return {
        'epoch_completed': epoch_completed,
        'training_loss': (
            progress['epoch_loss_sum']
            / (
                progress['epoch_examples']
                * NUM_LABELS
            )
        )
    }


# Define validation inference

def evaluate_validation_model(
    model,
    loss_name,
    dataset=None,
    targets=None,
    compute_metrics=True
):

    if dataset is None:
        dataset = val_dataset

    if targets is None:
        targets = y_val

    targets = np.asarray(targets)

    assert targets.shape == (
        len(dataset), NUM_LABELS
    )

    generator = torch.Generator()

    generator.manual_seed(SEED)

    loader = DataLoader(
        dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=dynamic_collate_fn,
        generator=generator
    )

    logits_array = np.empty(
        (len(dataset), NUM_LABELS),
        dtype=np.float32
    )

    scores_array = np.empty_like(logits_array)

    seen = np.zeros(
        len(dataset),
        dtype=bool
    )

    validation_loss_sum = 0.0

    model.eval()

    torch.cuda.synchronize()

    start_time = time.perf_counter()

    with torch.inference_mode():

        for batch in loader:

            row_indices = (
                batch['row_index'].numpy()
            )

            assert not seen[row_indices].any()

            np.testing.assert_array_equal(
                batch['labels'].numpy(),
                targets[row_indices]
            )

            input_ids = batch['input_ids'].to(
                DEVICE,
                non_blocking=True
            )

            attention_mask = batch['attention_mask'].to(
                DEVICE,
                non_blocking=True
            )

            batch_targets = batch['labels'].to(
                DEVICE,
                non_blocking=True
            )

            with torch.autocast(
                device_type='cuda',
                dtype=torch.float16,
                enabled=USE_FP16
            ):

                logits = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                ).logits

            logits = logits.float()

            scores = torch.sigmoid(logits)

            loss_sum = compute_multilabel_loss(
                logits,
                batch_targets,
                loss_name,
                reduction='sum'
            )

            assert torch.isfinite(logits).all()
            assert torch.isfinite(loss_sum)

            logits_array[row_indices] = (
                logits.cpu().numpy()
            )

            scores_array[row_indices] = (
                scores.cpu().numpy()
            )

            seen[row_indices] = True

            validation_loss_sum += float(
                loss_sum.item()
            )

    assert seen.all()

    assert np.isfinite(logits_array).all()
    assert np.isfinite(scores_array).all()

    assert (
        (scores_array >= 0)
        & (scores_array <= 1)
    ).all()

    validation_loss = (
        validation_loss_sum
        / (len(dataset) * NUM_LABELS)
    )

    overall_metrics = None
    per_label_metrics = None

    if compute_metrics:

        overall_metrics, per_label_metrics = (
            evaluate_multilabel(
                targets,
                scores_array,
                LABELS,
                thresholds=0.5
            )
        )

    torch.cuda.synchronize()

    validation_seconds = (
        time.perf_counter() - start_time
    )

    return {
        'logits': logits_array,
        'scores': scores_array,
        'validation_loss': validation_loss,
        'overall_metrics': overall_metrics,
        'per_label_metrics': per_label_metrics,
        'validation_time_seconds': validation_seconds
    }


# Save a selected best model and tokenizer

def save_best_model(
    model,
    run_model_dir,
    epoch
):

    run_model_dir = Path(run_model_dir)

    run_model_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    timestamp = datetime.now(
        timezone.utc
    ).strftime('%Y%m%d_%H%M%S_%f')

    model_dir = run_model_dir / (
        f'best_epoch_{epoch:02d}_{timestamp}'
    )

    staging_dir = model_dir.with_name(
        model_dir.name + '_staging'
    )

    assert not model_dir.exists()
    assert not staging_dir.exists()

    with tempfile.TemporaryDirectory(
        prefix='roberta_best_',
        dir='/content'
    ) as temporary_directory:

        local_dir = Path(temporary_directory)

        model.save_pretrained(local_dir)

        tokenizer.save_pretrained(local_dir)

        manifest = {
            'epoch': epoch,
            'labels': LABELS,
            'seed': SEED,
            'initial_model_sha256': INITIAL_STATE_HASH,
            'created_at': datetime.now(
                timezone.utc
            ).isoformat(),

            'model_state_sha256':
                model_state_sha256(model),

            'files': {
                path.name: calculate_sha256(path)
                for path in local_dir.iterdir()
                if path.is_file()
            }
        }

        save_json_atomic(
            local_dir / 'checkpoint_manifest.json',
            manifest
        )

        verify_model_files(
            local_dir,
            manifest
        )

        shutil.copytree(
            local_dir,
            staging_dir
        )

        verify_model_files(
            staging_dir,
            manifest
        )

        assert read_json(
            staging_dir / 'checkpoint_manifest.json'
        ) == manifest

        staging_dir.rename(model_dir)

        verify_model_files(
            model_dir,
            manifest
        )

    return model_dir


# Define safe epoch-boundary progress updates

def advance_epoch(progress):

    assert progress['next_row'] > 0

    assert progress['next_row'] == (
        progress['epoch_examples']
    )

    progress['epoch'] += 1
    progress['next_row'] = 0
    progress['epoch_loss_sum'] = 0.0
    progress['epoch_examples'] = 0

    return progress


# Verify deterministic ordering without model training

epoch_one_indices = get_epoch_indices(1)

epoch_one_repeat = get_epoch_indices(1)

epoch_two_indices = get_epoch_indices(2)

assert len(epoch_one_indices) == TRAINING_SAMPLES

assert len(
    np.unique(epoch_one_indices)
) == TRAINING_SAMPLES

np.testing.assert_array_equal(
    epoch_one_indices,
    epoch_one_repeat
)

assert not np.array_equal(
    epoch_one_indices,
    epoch_two_indices
)

# Verify that resumed ordering preserves the unprocessed rows

resume_row = EFFECTIVE_BATCH_SIZE * 2

resume_loader = create_epoch_loader(
    train_dataset,
    epoch=1,
    start_row=resume_row
)

first_resume_batch = next(iter(resume_loader))

np.testing.assert_array_equal(
    first_resume_batch['row_index'].numpy(),
    epoch_one_indices[
        resume_row:resume_row + TRAIN_BATCH_SIZE
    ]
)

del resume_loader, first_resume_batch

print("\nSample ordering verification")

display(pd.DataFrame([
    ('Unique rows per epoch', 'Passed'),
    ('Same-seed ordering', 'Passed'),
    ('Different epoch ordering', 'Passed'),
    ('Resume cursor alignment', 'Passed'),
    ('Final accumulation group accounting', 'Passed')
], columns=['Check', 'Verification']))


# Update the persistent checkpoint policy

training_policy.update({
    'local_checkpoint_interval_updates':
        CHECKPOINT_INTERVAL,

    'persistent_checkpoint_frequency':
        'epoch_boundary',

    'maximum_uncommitted_training':
        'one_epoch',

    'training_configuration_finalized': False,
    'full_model_smoke_test_completed': False,
    'checkpoint_resume_verified': False
})

storage_policy.update({
    'frequent_checkpoints_on_local_disk': True,
    'persistent_checkpoint_at_epoch_boundary': True,
    'local_checkpoint_lost_on_runtime_reset': True,
    'verify_persistent_checkpoint_sha256': True,
    'prune_only_after_successful_verification': True
})


# Verify matching controlled configurations

run_signatures = {
    run_id: get_run_signature(run_id)
    for run_id in ROBERTA_RUN_IDS
}

common_signature = None

for run_id, signature in run_signatures.items():

    controlled_signature = {
        key: value
        for key, value in signature.items()
        if key not in [
            'run_id',
            'loss_configuration'
        ]
    }

    if common_signature is None:

        common_signature = controlled_signature

    else:

        assert controlled_signature == common_signature

print("\nControlled RoBERTa signatures: Matched")


# Save the consolidated pipeline configuration

PIPELINE_CONFIG_PATH = (
    RUN_CONFIGS_DIR
    / 'roberta_training_pipeline.json'
)

pipeline_configuration = {
    'model': MODEL_NAME,
    'checkpoint': MODEL_CHECKPOINT,
    'revision': MODEL_REVISION,

    'labels': LABELS,
    'seed': SEED,

    'training_samples': TRAINING_SAMPLES,
    'validation_samples': VALIDATION_SAMPLES,

    'training_dataset_sha256': EXPECTED_TRAINING_HASH,
    'validation_dataset_sha256': EXPECTED_VALIDATION_HASH,

    'initial_model_path': str(
        INITIAL_MODEL_DIR.relative_to(PROJECT_ROOT)
    ),

    'initial_model_state_sha256': INITIAL_STATE_HASH,

    'loss_configuration_sha256': LOSS_CONFIG_HASH,
    'evaluation_module_sha256': evaluation_hash,

    'batches_per_epoch': BATCHES_PER_EPOCH,
    'updates_per_epoch': UPDATES_PER_EPOCH,
    'total_training_steps': TOTAL_TRAINING_STEPS,
    'warmup_steps': WARMUP_STEPS,

    'final_accumulation_size': FINAL_ACCUMULATION_SIZE,

    'training_policy': training_policy.copy(),
    'storage_policy': storage_policy.copy(),

    'run_signatures': run_signatures,

    'training_time_excludes_validation': True,
    'training_time_excludes_checkpoint_writing': True,

    'training_configuration_finalized': False,
    'full_model_smoke_test_completed': False,
    'checkpoint_resume_verified': False,

    'official_test_used': False,
    'model_training_performed': False
}

if PIPELINE_CONFIG_PATH.exists():

    previous_pipeline = read_json(
        PIPELINE_CONFIG_PATH
    )

    assert previous_pipeline['run_signatures'] == (
        run_signatures
    ), "Existing pipeline signature differs."

    if previous_pipeline[
        'training_configuration_finalized'
    ]:

        print(
            "Existing finalized configuration preserved."
        )

        pipeline_configuration = previous_pipeline

        training_policy.update(
            previous_pipeline['training_policy']
        )

    else:

        save_json_atomic(
            PIPELINE_CONFIG_PATH,
            pipeline_configuration
        )

else:

    save_json_atomic(
        PIPELINE_CONFIG_PATH,
        pipeline_configuration
    )


# Verify previous research records remain unchanged

assert calculate_sha256(REGISTRY_PATH) == (
    registry_original_hash
)

assert calculate_sha256(DECISION_LOG_PATH) == (
    decision_log_original_hash
)

assert calculate_sha256(LOSS_CONFIG_PATH) == (
    LOSS_CONFIG_HASH
)

assert calculate_sha256(TRAIN_PATH) == (
    EXPECTED_TRAINING_HASH
)

assert calculate_sha256(VALIDATION_PATH) == (
    EXPECTED_VALIDATION_HASH
)


# Display completion summary

print("\nRoBERTa training pipeline summary")

print("Physical batch size:", TRAIN_BATCH_SIZE)

print(
    "Gradient accumulation:",
    GRAD_ACCUMULATION_STEPS
)

print("Effective batch size:", EFFECTIVE_BATCH_SIZE)
print("Validation batch size:", EVAL_BATCH_SIZE)

print("\nUpdates per epoch:", UPDATES_PER_EPOCH)
print("Maximum training steps:", TOTAL_TRAINING_STEPS)
print("Warmup steps:", WARMUP_STEPS)
print("Final accumulation size:", FINAL_ACCUMULATION_SIZE)

print("\nOptimizer: AdamW")
print("Learning rate:", LEARNING_RATE)
print("Mixed precision: FP16")
print("Loss computation: FP32")

print("\nIndependent model loader: Ready")
print("Training components: Defined")
print("Deterministic sample ordering: Verified")
print("Gradient accumulation: Defined")
print("Validation inference: Defined")
print("Checkpoint save and restore: Defined")
print("Best model persistence: Defined")

print("\nLocal checkpoint interval:", CHECKPOINT_INTERVAL)
print("Persistent checkpoints: Epoch boundaries")

print(
    "Training configuration finalized:",
    pipeline_configuration[
        'training_configuration_finalized'
    ]
)

print("\nExperiment signatures: Saved")
print("Experiment registry: Unchanged")
print("Decision log: Unchanged")
print("Additional figures generated: 0")
print("Model training performed: No")
print("Official test dataset: Not loaded")

gc.collect()
torch.cuda.empty_cache()

print("\nCell 5 completed successfully.")

In [ ]:
import copy

# Verify the training pipeline before smoke testing

assert 'create_training_components' in globals()
assert 'train_one_epoch' in globals()
assert 'save_resume_checkpoint' in globals()
assert 'load_resume_checkpoint' in globals()

pipeline_configuration = read_json(
    PIPELINE_CONFIG_PATH
)

assert pipeline_configuration['run_signatures'] == {
    run_id: get_run_signature(run_id)
    for run_id in ROBERTA_RUN_IDS
}

assert calculate_sha256(LOSS_CONFIG_PATH) == LOSS_CONFIG_HASH

assert not registry['run_id'].isin(
    ROBERTA_RUN_IDS
).any(), "Full RoBERTa experiments already registered."


# Prepare small development subsets for diagnostic testing

SMOKE_TRAIN_SIZE = 93
SMOKE_VAL_SIZE = 65

SMOKE_TOTAL_STEPS = math.ceil(
    SMOKE_TRAIN_SIZE / EFFECTIVE_BATCH_SIZE
)

SMOKE_WARMUP_STEPS = 1

rng = np.random.default_rng(SEED)

smoke_train_indices = rng.choice(
    TRAINING_SAMPLES,
    size=SMOKE_TRAIN_SIZE,
    replace=False
)

smoke_val_indices = rng.choice(
    VALIDATION_SAMPLES,
    size=SMOKE_VAL_SIZE,
    replace=False
)

smoke_train_dataset = ToxicCommentDataset(
    [
        train_dataset.input_ids[int(index)]
        for index in smoke_train_indices
    ],
    y_train[smoke_train_indices]
)

smoke_val_dataset = ToxicCommentDataset(
    [
        val_dataset.input_ids[int(index)]
        for index in smoke_val_indices
    ],
    y_val[smoke_val_indices]
)

smoke_y_val = y_val[smoke_val_indices]

assert SMOKE_TOTAL_STEPS == 3

assert (
    SMOKE_TRAIN_SIZE % EFFECTIVE_BATCH_SIZE
) == 29

print("Smoke training samples:", SMOKE_TRAIN_SIZE)
print("Smoke validation samples:", SMOKE_VAL_SIZE)
print("Expected optimizer updates:", SMOKE_TOTAL_STEPS)
print("Final accumulation group: 29")


# Define fingerprints for nested training states

def state_fingerprint(state):

    digest = hashlib.sha256()

    def update(value):

        if isinstance(value, torch.Tensor):

            tensor = value.detach().cpu().contiguous()

            digest.update(b'tensor')
            digest.update(str(tensor.dtype).encode())
            digest.update(str(tuple(tensor.shape)).encode())
            digest.update(tensor.numpy().tobytes())

        elif isinstance(value, np.ndarray):

            digest.update(b'ndarray')
            digest.update(str(value.dtype).encode())
            digest.update(str(value.shape).encode())
            digest.update(value.tobytes())

        elif isinstance(value, dict):

            digest.update(b'dict')
            digest.update(str(len(value)).encode())

            for key in sorted(value, key=repr):

                update(key)
                update(value[key])

        elif isinstance(value, (list, tuple)):

            digest.update(
                type(value).__name__.encode()
            )

            digest.update(str(len(value)).encode())

            for item in value:
                update(item)

        else:

            digest.update(
                type(value).__name__.encode()
            )

            digest.update(repr(value).encode())
            digest.update(b'\x00')

    update(state)

    return digest.hexdigest()


# Exclude wall-clock timing from progress comparisons

def comparable_progress(progress):

    return {
        key: value
        for key, value in progress.items()
        if key != 'training_time_seconds'
    }


def capture_state_fingerprints(
    model,
    optimizer,
    scheduler,
    scaler,
    progress
):

    return {
        'Model': model_state_sha256(model),

        'Optimizer': state_fingerprint(
            optimizer.state_dict()
        ),

        'Scheduler': state_fingerprint(
            scheduler.state_dict()
        ),

        'AMP Scaler': state_fingerprint(
            scaler.state_dict()
        ),

        'RNG': state_fingerprint(
            capture_rng_state()
        ),

        'Training Progress': state_fingerprint(
            comparable_progress(progress)
        )
    }


# Define smoke-specific checkpoint signatures

def get_smoke_signature(run_id):

    signature = get_run_signature(run_id)

    signature.update({
        'smoke_test': True,
        'smoke_training_samples': SMOKE_TRAIN_SIZE,
        'smoke_validation_samples': SMOKE_VAL_SIZE,
        'smoke_total_steps': SMOKE_TOTAL_STEPS,
        'smoke_warmup_steps': SMOKE_WARMUP_STEPS,

        'smoke_train_indices_sha256':
            state_fingerprint(smoke_train_indices),

        'smoke_val_indices_sha256':
            state_fingerprint(smoke_val_indices)
    })

    return signature


# Define numerical diagnostics for state differences

def compare_tensor_states(reference, current):

    diagnostics = {
        'tensors_compared': 0,
        'different_tensors': 0,
        'maximum_absolute_difference': 0.0,
        'structure_matched': True
    }

    def compare(left, right):

        if isinstance(left, torch.Tensor):

            if not isinstance(right, torch.Tensor):

                diagnostics['structure_matched'] = False
                return

            left = left.detach().cpu()
            right = right.detach().cpu()

            diagnostics['tensors_compared'] += 1

            if (
                left.shape != right.shape
                or left.dtype != right.dtype
            ):

                diagnostics['structure_matched'] = False
                return

            if not torch.equal(left, right):

                diagnostics['different_tensors'] += 1

                if left.numel() > 0:

                    difference = (
                        left.double() - right.double()
                    ).abs().max().item()

                    diagnostics[
                        'maximum_absolute_difference'
                    ] = max(
                        diagnostics[
                            'maximum_absolute_difference'
                        ],
                        difference
                    )

        elif isinstance(left, dict):

            if (
                not isinstance(right, dict)
                or set(left) != set(right)
            ):

                diagnostics['structure_matched'] = False
                return

            for key in left:

                compare(left[key], right[key])

        elif isinstance(left, (tuple, list)):

            if (
                not isinstance(right, type(left))
                or len(left) != len(right)
            ):

                diagnostics['structure_matched'] = False
                return

            for left_item, right_item in zip(left, right):

                compare(left_item, right_item)

        elif left != right:

            diagnostics['structure_matched'] = False

    compare(reference, current)

    return diagnostics


# Execute a complete smoke training run

def run_smoke_test(run_id, reference_path=None):

    loss_name = LOSS_CONFIGS[run_id]['loss_name']

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    model, optimizer, scheduler, scaler = (
        create_training_components(
            total_steps=SMOKE_TOTAL_STEPS,
            warmup_steps=SMOKE_WARMUP_STEPS
        )
    )

    progress = new_training_progress()

    initial_hash = model_state_sha256(model)

    assert initial_hash == INITIAL_STATE_HASH

    print(
        f"\nRunning {loss_name} smoke test...",
        flush=True
    )

    result = train_one_epoch(
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        progress=progress,
        loss_name=loss_name,
        dataset=smoke_train_dataset,
        verbose=False
    )

    assert result['epoch_completed']

    assert progress['next_row'] == SMOKE_TRAIN_SIZE
    assert progress['epoch_examples'] == SMOKE_TRAIN_SIZE

    assert progress['attempted_updates'] == 3

    assert progress['successful_updates'] == 3, (
        f"{loss_name}: AMP skipped an optimizer update."
    )

    assert progress['skipped_updates'] == 0

    assert np.isfinite(result['training_loss'])

    assert result['training_loss'] > 0

    final_hash = model_state_sha256(model)

    assert final_hash != initial_hash

    assert all(
        parameter.grad is None
        for parameter in model.parameters()
    )

    # Save the uninterrupted BCE state for recovery comparison

    reference_fingerprints = None

    if reference_path is not None:

        reference_fingerprints = (
            capture_state_fingerprints(
                model,
                optimizer,
                scheduler,
                scaler,
                progress
            )
        )

        save_resume_checkpoint(
            path=reference_path,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            progress=progress,
            run_signature=get_smoke_signature(run_id)
        )

    # Verify full-model validation inference

    evaluation = evaluate_validation_model(
        model=model,
        loss_name=loss_name,
        dataset=smoke_val_dataset,
        targets=smoke_y_val,
        compute_metrics=False
    )

    assert evaluation['logits'].shape == (
        SMOKE_VAL_SIZE, NUM_LABELS
    )

    assert evaluation['scores'].shape == (
        SMOKE_VAL_SIZE, NUM_LABELS
    )

    assert np.isfinite(
        evaluation['logits']
    ).all()

    assert np.isfinite(
        evaluation['scores']
    ).all()

    assert (
        (evaluation['scores'] >= 0)
        & (evaluation['scores'] <= 1)
    ).all()

    assert np.isfinite(
        evaluation['validation_loss']
    )

    record = {
        'Run ID': run_id,
        'Loss': loss_name,
        'Training Samples': SMOKE_TRAIN_SIZE,
        'Validation Samples': SMOKE_VAL_SIZE,
        'Attempted Updates': progress['attempted_updates'],
        'Successful Updates': progress['successful_updates'],
        'Skipped Updates': progress['skipped_updates'],
        'Training Loss': float(result['training_loss']),
        'Validation Loss': float(
            evaluation['validation_loss']
        ),

        'Training Time (s)':
            float(progress['training_time_seconds']),

        'Peak Allocated GPU Memory (GiB)':
            torch.cuda.max_memory_allocated() / 1024**3,

        'Peak Reserved GPU Memory (GiB)':
            torch.cuda.max_memory_reserved() / 1024**3,

        'Initial Weights Matched': True,
        'Model Weights Updated': True,
        'Validation Output Shape': [SMOKE_VAL_SIZE, NUM_LABELS],
        'Finite Validation Scores': True,
        'Verification': 'Passed'
    }

    print(
        f"{loss_name}: "
        f"3/3 successful updates | "
        f"Training loss: {result['training_loss']:.6f}",
        flush=True
    )

    del model, optimizer, scheduler, scaler
    del evaluation, progress

    gc.collect()
    torch.cuda.empty_cache()

    return record, reference_fingerprints


# Execute checkpoint interruption and restoration

def run_checkpoint_recovery_test(
    reference_path,
    reference_fingerprints,
    recovery_path
):

    run_id = 'T04_BCE_S42'

    signature = get_smoke_signature(run_id)

    print(
        "\nTesting interrupted RoBERTa training...",
        flush=True
    )

    model, optimizer, scheduler, scaler = (
        create_training_components(
            total_steps=SMOKE_TOTAL_STEPS,
            warmup_steps=SMOKE_WARMUP_STEPS
        )
    )

    progress = new_training_progress()

    partial_result = train_one_epoch(
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        progress=progress,
        loss_name='BCE',
        dataset=smoke_train_dataset,
        stop_after_updates=2,
        verbose=False
    )

    assert not partial_result['epoch_completed']

    assert progress['next_row'] == 64
    assert progress['attempted_updates'] == 2
    assert progress['successful_updates'] == 2
    assert progress['skipped_updates'] == 0

    before_save = capture_state_fingerprints(
        model,
        optimizer,
        scheduler,
        scaler,
        progress
    )

    checkpoint_record = save_resume_checkpoint(
        path=recovery_path,
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        progress=progress,
        run_signature=signature
    )

    assert calculate_sha256(recovery_path) == (
        checkpoint_record['sha256']
    )

    print("Persistent checkpoint saved and verified.")
    print("Interrupted after 64 training comments.")

    del model, optimizer, scheduler, scaler
    del progress

    gc.collect()
    torch.cuda.empty_cache()

    # Recreate every training component before loading

    model, optimizer, scheduler, scaler = (
        create_training_components(
            total_steps=SMOKE_TOTAL_STEPS,
            warmup_steps=SMOKE_WARMUP_STEPS
        )
    )

    progress = load_resume_checkpoint(
        path=recovery_path,
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        run_signature=signature
    )

    after_restore = capture_state_fingerprints(
        model,
        optimizer,
        scheduler,
        scaler,
        progress
    )

    restore_records = [
        {
            'Component': component,
            'Exact Restore Match':
                before_save[component]
                == after_restore[component]
        }
        for component in before_save
    ]

    restore_passed = all(
        item['Exact Restore Match']
        for item in restore_records
    )

    print("\nImmediate checkpoint restoration")

    display(pd.DataFrame(restore_records))

    if not restore_passed:

        del model, optimizer, scheduler, scaler
        del progress

        gc.collect()
        torch.cuda.empty_cache()

        return {
            'checkpoint_size_mib':
                checkpoint_record['size_bytes'] / 1024**2,

            'immediate_restore_records': restore_records,
            'immediate_restore_passed': False,
            'continuation_records': [],
            'continuation_passed': False,
            'diagnostics': {},
            'failure_stage': 'checkpoint_restoration'
        }

    assert progress['next_row'] == 64

    print("\nContinuing remaining 29 comments...")

    resumed_result = train_one_epoch(
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        progress=progress,
        loss_name='BCE',
        dataset=smoke_train_dataset,
        verbose=False
    )

    assert resumed_result['epoch_completed']

    assert progress['next_row'] == SMOKE_TRAIN_SIZE
    assert progress['attempted_updates'] == 3
    assert progress['successful_updates'] == 3
    assert progress['skipped_updates'] == 0

    resumed_fingerprints = capture_state_fingerprints(
        model,
        optimizer,
        scheduler,
        scaler,
        progress
    )

    continuation_records = [
        {
            'Component': component,

            'Uninterrupted vs Resumed Exact Match':
                reference_fingerprints[component]
                == resumed_fingerprints[component]
        }
        for component in reference_fingerprints
    ]

    continuation_passed = all(
        item['Uninterrupted vs Resumed Exact Match']
        for item in continuation_records
    )

    print("\nUninterrupted versus resumed training")

    display(pd.DataFrame(continuation_records))

    # Inspect numerical differences if exact equality fails

    diagnostics = {}

    if not continuation_passed:

        reference_checkpoint = torch.load(
            reference_path,
            map_location='cpu',
            weights_only=False
        )

        diagnostics = {
            'model_state': compare_tensor_states(
                reference_checkpoint['model_state'],
                model.state_dict()
            ),

            'optimizer_state': compare_tensor_states(
                reference_checkpoint['optimizer_state'],
                optimizer.state_dict()
            ),

            'training_loss_sum_difference': abs(
                reference_checkpoint[
                    'progress'
                ]['epoch_loss_sum']
                - progress['epoch_loss_sum']
            )
        }

        del reference_checkpoint

        print("\nNumerical difference diagnostics")

        print(json.dumps(
            diagnostics,
            indent=4
        ))

    recovery_result = {
        'checkpoint_size_mib':
            checkpoint_record['size_bytes'] / 1024**2,

        'interrupted_after_rows': 64,
        'resumed_remaining_rows': 29,

        'immediate_restore_records': restore_records,
        'immediate_restore_passed': restore_passed,

        'continuation_records': continuation_records,
        'continuation_passed': continuation_passed,

        'diagnostics': diagnostics,

        'failure_stage':
            None if continuation_passed
            else 'subsequent_training_reproducibility'
    }

    del model, optimizer, scheduler, scaler
    del progress

    gc.collect()
    torch.cuda.empty_cache()

    return recovery_result


# Execute all smoke tests using temporary checkpoints

SMOKE_SUMMARY_PATH = (
    LOGS_DIR / 'roberta_smoke_test_summary.json'
)

smoke_records = []

with tempfile.TemporaryDirectory(
    prefix='roberta_smoke_local_',
    dir='/content'
) as local_directory:

    with tempfile.TemporaryDirectory(
        prefix='roberta_smoke_drive_',
        dir=MODELS_DIR
    ) as drive_directory:

        reference_path = (
            Path(local_directory)
            / 'uninterrupted_reference.pt'
        )

        recovery_path = (
            Path(drive_directory)
            / 'recovery_checkpoint.pt'
        )

        # Test standard BCE and retain a temporary reference

        bce_record, reference_fingerprints = (
            run_smoke_test(
                'T04_BCE_S42',
                reference_path=reference_path
            )
        )

        smoke_records.append(bce_record)

        # Test exact restoration and continuation

        recovery_result = (
            run_checkpoint_recovery_test(
                reference_path,
                reference_fingerprints,
                recovery_path
            )
        )

        # Remove diagnostic checkpoints after comparison

        reference_path.unlink()
        recovery_path.unlink()

        # Verify the remaining two loss functions

        for run_id in [
            'T05_WBCE_S42',
            'T06_FOCAL_S42'
        ]:

            record, _ = run_smoke_test(run_id)

            smoke_records.append(record)


# Summarize the full-model diagnostic tests

smoke_test_df = pd.DataFrame(smoke_records)

assert len(smoke_test_df) == 3

assert smoke_test_df['Verification'].eq(
    'Passed'
).all()

assert smoke_test_df['Successful Updates'].eq(3).all()
assert smoke_test_df['Skipped Updates'].eq(0).all()

print("\nFull-model smoke test results")

display(smoke_test_df.round(6))


# Determine whether the training configuration can be finalized

all_loss_tests_passed = True

checkpoint_restore_passed = (
    recovery_result['immediate_restore_passed']
)

checkpoint_continuation_passed = (
    recovery_result['continuation_passed']
)

all_smoke_tests_passed = (
    all_loss_tests_passed
    and checkpoint_restore_passed
    and checkpoint_continuation_passed
)

maximum_allocated_memory = float(
    smoke_test_df[
        'Peak Allocated GPU Memory (GiB)'
    ].max()
)

maximum_reserved_memory = float(
    smoke_test_df[
        'Peak Reserved GPU Memory (GiB)'
    ].max()
)


# Save one consolidated smoke test record

smoke_test_summary = {
    'model': MODEL_NAME,
    'seed': SEED,
    'initial_model_state_sha256': INITIAL_STATE_HASH,

    'training_samples': SMOKE_TRAIN_SIZE,
    'validation_samples': SMOKE_VAL_SIZE,

    'physical_batch_size': TRAIN_BATCH_SIZE,
    'effective_batch_size': EFFECTIVE_BATCH_SIZE,
    'gradient_accumulation_steps': GRAD_ACCUMULATION_STEPS,

    'expected_updates_per_smoke_run': SMOKE_TOTAL_STEPS,
    'final_accumulation_group': 29,

    'mixed_precision': 'fp16',

    'smoke_train_indices':
        smoke_train_indices.tolist(),

    'smoke_val_indices':
        smoke_val_indices.tolist(),

    'smoke_results': smoke_records,
    'checkpoint_recovery': recovery_result,

    'all_loss_smoke_tests_passed':
        all_loss_tests_passed,

    'checkpoint_restore_passed':
        checkpoint_restore_passed,

    'checkpoint_continuation_passed':
        checkpoint_continuation_passed,

    'maximum_allocated_gpu_memory_gib':
        maximum_allocated_memory,

    'maximum_reserved_gpu_memory_gib':
        maximum_reserved_memory,

    'training_configuration_finalized':
        all_smoke_tests_passed,

    'temporary_checkpoints_removed': True,

    'full_experiment_training_performed': False,

    'official_test_used': False
}

save_json_atomic(
    SMOKE_SUMMARY_PATH,
    smoke_test_summary
)

SMOKE_SUMMARY_HASH = calculate_sha256(
    SMOKE_SUMMARY_PATH
)


# Finalize the training policy only if all checks pass

training_policy.update({
    'training_configuration_finalized':
        all_smoke_tests_passed,

    'full_model_smoke_test_completed': True,

    'checkpoint_resume_verified':
        checkpoint_restore_passed
        and checkpoint_continuation_passed
})

pipeline_configuration.update({
    'training_policy': training_policy.copy(),

    'training_configuration_finalized':
        all_smoke_tests_passed,

    'full_model_smoke_test_completed': True,

    'checkpoint_resume_verified':
        checkpoint_restore_passed
        and checkpoint_continuation_passed,

    'smoke_test_summary_path': str(
        SMOKE_SUMMARY_PATH.relative_to(PROJECT_ROOT)
    ),

    'smoke_test_summary_sha256':
        SMOKE_SUMMARY_HASH
})

save_json_atomic(
    PIPELINE_CONFIG_PATH,
    pipeline_configuration
)


# Verify previous research records remain unchanged

assert calculate_sha256(REGISTRY_PATH) == (
    registry_original_hash
)

assert calculate_sha256(DECISION_LOG_PATH) == (
    decision_log_original_hash
)

assert calculate_sha256(LOSS_CONFIG_PATH) == (
    LOSS_CONFIG_HASH
)

assert calculate_sha256(TRAIN_PATH) == (
    EXPECTED_TRAINING_HASH
)

assert calculate_sha256(VALIDATION_PATH) == (
    EXPECTED_VALIDATION_HASH
)

verify_model_files(
    INITIAL_MODEL_DIR,
    initial_manifest
)


# Display final verification summary

print("\nRoBERTa smoke test summary")

print("Loss configurations tested: 3")

print(
    "Successful updates across loss tests:",
    int(smoke_test_df['Successful Updates'].sum())
)

print(
    "Skipped updates across loss tests:",
    int(smoke_test_df['Skipped Updates'].sum())
)

print("\nFP16 full-model training: Passed")
print("Incomplete accumulation group: Passed")
print("Validation inference: Passed")

print(
    "Immediate checkpoint restoration:",
    "Passed" if checkpoint_restore_passed else "Failed"
)

print(
    "Uninterrupted versus resumed training:",
    "Exact match"
    if checkpoint_continuation_passed
    else "Mismatch"
)

print(
    "\nCheckpoint size (MiB):",
    round(
        recovery_result['checkpoint_size_mib'],
        4
    )
)

print(
    "Peak allocated GPU memory (GiB):",
    round(maximum_allocated_memory, 4)
)

print(
    "Peak reserved GPU memory (GiB):",
    round(maximum_reserved_memory, 4)
)

print("\nTemporary smoke checkpoints: Removed")
print("Shared initialization: Preserved")
print("Smoke test metadata: Saved")
print("Experiment registry: Unchanged")
print("Decision log: Unchanged")

print("\nAdditional figures generated: 0")
print("Full experiment training performed: No")
print("Official test dataset: Not loaded")

print(
    "\nTraining configuration finalized:",
    all_smoke_tests_passed
)

gc.collect()
torch.cuda.empty_cache()

if all_smoke_tests_passed:

    print("\nCell 6 completed successfully.")
    print("RoBERTa full training is ready.")

else:

    print("\nCell 6 verification requires investigation.")
    print("Do not start full training yet.")

    raise RuntimeError(
        "Checkpoint verification gate did not pass."
    )

In [ ]:
# Verify the original smoke test evidence

original_smoke_summary = read_json(SMOKE_SUMMARY_PATH)

original_smoke_hash = calculate_sha256(
    SMOKE_SUMMARY_PATH
)

pipeline_configuration = read_json(
    PIPELINE_CONFIG_PATH
)

assert pipeline_configuration[
    'smoke_test_summary_sha256'
] == original_smoke_hash

assert original_smoke_summary[
    'all_loss_smoke_tests_passed'
]

assert original_smoke_summary[
    'checkpoint_restore_passed'
]

assert not original_smoke_summary[
    'checkpoint_continuation_passed'
]

assert not registry['run_id'].isin(
    ROBERTA_RUN_IDS
).any()

print("Original smoke test evidence verified.")


# Define numerical acceptance tolerances before the new test

RECOVERY_TOLERANCES = {
    'model_absolute_difference': 1e-5,
    'optimizer_absolute_difference': 1e-5,
    'mean_training_loss_difference': 1e-5,
    'validation_logit_difference': 2e-3,
    'validation_probability_difference': 5e-4
}

print("\nNumerical recovery tolerances")

display(pd.DataFrame(
    list(RECOVERY_TOLERANCES.items()),
    columns=['Component', 'Maximum Allowed Difference']
))


# Compare nested states without loading multiple GPU models

def compare_recovery_states(reference, current):

    result = {
        'structure_matched': True,
        'nonfloating_states_exact': True,
        'finite_values': True,
        'tensors_compared': 0,
        'different_tensors': 0,
        'maximum_absolute_difference': 0.0
    }

    def compare(left, right, path='root'):

        if isinstance(left, torch.Tensor):

            if not isinstance(right, torch.Tensor):

                result['structure_matched'] = False
                return

            if (
                left.shape != right.shape
                or left.dtype != right.dtype
            ):

                result['structure_matched'] = False
                return

            left = left.detach().cpu().reshape(-1)
            right = right.detach().cpu().reshape(-1)

            result['tensors_compared'] += 1

            if not torch.equal(left, right):

                result['different_tensors'] += 1

            if (
                not left.is_floating_point()
                or path.endswith('.step')
            ):

                if not torch.equal(left, right):

                    result['nonfloating_states_exact'] = False

                if left.is_floating_point():

                    if (
                        not torch.isfinite(left).all()
                        or not torch.isfinite(right).all()
                    ):

                        result['finite_values'] = False

                return

            # Process large tensors in chunks to limit RAM usage

            chunk_size = 1_000_000

            for start in range(
                0, left.numel(), chunk_size
            ):

                left_chunk = left[
                    start:start + chunk_size
                ].double()

                right_chunk = right[
                    start:start + chunk_size
                ].double()

                if (
                    not torch.isfinite(left_chunk).all()
                    or not torch.isfinite(right_chunk).all()
                ):

                    result['finite_values'] = False
                    continue

                difference = (
                    left_chunk - right_chunk
                ).abs().max().item()

                result['maximum_absolute_difference'] = max(
                    result['maximum_absolute_difference'],
                    difference
                )

        elif isinstance(left, dict):

            if (
                not isinstance(right, dict)
                or set(left) != set(right)
            ):

                result['structure_matched'] = False
                return

            for key in left:

                compare(
                    left[key],
                    right[key],
                    f'{path}.{key}'
                )

        elif isinstance(left, (list, tuple)):

            if (
                not isinstance(right, type(left))
                or len(left) != len(right)
            ):

                result['structure_matched'] = False
                return

            for index, (a, b) in enumerate(
                zip(left, right)
            ):

                compare(a, b, f'{path}.{index}')

        elif left != right:

            result['nonfloating_states_exact'] = False

    compare(reference, current)

    return result


# Compare discrete progress separately from floating-point loss

def discrete_training_progress(progress):

    return {
        key: value
        for key, value in progress.items()
        if key not in [
            'training_time_seconds',
            'epoch_loss_sum'
        ]
    }


# Run a shared-prefix checkpoint recovery test

def run_shared_prefix_recovery_test():

    run_id = 'T04_BCE_S42'

    signature = get_smoke_signature(run_id)

    gc.collect()
    torch.cuda.empty_cache()

    with tempfile.TemporaryDirectory(
        prefix='roberta_recovery_investigation_',
        dir='/content'
    ) as directory:

        directory = Path(directory)

        prefix_path = directory / 'shared_prefix.pt'
        reference_path = directory / 'direct_final.pt'

        # Train the common prefix exactly once

        model, optimizer, scheduler, scaler = (
            create_training_components(
                total_steps=SMOKE_TOTAL_STEPS,
                warmup_steps=SMOKE_WARMUP_STEPS
            )
        )

        progress = new_training_progress()

        assert model_state_sha256(model) == (
            INITIAL_STATE_HASH
        )

        print(
            "\nTraining shared intermediate state...",
            flush=True
        )

        prefix_result = train_one_epoch(
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            progress=progress,
            loss_name='BCE',
            dataset=smoke_train_dataset,
            stop_after_updates=2,
            verbose=False
        )

        assert not prefix_result['epoch_completed']

        assert progress['next_row'] == 64
        assert progress['attempted_updates'] == 2
        assert progress['successful_updates'] == 2
        assert progress['skipped_updates'] == 0

        prefix_fingerprints = capture_state_fingerprints(
            model,
            optimizer,
            scheduler,
            scaler,
            progress
        )

        checkpoint_record = save_resume_checkpoint(
            path=prefix_path,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            progress=progress,
            run_signature=signature
        )

        assert calculate_sha256(prefix_path) == (
            checkpoint_record['sha256']
        )

        # Verify checkpoint saving did not change training state

        assert capture_state_fingerprints(
            model,
            optimizer,
            scheduler,
            scaler,
            progress
        ) == prefix_fingerprints

        print("Shared checkpoint saved after 64 comments.")


        # Branch A: Continue without loading the checkpoint

        print(
            "\nBranch A: Direct continuation...",
            flush=True
        )

        direct_result = train_one_epoch(
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            progress=progress,
            loss_name='BCE',
            dataset=smoke_train_dataset,
            verbose=False
        )

        assert direct_result['epoch_completed']
        assert progress['next_row'] == 93
        assert progress['attempted_updates'] == 3
        assert progress['successful_updates'] == 3
        assert progress['skipped_updates'] == 0

        direct_fingerprints = capture_state_fingerprints(
            model,
            optimizer,
            scheduler,
            scaler,
            progress
        )

        direct_progress = discrete_training_progress(
            progress
        )

        direct_mean_loss = float(
            direct_result['training_loss']
        )

        save_resume_checkpoint(
            path=reference_path,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            progress=progress,
            run_signature=signature
        )

        direct_evaluation = evaluate_validation_model(
            model=model,
            loss_name='BCE',
            dataset=smoke_val_dataset,
            targets=smoke_y_val,
            compute_metrics=False
        )

        print("Branch A completed.")

        del model, optimizer, scheduler, scaler
        del progress

        gc.collect()
        torch.cuda.empty_cache()


        # Branch B: Recreate components and restore the shared prefix

        print(
            "\nBranch B: Restored continuation...",
            flush=True
        )

        model, optimizer, scheduler, scaler = (
            create_training_components(
                total_steps=SMOKE_TOTAL_STEPS,
                warmup_steps=SMOKE_WARMUP_STEPS
            )
        )

        progress = load_resume_checkpoint(
            path=prefix_path,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            run_signature=signature
        )

        restored_fingerprints = (
            capture_state_fingerprints(
                model,
                optimizer,
                scheduler,
                scaler,
                progress
            )
        )

        restoration_records = [
            {
                'Component': component,
                'Exact Match': (
                    prefix_fingerprints[component]
                    == restored_fingerprints[component]
                )
            }
            for component in prefix_fingerprints
        ]

        print("\nShared-prefix restoration")

        display(pd.DataFrame(restoration_records))

        exact_restoration_passed = all(
            row['Exact Match']
            for row in restoration_records
        )

        assert exact_restoration_passed, (
            "Shared checkpoint was not restored exactly."
        )

        assert progress['next_row'] == 64

        resumed_result = train_one_epoch(
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            progress=progress,
            loss_name='BCE',
            dataset=smoke_train_dataset,
            verbose=False
        )

        assert resumed_result['epoch_completed']
        assert progress['next_row'] == 93
        assert progress['attempted_updates'] == 3
        assert progress['successful_updates'] == 3
        assert progress['skipped_updates'] == 0

        resumed_fingerprints = (
            capture_state_fingerprints(
                model,
                optimizer,
                scheduler,
                scaler,
                progress
            )
        )

        resumed_progress = discrete_training_progress(
            progress
        )

        resumed_mean_loss = float(
            resumed_result['training_loss']
        )

        resumed_evaluation = evaluate_validation_model(
            model=model,
            loss_name='BCE',
            dataset=smoke_val_dataset,
            targets=smoke_y_val,
            compute_metrics=False
        )

        print("Branch B completed.")


        # Compare the discrete training states

        discrete_records = []

        for component in [
            'Scheduler',
            'AMP Scaler',
            'RNG'
        ]:

            discrete_records.append({
                'Component': component,
                'Exact Match': (
                    direct_fingerprints[component]
                    == resumed_fingerprints[component]
                )
            })

        discrete_records.append({
            'Component': 'Discrete training progress',
            'Exact Match': (
                direct_progress == resumed_progress
            )
        })

        discrete_states_passed = all(
            row['Exact Match']
            for row in discrete_records
        )

        print("\nDiscrete continuation states")

        display(pd.DataFrame(discrete_records))


        # Compare model and optimizer tensors

        reference_checkpoint = torch.load(
            reference_path,
            map_location='cpu',
            weights_only=False
        )

        model_diagnostics = compare_recovery_states(
            reference_checkpoint['model_state'],
            model.state_dict()
        )

        optimizer_diagnostics = compare_recovery_states(
            reference_checkpoint['optimizer_state'],
            optimizer.state_dict()
        )

        del reference_checkpoint

        gc.collect()


        # Compare losses and validation predictions

        mean_loss_difference = abs(
            direct_mean_loss - resumed_mean_loss
        )

        logit_difference = float(
            np.max(np.abs(
                direct_evaluation['logits']
                - resumed_evaluation['logits']
            ))
        )

        probability_difference = float(
            np.max(np.abs(
                direct_evaluation['scores']
                - resumed_evaluation['scores']
            ))
        )

        assert np.isfinite(
            direct_evaluation['scores']
        ).all()

        assert np.isfinite(
            resumed_evaluation['scores']
        ).all()

        measured_differences = {
            'model_absolute_difference':
                model_diagnostics[
                    'maximum_absolute_difference'
                ],

            'optimizer_absolute_difference':
                optimizer_diagnostics[
                    'maximum_absolute_difference'
                ],

            'mean_training_loss_difference':
                mean_loss_difference,

            'validation_logit_difference':
                logit_difference,

            'validation_probability_difference':
                probability_difference
        }

        numerical_records = []

        for name, difference in (
            measured_differences.items()
        ):

            tolerance = RECOVERY_TOLERANCES[name]

            numerical_records.append({
                'Component': name,
                'Observed Difference': float(difference),
                'Allowed Difference': tolerance,
                'Passed': bool(
                    np.isfinite(difference)
                    and difference <= tolerance
                )
            })

        print("\nNumerical continuation verification")

        display(pd.DataFrame(numerical_records))

        tensor_structure_passed = all(
            diagnostic['structure_matched']
            and diagnostic['nonfloating_states_exact']
            and diagnostic['finite_values']
            for diagnostic in [
                model_diagnostics,
                optimizer_diagnostics
            ]
        )

        numerical_continuation_passed = (
            discrete_states_passed
            and tensor_structure_passed
            and all(
                record['Passed']
                for record in numerical_records
            )
        )

        exact_continuation_match = all(
            direct_fingerprints[component]
            == resumed_fingerprints[component]
            for component in direct_fingerprints
        )

        result = {
            'shared_prefix_rows': 64,
            'remaining_rows': 29,

            'exact_restoration_passed':
                exact_restoration_passed,

            'restoration_records':
                restoration_records,

            'discrete_states_passed':
                discrete_states_passed,

            'discrete_state_records':
                discrete_records,

            'tensor_structure_passed':
                tensor_structure_passed,

            'numerical_continuation_passed':
                numerical_continuation_passed,

            'bitwise_continuation_match':
                exact_continuation_match,

            'model_diagnostics':
                model_diagnostics,

            'optimizer_diagnostics':
                optimizer_diagnostics,

            'numerical_records':
                numerical_records,

            'direct_mean_training_loss':
                direct_mean_loss,

            'resumed_mean_training_loss':
                resumed_mean_loss,

            'validation_logits_compared':
                SMOKE_VAL_SIZE * NUM_LABELS,

            'validation_probabilities_compared':
                SMOKE_VAL_SIZE * NUM_LABELS
        }

        del model, optimizer, scheduler, scaler
        del progress
        del direct_evaluation, resumed_evaluation

        gc.collect()
        torch.cuda.empty_cache()

    return result


# Execute the targeted recovery investigation

recovery_investigation = (
    run_shared_prefix_recovery_test()
)


# Verify that the original smoke evidence remains unchanged

assert calculate_sha256(SMOKE_SUMMARY_PATH) == (
    original_smoke_hash
)

assert calculate_sha256(REGISTRY_PATH) == (
    registry_original_hash
)

assert calculate_sha256(DECISION_LOG_PATH) == (
    decision_log_original_hash
)

assert calculate_sha256(LOSS_CONFIG_PATH) == (
    LOSS_CONFIG_HASH
)


# Determine whether the training configuration can be finalized

recovery_verified = bool(
    original_smoke_summary[
        'all_loss_smoke_tests_passed'
    ]
    and original_smoke_summary[
        'checkpoint_restore_passed'
    ]
    and recovery_investigation[
        'exact_restoration_passed'
    ]
    and recovery_investigation[
        'numerical_continuation_passed'
    ]
)


# Save the original failure and follow-up evidence separately

RECOVERY_RESOLUTION_PATH = (
    LOGS_DIR / 'roberta_smoke_test_resolution.json'
)

recovery_resolution = {
    'model': MODEL_NAME,

    'original_smoke_summary_path': str(
        SMOKE_SUMMARY_PATH.relative_to(PROJECT_ROOT)
    ),

    'original_smoke_summary_sha256':
        original_smoke_hash,

    'original_bitwise_continuation_passed':
        False,

    'original_failure_preserved':
        True,

    'original_loss_smoke_tests_passed':
        True,

    'original_persistent_checkpoint_restored_exactly':
        True,

    'investigation_method':
        'Shared-prefix direct versus restored continuation',

    'numerical_acceptance_tolerances':
        RECOVERY_TOLERANCES,

    'investigation_results':
        recovery_investigation,

    'possible_explanation': (
        'Floating-point or CUDA execution variability; '
        'exact cause not established.'
    ),

    'bitwise_reproducibility_established':
        False,

    'numerical_checkpoint_continuation_verified':
        recovery_verified,

    'training_hyperparameters_changed':
        False,

    'deterministic_algorithm_settings_changed':
        False,

    'training_configuration_finalized':
        recovery_verified,

    'full_experiment_training_performed':
        False,

    'official_test_used':
        False,

    'temporary_local_checkpoints_removed':
        True
}

save_json_atomic(
    RECOVERY_RESOLUTION_PATH,
    recovery_resolution
)

RECOVERY_RESOLUTION_HASH = calculate_sha256(
    RECOVERY_RESOLUTION_PATH
)


# Update the training gate using the new recovery evidence

training_policy.update({
    'training_configuration_finalized':
        recovery_verified,

    'full_model_smoke_test_completed':
        True,

    'checkpoint_resume_verified':
        recovery_verified,

    'checkpoint_restore_exact':
        True,

    'checkpoint_continuation_numerically_verified':
        recovery_verified,

    'bitwise_training_reproducibility_established':
        False
})

pipeline_configuration.update({
    'training_policy':
        training_policy.copy(),

    'training_configuration_finalized':
        recovery_verified,

    'full_model_smoke_test_completed':
        True,

    'checkpoint_resume_verified':
        recovery_verified,

    'checkpoint_restore_exact':
        True,

    'checkpoint_continuation_numerically_verified':
        recovery_verified,

    'bitwise_training_reproducibility_established':
        False,

    'smoke_test_resolution_path': str(
        RECOVERY_RESOLUTION_PATH.relative_to(
            PROJECT_ROOT
        )
    ),

    'smoke_test_resolution_sha256':
        RECOVERY_RESOLUTION_HASH
})

assert pipeline_configuration['run_signatures'] == {
    run_id: get_run_signature(run_id)
    for run_id in ROBERTA_RUN_IDS
}

save_json_atomic(
    PIPELINE_CONFIG_PATH,
    pipeline_configuration
)


# Display final investigation results

print("\nCheckpoint recovery investigation summary")

print("Original full-model smoke tests: Passed")
print("Original persistent checkpoint restoration: Exact")

print(
    "Shared-prefix restoration:",
    recovery_investigation['exact_restoration_passed']
)

print(
    "Discrete training states:",
    recovery_investigation['discrete_states_passed']
)

print(
    "Numerical continuation verified:",
    recovery_verified
)

print(
    "New test bitwise continuation match:",
    recovery_investigation['bitwise_continuation_match']
)

print(
    "Bitwise training reproducibility established: No"
)

print("\nTraining hyperparameters: Unchanged")
print("Original smoke test evidence: Preserved")
print("Recovery investigation evidence: Saved")
print("Temporary local checkpoints: Removed")
print("Experiment registry: Unchanged")
print("Decision log: Unchanged")
print("Additional figures generated: 0")
print("Official test dataset: Not loaded")

print(
    "\nTraining configuration finalized:",
    recovery_verified
)

gc.collect()
torch.cuda.empty_cache()

if recovery_verified:

    print("\nCell 6B completed successfully.")
    print("RoBERTa is ready for full training.")

else:

    raise RuntimeError(
        "Numerical recovery verification failed. "
        "Further investigation required."
    )

In [ ]:
# Verify the finalized training configuration

pipeline_configuration = read_json(
    PIPELINE_CONFIG_PATH
)

assert pipeline_configuration[
    'training_configuration_finalized'
]

assert pipeline_configuration[
    'checkpoint_resume_verified'
]

assert pipeline_configuration[
    'checkpoint_continuation_numerically_verified'
]

assert calculate_sha256(RECOVERY_RESOLUTION_PATH) == (
    pipeline_configuration['smoke_test_resolution_sha256']
)

assert calculate_sha256(SMOKE_SUMMARY_PATH) == (
    pipeline_configuration['smoke_test_summary_sha256']
)

assert pipeline_configuration['run_signatures'] == {
    run_id: get_run_signature(run_id)
    for run_id in ROBERTA_RUN_IDS
}

assert calculate_sha256(LOSS_CONFIG_PATH) == LOSS_CONFIG_HASH

print("Finalized training configuration verified.")
print("Full RoBERTa training authorized.")


# Define checkpoint paths and metadata

def checkpoint_metadata_path(path):

    return Path(path).with_suffix('.json')


def checkpoint_progress_key(progress):

    return [
        int(progress['attempted_updates']),
        int(progress['epoch']),
        int(progress['next_row'])
    ]


def persist_recovery_checkpoint(
    path,
    model,
    optimizer,
    scheduler,
    scaler,
    progress,
    signature
):

    record = save_resume_checkpoint(
        path,
        model,
        optimizer,
        scheduler,
        scaler,
        progress,
        signature
    )

    metadata = {
        'run_id': signature['run_id'],
        'run_signature': signature,
        'sha256': record['sha256'],
        'size_bytes': record['size_bytes'],
        'progress_key': checkpoint_progress_key(progress),
        'training_finished': progress['training_finished']
    }

    save_json_atomic(
        checkpoint_metadata_path(path),
        metadata
    )

    return metadata


# Locate the newest verified recovery checkpoint

def find_recovery_checkpoint(paths, signature):

    candidates = []
    invalid_paths = []

    for path in paths:

        path = Path(path)
        metadata_path = checkpoint_metadata_path(path)

        if not path.exists() and not metadata_path.exists():
            continue

        try:

            assert path.is_file()
            assert metadata_path.is_file()

            metadata = read_json(metadata_path)

            assert metadata['run_signature'] == signature

            assert path.stat().st_size == (
                metadata['size_bytes']
            )

            assert calculate_sha256(path) == (
                metadata['sha256']
            )

            candidates.append({
                'path': path,
                'metadata': metadata
            })

        except Exception as error:

            invalid_paths.append(str(path))

            print(
                f"Unusable recovery checkpoint: {path}\n"
                f"Reason: {error}"
            )

    if not candidates:

        assert not invalid_paths, (
            "Recovery files exist but none passed verification. "
            "Inspect them before restarting training."
        )

        return None

    return max(
        candidates,
        key=lambda item: (
            item['metadata']['progress_key']
        )
    )


# Verify selected model and validation score artifacts

def verify_selected_checkpoint(progress):

    assert progress['best_checkpoint_path'] is not None

    best_path = (
        PROJECT_ROOT / progress['best_checkpoint_path']
    )

    score_path = (
        PROJECT_ROOT / progress['best_validation_path']
    )

    manifest = read_json(
        best_path / 'checkpoint_manifest.json'
    )

    assert manifest['epoch'] == progress['best_epoch']

    assert manifest['initial_model_sha256'] == (
        INITIAL_STATE_HASH
    )

    verify_model_files(best_path, manifest)

    assert calculate_sha256(score_path) == (
        progress['best_validation_sha256']
    )

    return best_path, score_path, manifest


# Remove superseded models only after persistent checkpoint verification

def prune_superseded_models(
    run_model_dir,
    run_prediction_dir,
    progress,
    persistent_path,
    run_log_dir
):

    best_path, score_path, _ = (
        verify_selected_checkpoint(progress)
    )

    metadata = read_json(
        checkpoint_metadata_path(persistent_path)
    )

    assert calculate_sha256(persistent_path) == (
        metadata['sha256']
    )

    assert metadata['progress_key'] == (
        checkpoint_progress_key(progress)
    )

    pruning_path = (
        run_log_dir / 'checkpoint_pruning_history.json'
    )

    pruning_history = (
        read_json(pruning_path)
        if pruning_path.exists()
        else []
    )

    for old_path in sorted(
        run_model_dir.glob('best_epoch_*')
    ):

        if old_path == best_path:
            continue

        if not old_path.is_dir():
            continue

        if old_path.name.endswith('_staging'):
            continue

        old_manifest_path = (
            old_path / 'checkpoint_manifest.json'
        )

        assert old_manifest_path.is_file()

        old_manifest = read_json(old_manifest_path)

        assert old_manifest['initial_model_sha256'] == (
            INITIAL_STATE_HASH
        )

        verify_model_files(old_path, old_manifest)

        removed_bytes = sum(
            path.stat().st_size
            for path in old_path.rglob('*')
            if path.is_file()
        )

        record = {
            'removed_model': str(
                old_path.relative_to(PROJECT_ROOT)
            ),
            'removed_model_state_sha256':
                old_manifest['model_state_sha256'],
            'retained_model': str(
                best_path.relative_to(PROJECT_ROOT)
            ),
            'removed_bytes': removed_bytes,
            'timestamp': datetime.now(
                timezone.utc
            ).isoformat(),
            'status': 'Planned'
        }

        pruning_history.append(record)

        save_json_atomic(
            pruning_path,
            pruning_history
        )

        shutil.rmtree(old_path)

        assert not old_path.exists()

        record['status'] = 'Completed'

        save_json_atomic(
            pruning_path,
            pruning_history
        )

        print(
            "Superseded checkpoint removed:",
            old_path.name,
            "| MiB:",
            round(removed_bytes / 1024**2, 2)
        )

    # Keep only the validation scores belonging to the selected epoch

    for old_scores in run_prediction_dir.glob(
        'best_epoch_*_validation.npz'
    ):

        if old_scores != score_path:

            old_scores.unlink()


# Remove completed recovery checkpoints after model verification

def prune_completed_recovery(
    checkpoint_paths,
    run_log_dir,
    run_id,
    best_model_path
):

    retention_path = (
        run_log_dir / 'checkpoint_retention.json'
    )

    previous = (
        read_json(retention_path)
        if retention_path.exists()
        else {'removed_checkpoints': []}
    )

    removed_records = previous.get(
        'removed_checkpoints', []
    )

    unresolved = []

    for path in checkpoint_paths:

        path = Path(path)

        metadata_path = checkpoint_metadata_path(path)

        if not path.exists() and not metadata_path.exists():
            continue

        if not path.is_file() or not metadata_path.is_file():

            unresolved.append(str(path))
            continue

        metadata = read_json(metadata_path)

        if (
            metadata['run_id'] != run_id
            or path.stat().st_size != metadata['size_bytes']
            or calculate_sha256(path) != metadata['sha256']
        ):

            unresolved.append(str(path))
            continue

        record = {
            'path': str(path),
            'sha256': metadata['sha256'],
            'removed_bytes': metadata['size_bytes'],
            'status': 'Planned'
        }

        removed_records.append(record)

        save_json_atomic(
            retention_path,
            {
                'run_id': run_id,
                'removed_checkpoints': removed_records,
                'status': 'Pruning in progress'
            }
        )

        path.unlink()
        metadata_path.unlink()

        record['status'] = 'Completed'

        print(
            "Completed recovery checkpoint removed:",
            path.name,
            "| MiB:",
            round(record['removed_bytes'] / 1024**2, 2)
        )

    retention_summary = {
        'run_id': run_id,
        'selected_model_preserved': str(
            best_model_path.relative_to(PROJECT_ROOT)
        ),
        'removed_checkpoints': removed_records,
        'unresolved_checkpoint_paths': unresolved,
        'training_results_changed': False,
        'exact_optimizer_continuation_available': False
            if not unresolved else None,
        'status': 'Completed'
            if not unresolved else 'Inspection required'
    }

    save_json_atomic(
        retention_path,
        retention_summary
    )

    if unresolved:

        print(
            "Warning: Unverified recovery files retained.",
            "Manual inspection required."
        )

    return retention_summary


# Verify completed experiment artifacts

def verify_run_manifest(manifest_path, run_id):

    manifest = pd.read_csv(manifest_path)

    assert not manifest.empty

    assert manifest['Path'].is_unique

    assert manifest['Run ID'].eq(run_id).all()

    for _, artifact in manifest.iterrows():

        path = PROJECT_ROOT / artifact['Path']

        assert path.is_file(), f"Missing artifact: {path}"

        assert path.stat().st_size == int(
            artifact['Size Bytes']
        )

        assert calculate_sha256(path) == artifact['SHA256']

    return manifest


# Define the reusable RoBERTa experiment runner

def run_roberta_experiment(run_id):

    assert run_id in ROBERTA_RUN_IDS

    assert training_policy[
        'training_configuration_finalized'
    ]

    signature = get_run_signature(run_id)

    loss_name = LOSS_CONFIGS[run_id]['loss_name']

    run_dir = RESULTS_DIR / run_id

    run_table_dir = run_dir / 'tables'
    run_prediction_dir = run_dir / 'predictions'
    run_log_dir = run_dir / 'logs'

    run_model_dir = MODELS_DIR / run_id

    for folder in [
        run_table_dir,
        run_prediction_dir,
        run_log_dir,
        run_model_dir
    ]:

        folder.mkdir(
            parents=True,
            exist_ok=True
        )

    persistent_path = (
        run_model_dir / 'latest_checkpoint.pt'
    )

    local_path = (
        LOCAL_CHECKPOINT_DIR
        / run_id
        / 'latest_checkpoint.pt'
    )

    local_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    run_config_path = run_dir / 'config.json'

    run_summary_path = (
        run_log_dir / 'run_summary.json'
    )

    manifest_path = (
        run_table_dir / 'artifact_manifest.csv'
    )

    status_path = run_log_dir / 'run_status.json'

    current_registry = pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
        keep_default_na=False
    )

    assert current_registry['run_id'].is_unique

    existing_record = current_registry.loc[
        current_registry['run_id'].eq(run_id)
    ]

    # Reuse completed experiments rather than retraining

    if not existing_record.empty:

        assert len(existing_record) == 1

        assert existing_record.iloc[0]['status'] == (
            'Completed'
        )

        saved_config = read_json(run_config_path)

        assert saved_config['run_signature'] == signature

        manifest = verify_run_manifest(
            manifest_path,
            run_id
        )

        summary = read_json(run_summary_path)

        assert summary['run_id'] == run_id

        assert np.isclose(
            float(
                existing_record.iloc[0][
                    'validation_macro_ap'
                ]
            ),
            summary['validation_macro_ap'],
            rtol=0,
            atol=1e-12
        )

        prune_completed_recovery(
            [persistent_path, local_path],
            run_log_dir,
            run_id,
            PROJECT_ROOT / summary['best_model_path']
        )

        print(
            f"{run_id}: Already completed. "
            f"{len(manifest)} retained artifacts verified."
        )

        return summary

    registry_hash_at_start = calculate_sha256(
        REGISTRY_PATH
    )

    decision_hash_at_start = calculate_sha256(
        DECISION_LOG_PATH
    )

    # Restore or create the experiment configuration

    if run_config_path.exists():

        run_config = read_json(run_config_path)

        assert run_config['run_signature'] == signature

    else:

        run_config = {
            'run_id': run_id,
            'model': MODEL_NAME,
            'checkpoint': MODEL_CHECKPOINT,
            'revision': MODEL_REVISION,
            'loss': loss_name,
            'loss_configuration':
                LOSS_CONFIGS[run_id].copy(),
            'labels': LABELS,
            'seed': SEED,
            'sampling_strategy':
                'representative_multilabel_subset',
            'training_samples': TRAINING_SAMPLES,
            'validation_samples': VALIDATION_SAMPLES,
            'maximum_length': MAX_LENGTH,
            'training_policy': training_policy.copy(),
            'run_signature': signature,
            'initial_model_state_sha256': INITIAL_STATE_HASH,
            'training_dataset_sha256': EXPECTED_TRAINING_HASH,
            'validation_dataset_sha256': EXPECTED_VALIDATION_HASH,
            'evaluation_module_sha256': evaluation_hash,
            'start_time': datetime.now(
                timezone.utc
            ).isoformat(),
            'official_test_used': False,
            'training_completed': False
        }

        save_json_atomic(
            run_config_path,
            run_config
        )

    # Select a verified recovery state, if available

    recovery = find_recovery_checkpoint(
        [persistent_path, local_path],
        signature
    )

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    model, optimizer, scheduler, scaler = (
        create_training_components()
    )

    if recovery is None:

        assert not run_config['training_completed'], (
            "Completed configuration without a recovery "
            "checkpoint or registry entry. Inspect manually."
        )

        progress = new_training_progress()

        print(f"\nStarting new experiment: {run_id}")

    else:

        progress = load_resume_checkpoint(
            recovery['path'],
            model,
            optimizer,
            scheduler,
            scaler,
            signature
        )

        assert checkpoint_progress_key(progress) == (
            recovery['metadata']['progress_key']
        )

        if progress['best_epoch'] is not None:

            verify_selected_checkpoint(progress)

        print(f"\nResuming experiment: {run_id}")

        print("Recovery checkpoint:", recovery['path'])
        print("Epoch:", progress['epoch'])
        print("Next row:", progress['next_row'])
        print("Attempted updates:", progress['attempted_updates'])

    save_json_atomic(
        status_path,
        {
            'run_id': run_id,
            'status': 'Running',
            'epoch': progress['epoch'],
            'next_row': progress['next_row']
        }
    )

    # Define local recovery checkpoint callback

    def local_checkpoint_callback(
        current_model,
        current_optimizer,
        current_scheduler,
        current_scaler,
        current_progress
    ):

        persist_recovery_checkpoint(
            local_path,
            current_model,
            current_optimizer,
            current_scheduler,
            current_scaler,
            current_progress,
            signature
        )

        print(
            "Local recovery checkpoint saved | Update",
            current_progress['attempted_updates'],
            flush=True
        )

    # Train and validate one epoch at a time

    while (
        not progress['training_finished']
        and progress['epoch'] <= MAX_EPOCHS
    ):

        epoch = progress['epoch']

        print(
            f"\n{run_id} | Epoch {epoch}/{MAX_EPOCHS}",
            flush=True
        )

        training_result = train_one_epoch(
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            progress=progress,
            loss_name=loss_name,
            checkpoint_callback=local_checkpoint_callback
        )

        assert training_result['epoch_completed']

        assert progress['next_row'] == TRAINING_SAMPLES

        print(
            "\nRunning validation inference...",
            flush=True
        )

        validation_result = evaluate_validation_model(
            model,
            loss_name
        )

        metrics = validation_result['overall_metrics']

        macro_ap = float(metrics['Macro AP'])

        assert np.isfinite(macro_ap)

        improved = (
            progress['best_macro_ap'] is None
            or macro_ap > progress['best_macro_ap']
        )

        # Save an improved model and its validation scores

        if improved:

            best_path = save_best_model(
                model,
                run_model_dir,
                epoch
            )

            score_path = run_prediction_dir / (
                best_path.name + '_validation.npz'
            )

            temporary_score_path = (
                score_path.with_suffix('.tmp.npz')
            )

            np.savez_compressed(
                temporary_score_path,
                logits=validation_result['logits'],
                scores=validation_result['scores']
            )

            temporary_score_path.replace(score_path)

            progress.update({
                'best_macro_ap': macro_ap,
                'best_epoch': epoch,
                'best_checkpoint_path': str(
                    best_path.relative_to(PROJECT_ROOT)
                ),
                'best_validation_path': str(
                    score_path.relative_to(PROJECT_ROOT)
                ),
                'best_validation_sha256':
                    calculate_sha256(score_path),
                'epochs_without_improvement': 0
            })

            verify_selected_checkpoint(progress)

            print(
                f"New best checkpoint | "
                f"Macro AP: {macro_ap:.6f}"
            )

        else:

            progress['epochs_without_improvement'] += 1

        # Record the training and validation history

        epoch_record = {
            'Epoch': epoch,
            'Training Loss': float(
                training_result['training_loss']
            ),
            'Validation Loss': float(
                validation_result['validation_loss']
            ),
            'Validation Macro AP': macro_ap,
            'Validation Macro ROC-AUC': float(
                metrics['Macro ROC-AUC']
            ),
            'Validation Macro F1': float(
                metrics['Macro F1']
            ),
            'Validation Micro F1': float(
                metrics['Micro F1']
            ),
            'Validation Macro Precision': float(
                metrics['Macro Precision']
            ),
            'Validation Macro Recall': float(
                metrics['Macro Recall']
            ),
            'Learning Rate': float(
                optimizer.param_groups[0]['lr']
            ),
            'Cumulative Attempted Updates':
                progress['attempted_updates'],
            'Cumulative Successful Updates':
                progress['successful_updates'],
            'Cumulative Skipped Updates':
                progress['skipped_updates'],
            'Gradient Scale': float(
                scaler.get_scale()
            ),
            'Cumulative Training Time (s)':
                progress['training_time_seconds'],
            'Validation Processing Time (s)':
                validation_result['validation_time_seconds'],
            'Best Checkpoint Updated': improved,
            'Peak Allocated GPU Memory (GiB)':
                torch.cuda.max_memory_allocated() / 1024**3,
            'Peak Reserved GPU Memory (GiB)':
                torch.cuda.max_memory_reserved() / 1024**3
        }

        progress['history'].append(epoch_record)

        print(f"\nEpoch {epoch} results")

        for key in [
            'Training Loss',
            'Validation Loss',
            'Validation Macro AP',
            'Validation Macro F1',
            'Validation Micro F1'
        ]:

            print(
                f"{key}: {epoch_record[key]:.6f}"
            )

        # Apply the predefined stopping policy

        if epoch >= MAX_EPOCHS:

            progress['training_finished'] = True
            progress['stopping_reason'] = 'maximum_epochs'

        elif (
            progress['epochs_without_improvement']
            >= EARLY_STOPPING_PATIENCE
        ):

            progress['training_finished'] = True
            progress['stopping_reason'] = 'early_stopping'

        advance_epoch(progress)

        # Persist the completed epoch before removing older models

        persist_recovery_checkpoint(
            persistent_path,
            model,
            optimizer,
            scheduler,
            scaler,
            progress,
            signature
        )

        save_csv_atomic(
            run_table_dir / 'epoch_history.csv',
            pd.DataFrame(progress['history'])
        )

        prune_superseded_models(
            run_model_dir,
            run_prediction_dir,
            progress,
            persistent_path,
            run_log_dir
        )

        print(
            "Persistent epoch checkpoint verified | Update",
            progress['attempted_updates']
        )

        save_json_atomic(
            status_path,
            {
                'run_id': run_id,
                'status': 'Training finished'
                    if progress['training_finished']
                    else 'Running',
                'epoch': progress['epoch'],
                'next_row': progress['next_row'],
                'best_epoch': progress['best_epoch']
            }
        )

        del validation_result

    assert progress['training_finished']

    assert len(progress['history']) >= 1

    assert progress['best_epoch'] is not None

    assert (
        progress['attempted_updates']
        == progress['successful_updates']
        + progress['skipped_updates']
    )

    # Recover the selected model and saved best-epoch scores

    best_model_path, best_scores_path, best_manifest = (
        verify_selected_checkpoint(progress)
    )

    with np.load(
        best_scores_path,
        allow_pickle=False
    ) as saved_scores:

        best_logits = saved_scores['logits'].copy()
        best_scores = saved_scores['scores'].copy()

    assert best_logits.shape == (
        VALIDATION_SAMPLES, NUM_LABELS
    )

    assert best_scores.shape == best_logits.shape

    assert np.isfinite(best_logits).all()
    assert np.isfinite(best_scores).all()

    assert (
        (best_scores >= 0)
        & (best_scores <= 1)
    ).all()

    # Release training components before reloading the selected model

    del model, optimizer, scheduler, scaler

    gc.collect()
    torch.cuda.empty_cache()

    print(
        "\nReloading selected best checkpoint...",
        flush=True
    )

    with tempfile.TemporaryDirectory(
        prefix='roberta_selected_reload_',
        dir='/content'
    ) as reload_directory:

        reload_directory = Path(reload_directory)

        shutil.copytree(
            best_model_path,
            reload_directory,
            dirs_exist_ok=True
        )

        verify_model_files(
            reload_directory,
            best_manifest
        )

        best_model = (
            AutoModelForSequenceClassification
            .from_pretrained(
                reload_directory,
                local_files_only=True,
                dtype=torch.float32
            )
        )

        verify_model_configuration(best_model)

        assert model_state_sha256(best_model) == (
            best_manifest['model_state_sha256']
        )

        best_model = best_model.to(DEVICE)

        reloaded_validation = evaluate_validation_model(
            best_model,
            loss_name
        )

        del best_model

    gc.collect()
    torch.cuda.empty_cache()

    # Verify selected-model inference consistency

    maximum_logit_error = float(
        np.max(np.abs(
            best_logits
            - reloaded_validation['logits']
        ))
    )

    maximum_probability_error = float(
        np.max(np.abs(
            best_scores
            - reloaded_validation['scores']
        ))
    )

    reload_ap_difference = abs(
        float(
            reloaded_validation[
                'overall_metrics'
            ]['Macro AP']
        )
        - progress['best_macro_ap']
    )

    reload_verification = {
        'model_state_fingerprint_exact': True,
        'maximum_logit_difference': maximum_logit_error,
        'maximum_probability_difference':
            maximum_probability_error,
        'macro_ap_difference': reload_ap_difference,
        'logit_tolerance': 2e-3,
        'probability_tolerance': 5e-4
    }

    reload_verification_path = (
        run_log_dir / 'model_reload_verification.json'
    )

    save_json_atomic(
        reload_verification_path,
        reload_verification
    )

    print("\nSelected-model reload verification")

    print("Maximum logit difference:", maximum_logit_error)

    print(
        "Maximum probability difference:",
        maximum_probability_error
    )

    print("Macro AP difference:", reload_ap_difference)

    assert maximum_logit_error <= 2e-3, (
        "Reloaded logit difference requires investigation."
    )

    assert maximum_probability_error <= 5e-4, (
        "Reloaded probability difference requires investigation."
    )

    # Derive all final validation metrics from saved best-epoch scores

    default_metrics, default_per_label = (
        evaluate_multilabel(
            y_val,
            best_scores,
            LABELS,
            thresholds=0.5
        )
    )

    assert np.isclose(
        default_metrics['Macro AP'],
        progress['best_macro_ap'],
        rtol=0,
        atol=1e-12
    )

    threshold_df = optimize_thresholds(
        y_val,
        best_scores,
        LABELS
    ).set_index('Label').loc[LABELS].reset_index()

    optimized_thresholds = threshold_df[
        'Optimal Threshold'
    ].to_numpy(dtype=np.float64)

    assert optimized_thresholds.shape == (NUM_LABELS,)
    assert np.isfinite(optimized_thresholds).all()

    optimized_metrics, optimized_per_label = (
        evaluate_multilabel(
            y_val,
            best_scores,
            LABELS,
            thresholds=optimized_thresholds
        )
    )

    assert np.isclose(
        default_metrics['Macro AP'],
        optimized_metrics['Macro AP'],
        rtol=0,
        atol=1e-12
    )

    assert (
        optimized_metrics['Macro F1']
        >= default_metrics['Macro F1'] - 1e-12
    )

    # Prepare default and optimized prediction decisions

    default_predictions = apply_thresholds(
        best_scores,
        0.5
    )

    optimized_predictions = apply_thresholds(
        best_scores,
        optimized_thresholds
    )

    predictions_df = pd.DataFrame({
        'id': val_comment_ids,
        'split': 'validation',
        'run_id': run_id
    })

    for index, label in enumerate(LABELS):

        predictions_df[f'true_{label}'] = (
            y_val[:, index]
        )

        predictions_df[f'logit_{label}'] = (
            best_logits[:, index]
        )

        predictions_df[f'score_{label}'] = (
            best_scores[:, index]
        )

        predictions_df[f'pred_default_{label}'] = (
            default_predictions[:, index]
        )

        predictions_df[f'pred_optimized_{label}'] = (
            optimized_predictions[:, index]
        )

    assert predictions_df['id'].is_unique

    predictions_path = (
        run_prediction_dir / 'validation_predictions.parquet'
    )

    temporary_predictions_path = (
        predictions_path.with_suffix('.parquet.tmp')
    )

    predictions_df.to_parquet(
        temporary_predictions_path,
        index=False,
        compression='zstd'
    )

    temporary_predictions_path.replace(
        predictions_path
    )

    reloaded_predictions = pd.read_parquet(
        predictions_path
    )

    pd.testing.assert_frame_equal(
        predictions_df,
        reloaded_predictions,
        check_exact=True
    )

    # Save the selected validation thresholds

    threshold_path = run_dir / 'thresholds.json'

    threshold_configuration = {
        'run_id': run_id,
        'labels': LABELS,
        'score_type': 'sigmoid_probability',
        'default_threshold': 0.5,
        'optimized_thresholds': {
            label: float(threshold)
            for label, threshold in zip(
                LABELS, optimized_thresholds
            )
        },
        'selection_partition': 'validation',
        'optimization_metric': 'per_label_f1',
        'comparison_operator': '>=',
        'tie_breaking': 'highest_threshold',
        'probability_calibration_established': False,
        'official_test_used': False
    }

    save_json_atomic(
        threshold_path,
        threshold_configuration
    )

    # Prepare essential evaluation tables

    overall_comparison = pd.DataFrame([
        {
            'Metric': metric,
            'Default': float(default_metrics[metric]),
            'Optimized': float(optimized_metrics[metric]),
            'Change': float(
                optimized_metrics[metric]
                - default_metrics[metric]
            )
        }
        for metric in default_metrics
    ])

    per_label_comparison = (
        default_per_label[[
            'Label',
            'Threshold',
            'Precision',
            'Recall',
            'F1'
        ]]
        .merge(
            optimized_per_label[[
                'Label',
                'Threshold',
                'Precision',
                'Recall',
                'F1'
            ]],
            on='Label',
            suffixes=('_Default', '_Optimized')
        )
    )

    epoch_history_df = pd.DataFrame(
        progress['history']
    )

    result_tables = {
        'epoch_history.csv': epoch_history_df,
        'default_overall_metrics.csv':
            pd.DataFrame([default_metrics]),
        'optimized_overall_metrics.csv':
            pd.DataFrame([optimized_metrics]),
        'default_per_label_metrics.csv':
            default_per_label,
        'optimized_per_label_metrics.csv':
            optimized_per_label,
        'optimized_thresholds.csv':
            threshold_df,
        'overall_threshold_comparison.csv':
            overall_comparison,
        'per_label_threshold_comparison.csv':
            per_label_comparison
    }

    for filename, dataframe in result_tables.items():

        save_csv_atomic(
            run_table_dir / filename,
            dataframe
        )

    # Summarize the completed experiment

    model_weights_size_mib = sum(
        path.stat().st_size
        for path in best_model_path.glob('*.safetensors')
    ) / 1024**2

    run_summary = {
        'run_id': run_id,
        'model': MODEL_NAME,
        'loss': loss_name,
        'training_samples': TRAINING_SAMPLES,
        'validation_samples': VALIDATION_SAMPLES,
        'seed': SEED,
        'maximum_length': MAX_LENGTH,
        'epochs_completed': len(progress['history']),
        'best_epoch': progress['best_epoch'],
        'stopping_reason': progress['stopping_reason'],
        'attempted_updates': progress['attempted_updates'],
        'successful_updates': progress['successful_updates'],
        'skipped_updates': progress['skipped_updates'],
        'validation_macro_ap': float(
            optimized_metrics['Macro AP']
        ),
        'validation_macro_roc_auc': float(
            optimized_metrics['Macro ROC-AUC']
        ),
        'validation_default_macro_f1': float(
            default_metrics['Macro F1']
        ),
        'validation_optimized_macro_f1': float(
            optimized_metrics['Macro F1']
        ),
        'validation_default_micro_f1': float(
            default_metrics['Micro F1']
        ),
        'validation_optimized_micro_f1': float(
            optimized_metrics['Micro F1']
        ),
        'training_time_seconds': float(
            progress['training_time_seconds']
        ),
        'validation_prediction_time_seconds': float(
            reloaded_validation['validation_time_seconds']
        ),
        'validation_time_includes_metric_computation': True,
        'training_time_excludes_validation': True,
        'training_time_excludes_checkpoint_writing': True,
        'model_weights_size_mib': float(model_weights_size_mib),
        'maximum_logit_reload_error': maximum_logit_error,
        'maximum_probability_reload_error':
            maximum_probability_error,
        'reload_macro_ap_difference': float(
            reload_ap_difference
        ),
        'best_model_state_sha256':
            best_manifest['model_state_sha256'],
        'best_model_path': str(
            best_model_path.relative_to(PROJECT_ROOT)
        ),
        'threshold_path': str(
            threshold_path.relative_to(PROJECT_ROOT)
        ),
        'predictions_path': str(
            predictions_path.relative_to(PROJECT_ROOT)
        ),
        'predictions_sha256':
            calculate_sha256(predictions_path),
        'initial_model_state_sha256': INITIAL_STATE_HASH,
        'checkpoint_restoration_verified': True,
        'numerical_checkpoint_continuation_verified': True,
        'bitwise_reproducibility_established': False,
        'official_test_used': False,
        'final_project_model_frozen': False
    }

    save_json_atomic(
        run_summary_path,
        run_summary
    )

    run_summary_table_path = (
        run_table_dir / 'run_summary.csv'
    )

    save_csv_atomic(
        run_summary_table_path,
        pd.DataFrame([run_summary])
    )

    run_config.update({
        'best_epoch': progress['best_epoch'],
        'epochs_completed': len(progress['history']),
        'best_model_path': run_summary['best_model_path'],
        'threshold_path': run_summary['threshold_path'],
        'predictions_path': run_summary['predictions_path'],
        'validation_macro_ap':
            run_summary['validation_macro_ap'],
        'training_completed': True,
        'best_checkpoint_verified': True,
        'final_project_model_frozen': False
    })

    save_json_atomic(
        run_config_path,
        run_config
    )

    # Build the retained artifact manifest

    artifact_paths = [
        run_config_path,
        threshold_path,
        predictions_path,
        best_scores_path,
        run_summary_path,
        run_summary_table_path,
        reload_verification_path
    ]

    artifact_paths.extend([
        run_table_dir / filename
        for filename in result_tables
    ])

    artifact_paths.extend([
        path
        for path in best_model_path.rglob('*')
        if path.is_file()
    ])

    artifact_records = []

    for path in sorted(set(artifact_paths)):

        assert path.is_file()

        artifact_records.append({
            'Run ID': run_id,
            'Path': str(
                path.relative_to(PROJECT_ROOT)
            ),
            'Size Bytes': path.stat().st_size,
            'SHA256': calculate_sha256(path),
            'Verification': 'Passed'
        })

    save_csv_atomic(
        manifest_path,
        pd.DataFrame(artifact_records)
    )

    verified_manifest = verify_run_manifest(
        manifest_path,
        run_id
    )

    # Verify the unchanged research inputs before registration

    assert calculate_sha256(TRAIN_PATH) == (
        EXPECTED_TRAINING_HASH
    )

    assert calculate_sha256(VALIDATION_PATH) == (
        EXPECTED_VALIDATION_HASH
    )

    assert calculate_sha256(
        SRC_DIR / 'evaluation_utils.py'
    ) == evaluation_hash

    assert calculate_sha256(REGISTRY_PATH) == (
        registry_hash_at_start
    )

    assert calculate_sha256(DECISION_LOG_PATH) == (
        decision_hash_at_start
    )

    # Register the completed experiment

    registry_record = {
        'run_id': run_id,
        'notebook': '05_roberta',
        'model': MODEL_NAME,
        'feature_representation':
            'contextual_token_embeddings',
        'loss_function': loss_name,
        'sampling_strategy':
            'representative_multilabel_subset',
        'training_samples': TRAINING_SAMPLES,
        'seed': SEED,
        'max_length': MAX_LENGTH,
        'config_path': str(
            run_config_path.relative_to(PROJECT_ROOT)
        ),
        'model_path': run_summary['best_model_path'],
        'results_path': str(
            run_dir.relative_to(PROJECT_ROOT)
        ),
        'validation_macro_ap':
            run_summary['validation_macro_ap'],
        'validation_macro_f1':
            run_summary['validation_optimized_macro_f1'],
        'training_time_seconds':
            run_summary['training_time_seconds'],
        'status': 'Completed',
        'start_time': run_config['start_time'],
        'end_time': datetime.now(
            timezone.utc
        ).isoformat(),
        'notes': (
            'Validation-only selection; '
            'identical shared initialization; '
            'numerical recovery verified; '
            'official test not evaluated'
        )
    }

    assert set(registry_record) == set(
        current_registry.columns
    )

    assert not current_registry['run_id'].eq(
        run_id
    ).any()

    updated_registry = pd.concat([
        current_registry,
        pd.DataFrame([registry_record])[
            current_registry.columns
        ]
    ], ignore_index=True)

    assert updated_registry['run_id'].is_unique

    save_csv_atomic(
        REGISTRY_PATH,
        updated_registry
    )

    persisted_registry = pd.read_csv(
        REGISTRY_PATH,
        dtype=str,
        keep_default_na=False
    )

    assert len(persisted_registry) == (
        len(current_registry) + 1
    )

    assert persisted_registry['run_id'].eq(
        run_id
    ).sum() == 1

    save_json_atomic(
        status_path,
        {
            'run_id': run_id,
            'status': 'Completed',
            'best_epoch': progress['best_epoch'],
            'validation_macro_ap':
                run_summary['validation_macro_ap']
        }
    )

    # Remove unnecessary recovery states after verified completion

    prune_superseded_models(
        run_model_dir,
        run_prediction_dir,
        progress,
        persistent_path,
        run_log_dir
    )

    retention_summary = prune_completed_recovery(
        [persistent_path, local_path],
        run_log_dir,
        run_id,
        best_model_path
    )

    verify_run_manifest(
        manifest_path,
        run_id
    )

    # Display the final experimental evidence

    print("\nEpoch history")

    display(
        epoch_history_df.round(6)
    )

    print("\nDefault versus optimized thresholds")

    display(
        overall_comparison.round(6)
    )

    print("\nPer-label optimized metrics")

    display(
        optimized_per_label.round(6)
    )

    print("\nSelected thresholds")

    display(
        threshold_df.round(6)
    )

    print(f"\n{run_id} completion summary")

    print(
        "Epochs completed:",
        run_summary['epochs_completed']
    )

    print("Best epoch:", run_summary['best_epoch'])

    print(
        "Stopping reason:",
        run_summary['stopping_reason']
    )

    print(
        "Validation Macro AP:",
        round(
            run_summary['validation_macro_ap'], 6
        )
    )

    print(
        "Validation Macro ROC-AUC:",
        round(
            run_summary['validation_macro_roc_auc'], 6
        )
    )

    print(
        "Default Macro F1:",
        round(
            run_summary['validation_default_macro_f1'], 6
        )
    )

    print(
        "Optimized Macro F1:",
        round(
            run_summary['validation_optimized_macro_f1'], 6
        )
    )

    print(
        "Default Micro F1:",
        round(
            run_summary['validation_default_micro_f1'], 6
        )
    )

    print(
        "Optimized Micro F1:",
        round(
            run_summary['validation_optimized_micro_f1'], 6
        )
    )

    print(
        "\nTraining time (minutes):",
        round(
            run_summary['training_time_seconds'] / 60,
            2
        )
    )

    print(
        "Attempted updates:",
        run_summary['attempted_updates']
    )

    print(
        "Successful updates:",
        run_summary['successful_updates']
    )

    print(
        "Skipped updates:",
        run_summary['skipped_updates']
    )

    print(
        "\nRetained artifact verification:",
        len(verified_manifest)
    )

    print("Shared initialization: Preserved")
    print("Selected model and tokenizer: Saved")
    print("Validation predictions: Saved")
    print("Optimized thresholds: Saved")
    print("Epoch history: Saved")

    print(
        "Checkpoint retention:",
        retention_summary['status']
    )

    print("Experiment registry: Updated")

    print(
        "Total registered experiments:",
        len(persisted_registry)
    )

    print("Additional figures generated: 0")
    print("Official test dataset: Not loaded")
    print("Final project model frozen: No")

    gc.collect()
    torch.cuda.empty_cache()

    print(f"\n{run_id} completed successfully.")

    return run_summary


# Execute only the standard BCE experiment

print("\nStarting T04: RoBERTa with Standard BCE")

print(
    "Persistent checkpoints: Epoch boundaries"
)

print(
    "Local recovery checkpoints: Every 500 updates"
)

t04_summary = run_roberta_experiment(
    'T04_BCE_S42'
)


# Measure active project storage after training

project_size = sum(
    path.stat().st_size
    for path in PROJECT_ROOT.rglob('*')
    if path.is_file()
) / 1024**3

print(
    "\nCurrent project storage (GiB):",
    round(project_size, 4)
)

print("Cell 7 completed successfully.")

In [ ]:
# Verify the completed T04 experiment

assert 'run_roberta_experiment' in globals(), (
    "Execute Cell 7 to restore the experiment runner."
)

pipeline_configuration = read_json(
    PIPELINE_CONFIG_PATH
)

assert pipeline_configuration[
    'training_configuration_finalized'
]

assert pipeline_configuration[
    'checkpoint_resume_verified'
]

assert pipeline_configuration['run_signatures'] == {
    run_id: get_run_signature(run_id)
    for run_id in ROBERTA_RUN_IDS
}

current_registry = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
    keep_default_na=False
)

assert current_registry['run_id'].is_unique

t04_record = current_registry.loc[
    current_registry['run_id'].eq('T04_BCE_S42')
]

assert len(t04_record) == 1

assert t04_record.iloc[0]['status'] == 'Completed'

t04_dir = (
    PROJECT_ROOT
    / t04_record.iloc[0]['results_path']
)

t04_manifest = verify_run_manifest(
    t04_dir / 'tables/artifact_manifest.csv',
    'T04_BCE_S42'
)

t04_summary = read_json(
    t04_dir / 'logs/run_summary.json'
)

t04_retention = read_json(
    t04_dir / 'logs/checkpoint_retention.json'
)

assert t04_retention['status'] == 'Completed'

assert not t04_summary['official_test_used']

assert not (
    MODELS_DIR
    / 'T04_BCE_S42/latest_checkpoint.pt'
).exists()

print("T04: Completed and preserved.")

print(
    "T04 retained artifacts verified:",
    len(t04_manifest)
)


# Verify weighted BCE configuration

assert LOSS_CONFIGS['T05_WBCE_S42'][
    'loss_name'
] == 'WBCE'

assert LOSS_CONFIGS['T05_WBCE_S42'][
    'positive_weighting'
]

assert calculate_sha256(LOSS_CONFIG_PATH) == (
    LOSS_CONFIG_HASH
)

expected_weights = np.sqrt(
    train_negatives / train_positives
).astype(np.float32)

np.testing.assert_array_equal(
    POS_WEIGHT.detach().cpu().numpy(),
    expected_weights
)

print("\nWeighted BCE class weights")

display(pd.DataFrame({
    'Label': LABELS,
    'Positive Weight': expected_weights
}).round(6))


# Execute RoBERTa weighted BCE training

print("\nStarting T05: RoBERTa with Weighted BCE")

print("Shared initialization: Verified")
print("Training settings: Unchanged")
print("Class weights: Training-derived")
print("Local recovery checkpoints: Every 500 updates")
print("Persistent checkpoints: Epoch boundaries")

t05_summary = run_roberta_experiment(
    'T05_WBCE_S42'
)


# Verify completed experiment registration

updated_registry = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
    keep_default_na=False
)

assert updated_registry['run_id'].is_unique

for run_id in [
    'T04_BCE_S42',
    'T05_WBCE_S42'
]:

    record = updated_registry.loc[
        updated_registry['run_id'].eq(run_id)
    ]

    assert len(record) == 1
    assert record.iloc[0]['status'] == 'Completed'


# Compare standard BCE and weighted BCE

comparison_df = pd.DataFrame([
    {
        'Loss': summary['loss'],
        'Macro AP': summary['validation_macro_ap'],
        'Macro ROC-AUC':
            summary['validation_macro_roc_auc'],
        'Default Macro F1':
            summary['validation_default_macro_f1'],
        'Optimized Macro F1':
            summary['validation_optimized_macro_f1'],
        'Optimized Micro F1':
            summary['validation_optimized_micro_f1'],
        'Training Time (min)':
            summary['training_time_seconds'] / 60
    }
    for summary in [t04_summary, t05_summary]
])

print("\nRoBERTa BCE versus Weighted BCE")

display(comparison_df.round(6))

print(
    "\nMacro AP change (percentage points):",
    round(
        (
            t05_summary['validation_macro_ap']
            - t04_summary['validation_macro_ap']
        ) * 100,
        6
    )
)


# Measure active project storage

project_size = sum(
    path.stat().st_size
    for path in PROJECT_ROOT.rglob('*')
    if path.is_file()
) / 1024**3

print(
    "\nCurrent project storage (GiB):",
    round(project_size, 4)
)

print(
    "Total registered experiments:",
    len(updated_registry)
)

print("Additional comparison figures: 0")
print("Official test dataset: Not loaded")

print("\nCell 8 completed successfully.")

In [ ]:
import gc
import torch
import psutil
import os
import sys
import ctypes

def show_memory():
    ram = psutil.virtual_memory()

    print("Total RAM (GiB):", round(ram.total / 1024**3, 2))
    print("Available RAM (GiB):", round(ram.available / 1024**3, 2))
    print("RAM usage (%):", ram.percent)

    print(
        "Python process RAM (GiB):",
        round(psutil.Process(os.getpid()).memory_info().rss / 1024**3, 2)
    )

    if torch.cuda.is_available():
        print(
            "GPU allocated (GiB):",
            round(torch.cuda.memory_allocated() / 1024**3, 2)
        )
        print(
            "GPU reserved (GiB):",
            round(torch.cuda.memory_reserved() / 1024**3, 2)
        )

print("Before memory cleanup:\n")
show_memory()

# Release references to previous exceptions

for name in [
    'last_traceback',
    'last_value',
    'last_type',
    'last_exc'
]:
    if hasattr(sys, name):
        setattr(sys, name, None)

# Release unused memory

for _ in range(3):
    gc.collect()

if torch.cuda.is_available():
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

# Return unused CPU allocator memory to the operating system

try:
    ctypes.CDLL("libc.so.6").malloc_trim(0)
except Exception:
    pass

gc.collect()

print("\nAfter memory cleanup:\n")
show_memory()

# Verify the required training components remain available

for name in [
    'train_dataset',
    'val_dataset',
    'tokenizer',
    'POS_WEIGHT',
    'load_fresh_model',
    'train_one_epoch',
    'run_roberta_experiment'
]:
    assert name in globals(), f"Missing component: {name}"
    print(name, ": Available")

print("\nRequired training components preserved.")
print("No project files deleted.")
print("No training performed.")

In [ ]:
# Verify the finalized training configuration

assert 'run_roberta_experiment' in globals()

pipeline_configuration = read_json(
    PIPELINE_CONFIG_PATH
)

assert pipeline_configuration[
    'training_configuration_finalized'
]

assert pipeline_configuration[
    'checkpoint_resume_verified'
]

assert pipeline_configuration['run_signatures'] == {
    run_id: get_run_signature(run_id)
    for run_id in ROBERTA_RUN_IDS
}


# Verify the completed RoBERTa experiments

current_registry = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
    keep_default_na=False
)

assert current_registry['run_id'].is_unique

completed_summaries = []

for run_id in ['T04_BCE_S42', 'T05_WBCE_S42']:

    record = current_registry.loc[
        current_registry['run_id'].eq(run_id)
    ]

    assert len(record) == 1

    assert record.iloc[0]['status'] == 'Completed'

    run_dir = (
        PROJECT_ROOT / record.iloc[0]['results_path']
    )

    manifest = verify_run_manifest(
        run_dir / 'tables/artifact_manifest.csv',
        run_id
    )

    summary = read_json(
        run_dir / 'logs/run_summary.json'
    )

    retention = read_json(
        run_dir / 'logs/checkpoint_retention.json'
    )

    assert retention['status'] == 'Completed'
    assert not summary['official_test_used']

    assert summary['initial_model_state_sha256'] == (
        INITIAL_STATE_HASH
    )

    completed_summaries.append(summary)

    print(
        f"{run_id}: Completed | "
        f"{len(manifest)} artifacts verified."
    )

t04_summary, t05_summary = completed_summaries


# Verify the focal loss configuration

assert LOSS_CONFIGS['T06_FOCAL_S42'][
    'loss_name'
] == 'FOCAL'

assert LOSS_CONFIGS['T06_FOCAL_S42'][
    'focal_gamma'
] == 2.0

assert LOSS_CONFIGS['T06_FOCAL_S42'][
    'focal_alpha_factor'
] == 1.0

assert not LOSS_CONFIGS['T06_FOCAL_S42'][
    'positive_weighting'
]

assert calculate_sha256(LOSS_CONFIG_PATH) == (
    LOSS_CONFIG_HASH
)


# Execute the final RoBERTa experiment

print("\nStarting T06: RoBERTa with Focal Loss")

print("Focal gamma:", FOCAL_GAMMA)
print("Focal alpha factor:", FOCAL_ALPHA_FACTOR)
print("Shared initialization: Verified")
print("Training configuration: Unchanged")
print("Local checkpoints: Every 500 updates")
print("Persistent checkpoints: Epoch boundaries")

t06_summary = run_roberta_experiment(
    'T06_FOCAL_S42'
)


# Verify all three completed RoBERTa experiments

updated_registry = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
    keep_default_na=False
)

assert updated_registry['run_id'].is_unique

roberta_registry = updated_registry.loc[
    updated_registry['run_id'].isin(ROBERTA_RUN_IDS)
]

assert len(roberta_registry) == 3

assert roberta_registry['status'].eq(
    'Completed'
).all()


# Compare all RoBERTa loss configurations

roberta_comparison_df = pd.DataFrame([
    {
        'Run ID': summary['run_id'],
        'Loss': summary['loss'],
        'Macro AP':
            summary['validation_macro_ap'],
        'Macro ROC-AUC':
            summary['validation_macro_roc_auc'],
        'Default Macro F1':
            summary['validation_default_macro_f1'],
        'Optimized Macro F1':
            summary['validation_optimized_macro_f1'],
        'Optimized Micro F1':
            summary['validation_optimized_micro_f1'],
        'Training Time (min)':
            summary['training_time_seconds'] / 60
    }
    for summary in [
        t04_summary,
        t05_summary,
        t06_summary
    ]
]).sort_values(
    'Macro AP',
    ascending=False
).reset_index(drop=True)

print("\nRoBERTa validation comparison")

display(roberta_comparison_df.round(6))

print(
    "\nLeading RoBERTa configuration:",
    roberta_comparison_df.iloc[0]['Run ID']
)

print(
    "Highest RoBERTa Macro AP:",
    round(
        roberta_comparison_df.iloc[0]['Macro AP'],
        6
    )
)


# Measure active project storage

project_size = sum(
    path.stat().st_size
    for path in PROJECT_ROOT.rglob('*')
    if path.is_file()
) / 1024**3

print(
    "\nCurrent project storage (GiB):",
    round(project_size, 4)
)

print(
    "Total registered experiments:",
    len(updated_registry)
)

print("Completed RoBERTa experiments: 3/3")
print("Additional comparison figures: 0")
print("Official test dataset: Not loaded")
print("Final project model frozen: No")

print("\nCell 9 completed successfully.")

In [ ]:
# Verify the completed experiment registry

current_registry = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
    keep_default_na=False
)

assert current_registry['run_id'].is_unique
assert len(current_registry) == 17
assert current_registry['status'].eq('Completed').all()

registry_hash_before = calculate_sha256(REGISTRY_PATH)
decision_hash_before = calculate_sha256(DECISION_LOG_PATH)

TRANSFORMER_RUN_IDS = (
    DISTILBERT_RUN_IDS + ROBERTA_RUN_IDS
)

assert set(TRANSFORMER_RUN_IDS).issubset(
    current_registry['run_id']
)

print("Completed experiments:", len(current_registry))


# Reconstruct transformer metrics from saved validation predictions

transformer_records = []
per_label_records = []
verification_records = []

all_run_summaries = {}
all_run_configurations = {}
all_run_thresholds = {}
all_epoch_histories = {}

for run_id in TRANSFORMER_RUN_IDS:

    print(f"\nVerifying {run_id}...", flush=True)

    registry_row = current_registry.loc[
        current_registry['run_id'].eq(run_id)
    ].iloc[0]

    run_dir = PROJECT_ROOT / registry_row['results_path']

    summary = read_json(
        run_dir / 'logs/run_summary.json'
    )

    run_config = read_json(
        run_dir / 'config.json'
    )

    threshold_config = read_json(
        run_dir / 'thresholds.json'
    )

    manifest = verify_run_manifest(
        run_dir / 'tables/artifact_manifest.csv',
        run_id
    )

    assert len(manifest) == 20

    assert not summary['official_test_used']
    assert not run_config['official_test_used']
    assert not threshold_config['official_test_used']

    assert run_config['training_dataset_sha256'] == (
        EXPECTED_TRAINING_HASH
    )

    assert run_config['validation_dataset_sha256'] == (
        EXPECTED_VALIDATION_HASH
    )

    assert run_config['evaluation_module_sha256'] == (
        evaluation_hash
    )

    assert summary['training_samples'] == TRAINING_SAMPLES
    assert summary['validation_samples'] == VALIDATION_SAMPLES
    assert summary['maximum_length'] == MAX_LENGTH
    assert summary['seed'] == SEED

    assert threshold_config['labels'] == LABELS
    assert threshold_config['selection_partition'] == 'validation'
    assert threshold_config['comparison_operator'] == '>='

    predictions_path = (
        PROJECT_ROOT / summary['predictions_path']
    )

    assert calculate_sha256(predictions_path) == (
        summary['predictions_sha256']
    )

    predictions = pd.read_parquet(predictions_path)

    np.testing.assert_array_equal(
        predictions['id'].astype(str).to_numpy(),
        val_comment_ids
    )

    assert predictions['split'].eq('validation').all()
    assert predictions['run_id'].eq(run_id).all()

    targets = predictions[
        [f'true_{label}' for label in LABELS]
    ].to_numpy(dtype=np.int8)

    scores = predictions[
        [f'score_{label}' for label in LABELS]
    ].to_numpy(dtype=np.float64)

    np.testing.assert_array_equal(
        targets,
        y_val
    )

    assert np.isfinite(scores).all()
    assert ((scores >= 0) & (scores <= 1)).all()

    thresholds = np.array([
        threshold_config['optimized_thresholds'][label]
        for label in LABELS
    ], dtype=np.float64)

    reconstructed_thresholds = optimize_thresholds(
        targets,
        scores,
        LABELS
    ).set_index('Label').loc[
        LABELS, 'Optimal Threshold'
    ].to_numpy(dtype=np.float64)

    np.testing.assert_allclose(
        thresholds,
        reconstructed_thresholds,
        rtol=0,
        atol=1e-12
    )

    # Verify both saved sets of classification decisions

    saved_default = predictions[
        [f'pred_default_{label}' for label in LABELS]
    ].to_numpy()

    saved_optimized = predictions[
        [f'pred_optimized_{label}' for label in LABELS]
    ].to_numpy()

    np.testing.assert_array_equal(
        saved_default,
        apply_thresholds(scores, 0.5)
    )

    np.testing.assert_array_equal(
        saved_optimized,
        apply_thresholds(scores, thresholds)
    )

    # Independently reconstruct validation metrics

    default_metrics, default_per_label = (
        evaluate_multilabel(
            targets,
            scores,
            LABELS,
            thresholds=0.5
        )
    )

    optimized_metrics, optimized_per_label = (
        evaluate_multilabel(
            targets,
            scores,
            LABELS,
            thresholds=thresholds
        )
    )

    summary_checks = {
        'validation_macro_ap':
            optimized_metrics['Macro AP'],

        'validation_macro_roc_auc':
            optimized_metrics['Macro ROC-AUC'],

        'validation_default_macro_f1':
            default_metrics['Macro F1'],

        'validation_optimized_macro_f1':
            optimized_metrics['Macro F1'],

        'validation_default_micro_f1':
            default_metrics['Micro F1'],

        'validation_optimized_micro_f1':
            optimized_metrics['Micro F1']
    }

    for key, actual in summary_checks.items():

        assert np.isclose(
            float(summary[key]),
            float(actual),
            rtol=0,
            atol=1e-10
        ), f"Metric reconstruction failed: {run_id}, {key}"

    # Verify stored epoch selection

    history = pd.read_csv(
        run_dir / 'tables/epoch_history.csv'
    )

    best_index = history[
        'Validation Macro AP'
    ].idxmax()

    assert int(
        history.loc[best_index, 'Epoch']
    ) == summary['best_epoch']

    assert np.isclose(
        history.loc[best_index, 'Validation Macro AP'],
        optimized_metrics['Macro AP'],
        rtol=0,
        atol=1e-10
    )

    all_epoch_histories[run_id] = history.copy()

    # Verify retention status and selected model availability

    retention = read_json(
        run_dir / 'logs/checkpoint_retention.json'
    )

    best_model_path = (
        PROJECT_ROOT / summary['best_model_path']
    )

    assert best_model_path.is_dir()

    assert not (
        best_model_path.parent / 'latest_checkpoint.pt'
    ).exists()

    model_manifest = read_json(
        best_model_path / 'checkpoint_manifest.json'
    )

    assert model_manifest['model_state_sha256'] == (
        summary['best_model_state_sha256']
    )

    if run_id in ROBERTA_RUN_IDS:

        assert retention['status'] == 'Completed'

        assert summary['initial_model_state_sha256'] == (
            INITIAL_STATE_HASH
        )

        assert run_config['run_signature'] == (
            get_run_signature(run_id)
        )

        assert len([
            path
            for path in best_model_path.parent.glob(
                'best_epoch_*'
            )
            if path.is_dir()
        ]) == 1

    transformer_records.append({
        'Run ID': run_id,
        'Model': summary['model'],
        'Loss': summary['loss'],

        'Macro AP':
            optimized_metrics['Macro AP'],

        'Micro AP':
            optimized_metrics['Micro AP'],

        'Macro ROC-AUC':
            optimized_metrics['Macro ROC-AUC'],

        'Micro ROC-AUC':
            optimized_metrics['Micro ROC-AUC'],

        'Default Macro F1':
            default_metrics['Macro F1'],

        'Optimized Macro F1':
            optimized_metrics['Macro F1'],

        'Optimized Micro F1':
            optimized_metrics['Micro F1'],

        'Optimized Macro Precision':
            optimized_metrics['Macro Precision'],

        'Optimized Macro Recall':
            optimized_metrics['Macro Recall'],

        'Hamming Loss':
            optimized_metrics['Hamming Loss'],

        'Exact Match Accuracy':
            optimized_metrics['Exact Match Accuracy'],

        'Training Time (s)':
            summary['training_time_seconds'],

        'Training Time (min)':
            summary['training_time_seconds'] / 60,

        'Model Weights (MiB)':
            summary['model_weights_size_mib'],

        'Best Epoch': summary['best_epoch'],

        'Successful Updates':
            summary['successful_updates'],

        'Skipped Updates':
            summary['skipped_updates']
    })

    label_table = optimized_per_label.copy()

    label_table.insert(0, 'Loss', summary['loss'])
    label_table.insert(0, 'Model', summary['model'])
    label_table.insert(0, 'Run ID', run_id)

    per_label_records.append(label_table)

    verification_records.append({
        'Run ID': run_id,
        'Retained Artifacts Verified': len(manifest),
        'Validation Predictions Verified': True,
        'Thresholds Reconstructed': True,
        'Classification Decisions Verified': True,
        'Metrics Reconstructed': True,
        'Best Epoch Verified': True,
        'Selected Model Available': True,
        'Official Test Used': False,
        'Verification': 'Passed'
    })

    all_run_summaries[run_id] = summary
    all_run_configurations[run_id] = run_config
    all_run_thresholds[run_id] = threshold_config

    print(
        f"{run_id}: "
        f"{len(manifest)} artifacts verified | "
        "Metrics reconstructed successfully."
    )

    del predictions, targets, scores


# Verify controlled configurations within each architecture

for family_ids in [
    DISTILBERT_RUN_IDS,
    ROBERTA_RUN_IDS
]:

    signatures = []

    for run_id in family_ids:

        signature = all_run_configurations[
            run_id
        ]['run_signature']

        signatures.append({
            key: value
            for key, value in signature.items()
            if key not in [
                'run_id',
                'loss_configuration'
            ]
        })

    assert all(
        signature == signatures[0]
        for signature in signatures
    )

    assert len({
        all_run_summaries[run_id][
            'initial_model_state_sha256'
        ]
        for run_id in family_ids
    }) == 1

print("\nWithin-architecture controlled settings: Verified")


# Verify matched training settings across architectures

matched_architecture_runs = [
    ('T01_BCE_S42', 'T04_BCE_S42'),
    ('T02_WBCE_S42', 'T05_WBCE_S42'),
    ('T03_FOCAL_S42', 'T06_FOCAL_S42')
]

controlled_keys = [
    'loss_configuration',
    'training_dataset_sha256',
    'validation_dataset_sha256',
    'evaluation_module_sha256',
    'labels',
    'seed',
    'physical_batch_size',
    'validation_batch_size',
    'effective_batch_size',
    'gradient_accumulation_steps',
    'learning_rate',
    'weight_decay',
    'optimizer_betas',
    'optimizer_epsilon',
    'maximum_gradient_norm',
    'maximum_epochs',
    'early_stopping_patience',
    'warmup_steps',
    'total_training_steps',
    'fp16',
    'amp_initial_scale',
    'maximum_length',
    'padding_multiple'
]

for distilbert_run, roberta_run in matched_architecture_runs:

    distilbert_signature = all_run_configurations[
        distilbert_run
    ]['run_signature']

    roberta_signature = all_run_configurations[
        roberta_run
    ]['run_signature']

    for key in controlled_keys:

        assert distilbert_signature[key] == (
            roberta_signature[key]
        ), f"Controlled setting mismatch: {key}"

print("Matched architecture training settings: Verified")


# Prepare the transformer comparison tables

transformer_comparison_df = pd.DataFrame(
    transformer_records
).sort_values(
    'Macro AP',
    ascending=False
).reset_index(drop=True)

transformer_comparison_df.insert(
    0,
    'Rank',
    np.arange(1, len(transformer_comparison_df) + 1)
)

transformer_per_label_df = pd.concat(
    per_label_records,
    ignore_index=True
)

transformer_verification_df = pd.DataFrame(
    verification_records
)

assert len(transformer_comparison_df) == 6
assert len(transformer_per_label_df) == 36

assert transformer_verification_df[
    'Verification'
].eq('Passed').all()

assert transformer_verification_df[
    'Retained Artifacts Verified'
].sum() == 120

print("\nComplete transformer validation ranking")

display(
    transformer_comparison_df.round(6)
)


# Calculate within-architecture loss ablations

ablation_records = []

for family, baseline_id, comparison_ids in [
    (
        'DistilBERT',
        'T01_BCE_S42',
        ['T02_WBCE_S42', 'T03_FOCAL_S42']
    ),
    (
        'RoBERTa',
        'T04_BCE_S42',
        ['T05_WBCE_S42', 'T06_FOCAL_S42']
    )
]:

    baseline = transformer_comparison_df.loc[
        transformer_comparison_df['Run ID'].eq(
            baseline_id
        )
    ].iloc[0]

    for run_id in comparison_ids:

        candidate = transformer_comparison_df.loc[
            transformer_comparison_df['Run ID'].eq(
                run_id
            )
        ].iloc[0]

        ablation_records.append({
            'Comparison Type': 'Loss Ablation',
            'Model': family,
            'Baseline Run': baseline_id,
            'Candidate Run': run_id,

            'Macro AP Change':
                candidate['Macro AP']
                - baseline['Macro AP'],

            'Optimized Macro F1 Change':
                candidate['Optimized Macro F1']
                - baseline['Optimized Macro F1'],

            'Optimized Micro F1 Change':
                candidate['Optimized Micro F1']
                - baseline['Optimized Micro F1'],

            'Training Time Ratio':
                candidate['Training Time (s)']
                / baseline['Training Time (s)']
        })


# Calculate matched-loss architecture comparisons

for distilbert_run, roberta_run in matched_architecture_runs:

    baseline = transformer_comparison_df.loc[
        transformer_comparison_df['Run ID'].eq(
            distilbert_run
        )
    ].iloc[0]

    candidate = transformer_comparison_df.loc[
        transformer_comparison_df['Run ID'].eq(
            roberta_run
        )
    ].iloc[0]

    ablation_records.append({
        'Comparison Type': 'Architecture Comparison',
        'Model': 'RoBERTa versus DistilBERT',
        'Baseline Run': distilbert_run,
        'Candidate Run': roberta_run,

        'Macro AP Change':
            candidate['Macro AP'] - baseline['Macro AP'],

        'Optimized Macro F1 Change':
            candidate['Optimized Macro F1']
            - baseline['Optimized Macro F1'],

        'Optimized Micro F1 Change':
            candidate['Optimized Micro F1']
            - baseline['Optimized Micro F1'],

        'Training Time Ratio':
            candidate['Training Time (s)']
            / baseline['Training Time (s)']
    })

ablation_df = pd.DataFrame(ablation_records)

print("\nControlled loss and architecture comparisons")

display(ablation_df.round(6))


# Inspect rare-label classification

rare_label_df = transformer_per_label_df.loc[
    transformer_per_label_df['Label'].isin([
        'severe_toxic',
        'threat',
        'identity_hate'
    ]),
    [
        'Run ID',
        'Model',
        'Loss',
        'Label',
        'Average Precision',
        'Precision',
        'Recall',
        'F1',
        'TP',
        'FP',
        'FN'
    ]
].copy()

print("\nRare-label Average Precision")

display(
    rare_label_df.pivot(
        index='Run ID',
        columns='Label',
        values='Average Precision'
    ).round(6)
)

print("\nRare-label optimized F1")

display(
    rare_label_df.pivot(
        index='Run ID',
        columns='Label',
        values='F1'
    ).round(6)
)


# Rank all 17 registered configurations

overall_ranking_df = current_registry[[
    'run_id',
    'model',
    'loss_function',
    'validation_macro_ap',
    'validation_macro_f1',
    'training_time_seconds'
]].copy()

for column in [
    'validation_macro_ap',
    'validation_macro_f1',
    'training_time_seconds'
]:

    overall_ranking_df[column] = pd.to_numeric(
        overall_ranking_df[column],
        errors='raise'
    )

overall_ranking_df = overall_ranking_df.sort_values(
    'validation_macro_ap',
    ascending=False
).reset_index(drop=True)

overall_ranking_df.insert(
    0,
    'Rank',
    np.arange(1, len(overall_ranking_df) + 1)
)

print("\nOverall validation ranking")

display(overall_ranking_df.round(6))


# Select the leading RoBERTa configurations

roberta_comparison_df = (
    transformer_comparison_df.loc[
        transformer_comparison_df['Run ID'].isin(
            ROBERTA_RUN_IDS
        )
    ].sort_values(
        'Macro AP',
        ascending=False
    ).reset_index(drop=True)
)

selected_run_id = roberta_comparison_df.iloc[0]['Run ID']
runner_up_run_id = roberta_comparison_df.iloc[1]['Run ID']

best_f1_run_id = roberta_comparison_df.loc[
    roberta_comparison_df[
        'Optimized Macro F1'
    ].idxmax(),
    'Run ID'
]

selected_summary = all_run_summaries[selected_run_id]

selected_thresholds = all_run_thresholds[
    selected_run_id
]['optimized_thresholds']

macro_ap_gap = float(
    roberta_comparison_df.iloc[0]['Macro AP']
    - roberta_comparison_df.iloc[1]['Macro AP']
)

print("\nRoBERTa model selection")

print("Selected by Macro AP:", selected_run_id)
print("Runner-up:", runner_up_run_id)
print("Highest optimized Macro F1:", best_f1_run_id)
print("Macro AP gap:", round(macro_ap_gap, 6))


# Generate one consolidated comparison figure

fig, axes = plt.subplots(
    2, 2,
    figsize=(14, 10)
)

ordered_losses = ['BCE', 'WBCE', 'FOCAL']

for model_name, color in [
    ('DistilBERT', '#4C78A8'),
    ('RoBERTa', '#E45756')
]:

    subset = transformer_comparison_df.loc[
        transformer_comparison_df['Model'].eq(
            model_name
        )
    ].set_index('Loss').loc[ordered_losses]

    axes[0, 0].plot(
        ordered_losses,
        subset['Macro AP'],
        marker='o',
        linewidth=2,
        label=model_name,
        color=color
    )

    axes[0, 1].plot(
        ordered_losses,
        subset['Optimized Macro F1'],
        marker='o',
        linewidth=2,
        label=model_name,
        color=color
    )

    family_ids = (
        DISTILBERT_RUN_IDS
        if model_name == 'DistilBERT'
        else ROBERTA_RUN_IDS
    )

    for run_id in family_ids:

        history = all_epoch_histories[run_id]

        loss = all_run_summaries[run_id]['loss']

        axes[1, 0].plot(
            history['Epoch'],
            history['Validation Macro AP'],
            marker='o',
            linestyle='--'
                if model_name == 'DistilBERT'
                else '-',
            color=LOSS_COLORS[loss],
            label=f'{model_name} {loss}'
        )

    axes[1, 1].scatter(
        subset['Training Time (min)'],
        subset['Macro AP'],
        s=65,
        color=color,
        label=model_name
    )

    for loss, row in subset.iterrows():

        axes[1, 1].annotate(
            loss,
            (
                row['Training Time (min)'],
                row['Macro AP']
            ),
            xytext=(5, 5),
            textcoords='offset points',
            fontsize=8
        )

axes[0, 0].set_title('A. Loss Function Comparison')
axes[0, 0].set_ylabel('Validation Macro AP')
axes[0, 0].set_xlabel('Loss Function')

axes[0, 1].set_title('B. Threshold-Optimized Performance')
axes[0, 1].set_ylabel('Optimized Macro F1')
axes[0, 1].set_xlabel('Loss Function')

axes[1, 0].set_title('C. Validation Learning Curves')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Validation Macro AP')
axes[1, 0].set_xticks([1, 2, 3])

axes[1, 1].set_title('D. Performance versus Training Cost')
axes[1, 1].set_xlabel('Training Time (minutes)')
axes[1, 1].set_ylabel('Validation Macro AP')

for axis in axes.ravel():

    axis.grid(alpha=0.25)
    axis.legend(fontsize=8)

axes[0, 0].set_ylim(0.65, 0.77)
axes[0, 1].set_ylim(0.65, 0.74)
axes[1, 0].set_ylim(0.5, 0.77)
axes[1, 1].set_ylim(0.65, 0.77)

fig.suptitle(
    'Controlled Transformer Comparison on the Validation Set',
    fontsize=15,
    fontweight='bold'
)

plt.tight_layout(rect=[0, 0, 1, 0.96])

comparison_figure_path = (
    FIGURES_DIR / 'transformer_consolidated_comparison.png'
)

fig.savefig(
    comparison_figure_path,
    dpi=300,
    bbox_inches='tight',
    facecolor='white'
)

plt.show()
plt.close(fig)


# Save essential comparison tables

comparison_tables = {
    'transformer_validation_comparison.csv':
        transformer_comparison_df,

    'transformer_per_label_comparison.csv':
        transformer_per_label_df,

    'transformer_controlled_ablation.csv':
        ablation_df,

    'overall_validation_ranking.csv':
        overall_ranking_df
}

for filename, dataframe in comparison_tables.items():

    save_csv_atomic(
        TABLES_DIR / filename,
        dataframe
    )


# Save the RoBERTa model selection configuration

ROBERTA_SELECTION_PATH = (
    RUN_CONFIGS_DIR / 'roberta_final_selection.json'
)

roberta_selection = {
    'model_family': 'RoBERTa',
    'labels': LABELS,
    'configurations_compared': 3,
    'transformer_configurations_compared': 6,
    'total_registered_experiments': 17,

    'selection_partition': 'validation',

    'primary_selection_metric':
        'macro_average_precision',

    'secondary_selection_metric':
        'per_label_optimized_macro_f1',

    'selected_run_id': selected_run_id,

    'selected_imbalance_aware_run_id':
        selected_run_id,

    'runner_up_run_id': runner_up_run_id,

    'best_optimized_macro_f1_run_id':
        best_f1_run_id,

    'macro_ap_gap_to_runner_up':
        macro_ap_gap,

    'selected_summary':
        selected_summary,

    'optimized_thresholds':
        selected_thresholds,

    'training_dataset_sha256':
        EXPECTED_TRAINING_HASH,

    'validation_dataset_sha256':
        EXPECTED_VALIDATION_HASH,

    'evaluation_module_sha256':
        evaluation_hash,

    'common_training_signature': {
        key: value
        for key, value in get_run_signature(
            selected_run_id
        ).items()
        if key not in [
            'run_id',
            'loss_configuration'
        ]
    },

    'identical_initialization_verified': True,

    'within_architecture_controlled_settings_verified':
        True,

    'matched_architecture_training_settings_verified':
        True,

    'metrics_reconstructed_from_predictions': True,

    'retained_transformer_artifacts_verified': 120,

    'verification_records': verification_records,

    'checkpoint_recovery': {
        'exact_restoration_verified': True,
        'numerical_continuation_verified': True,
        'bitwise_reproducibility_established': False
    },

    'statistical_significance_established': False,

    'probability_calibration_established': False,

    'official_test_used': False,

    'final_project_model_frozen': False,

    'selection_status':
        'Representative-subset validation candidate',

    'next_stage':
        '06_model_optimization'
}

save_json_atomic(
    ROBERTA_SELECTION_PATH,
    roberta_selection
)

assert read_json(
    ROBERTA_SELECTION_PATH
) == roberta_selection


# Verify previous records and development datasets

assert calculate_sha256(REGISTRY_PATH) == (
    registry_hash_before
)

assert calculate_sha256(DECISION_LOG_PATH) == (
    decision_hash_before
)

assert calculate_sha256(TRAIN_PATH) == (
    EXPECTED_TRAINING_HASH
)

assert calculate_sha256(VALIDATION_PATH) == (
    EXPECTED_VALIDATION_HASH
)


# Verify the newly created comparison artifacts

comparison_artifacts = [
    TABLES_DIR / filename
    for filename in comparison_tables
] + [
    comparison_figure_path,
    ROBERTA_SELECTION_PATH
]

for path in comparison_artifacts:

    assert path.is_file()
    assert path.stat().st_size > 0


# Display final comparison summary

print("\nTransformer comparison summary")

print("Classical experiments completed: 11")
print("DistilBERT experiments completed: 3")
print("RoBERTa experiments completed: 3")
print("Total completed experiments: 17")

print("\nTransformer artifacts verified: 120")
print("Transformer metrics reconstructed: 6/6")
print("Per-label metric records:", len(transformer_per_label_df))
print("Controlled loss comparisons: 4")
print("Matched architecture comparisons: 3")

print("\nSelected RoBERTa candidate:", selected_run_id)

print(
    "Selected validation Macro AP:",
    round(selected_summary['validation_macro_ap'], 6)
)

print(
    "Selected optimized Macro F1:",
    round(
        selected_summary['validation_optimized_macro_f1'],
        6
    )
)

print("Runner-up:", runner_up_run_id)
print("Best optimized Macro F1:", best_f1_run_id)

print("\nComparison tables saved: 4")
print("Consolidated figures saved: 1")
print("Selection configurations saved: 1")

print("Experiment registry: Unchanged")
print("Decision log: Unchanged")
print("Additional model training performed: No")
print("Official test dataset: Not loaded")
print("Statistical superiority established: No")
print("Final project model frozen: No")

gc.collect()
torch.cuda.empty_cache()

print("\nCell 10 completed successfully.")

In [ ]:
# Verify all completed experiments

current_registry = pd.read_csv(
    REGISTRY_PATH,
    dtype=str,
    keep_default_na=False
)

assert len(current_registry) == 17
assert current_registry['run_id'].is_unique
assert current_registry['status'].eq('Completed').all()

registry_hash_before = calculate_sha256(REGISTRY_PATH)

assert set(ROBERTA_RUN_IDS).issubset(
    current_registry['run_id']
)

print("Experiment registry: 17 completed experiments verified.")


# Verify the finalized RoBERTa selection

ROBERTA_SELECTION_PATH = (
    RUN_CONFIGS_DIR / 'roberta_final_selection.json'
)

roberta_selection = read_json(
    ROBERTA_SELECTION_PATH
)

assert roberta_selection['model_family'] == 'RoBERTa'

assert roberta_selection['configurations_compared'] == 3

assert roberta_selection[
    'metrics_reconstructed_from_predictions'
]

assert roberta_selection[
    'identical_initialization_verified'
]

assert roberta_selection[
    'matched_architecture_training_settings_verified'
]

assert roberta_selection[
    'retained_transformer_artifacts_verified'
] == 120

assert roberta_selection['selection_partition'] == 'validation'

assert not roberta_selection['official_test_used']
assert not roberta_selection['final_project_model_frozen']

selected_run_id = roberta_selection['selected_run_id']

selected_summary = roberta_selection['selected_summary']

assert selected_run_id == 'T06_FOCAL_S42'

print("RoBERTa model selection: Verified.")


# Verify the original smoke test and recovery investigation

pipeline_configuration = read_json(
    PIPELINE_CONFIG_PATH
)

assert pipeline_configuration[
    'training_configuration_finalized'
]

assert pipeline_configuration[
    'checkpoint_resume_verified'
]

assert calculate_sha256(SMOKE_SUMMARY_PATH) == (
    pipeline_configuration['smoke_test_summary_sha256']
)

assert calculate_sha256(RECOVERY_RESOLUTION_PATH) == (
    pipeline_configuration['smoke_test_resolution_sha256']
)

original_smoke = read_json(SMOKE_SUMMARY_PATH)

recovery_resolution = read_json(
    RECOVERY_RESOLUTION_PATH
)

assert original_smoke['checkpoint_restore_passed']

assert not original_smoke[
    'checkpoint_continuation_passed'
]

assert recovery_resolution[
    'numerical_checkpoint_continuation_verified'
]

assert recovery_resolution[
    'original_failure_preserved'
]

assert not recovery_resolution[
    'bitwise_reproducibility_established'
]

print("Checkpoint recovery evidence: Verified and preserved.")


# Verify all retained RoBERTa artifacts

artifact_inventory = {}

def add_verified_artifact(path):

    path = Path(path)

    assert path.is_file(), f"Missing artifact: {path}"

    assert path.stat().st_size > 0

    relative_path = str(
        path.relative_to(PROJECT_ROOT)
    )

    artifact_inventory[relative_path] = {
        'path': relative_path,
        'size_bytes': path.stat().st_size,
        'sha256': calculate_sha256(path)
    }


retained_artifacts = 0
retention_records = []

for run_id in ROBERTA_RUN_IDS:

    registry_row = current_registry.loc[
        current_registry['run_id'].eq(run_id)
    ].iloc[0]

    run_dir = PROJECT_ROOT / registry_row['results_path']

    run_model_dir = MODELS_DIR / run_id

    manifest_path = (
        run_dir / 'tables/artifact_manifest.csv'
    )

    manifest = verify_run_manifest(
        manifest_path,
        run_id
    )

    assert len(manifest) == 20

    retained_artifacts += len(manifest)

    for _, artifact in manifest.iterrows():

        artifact_inventory[artifact['Path']] = {
            'path': artifact['Path'],
            'size_bytes': int(artifact['Size Bytes']),
            'sha256': artifact['SHA256']
        }

    summary = read_json(
        run_dir / 'logs/run_summary.json'
    )

    retention_path = (
        run_dir / 'logs/checkpoint_retention.json'
    )

    retention = read_json(retention_path)

    assert retention['status'] == 'Completed'

    assert retention[
        'exact_optimizer_continuation_available'
    ] is False

    assert summary['successful_updates'] == 5625
    assert summary['skipped_updates'] == 0
    assert summary['best_epoch'] == 3

    assert not summary['official_test_used']

    assert not (
        run_model_dir / 'latest_checkpoint.pt'
    ).exists()

    assert not (
        LOCAL_CHECKPOINT_DIR
        / run_id
        / 'latest_checkpoint.pt'
    ).exists()

    model_directories = [
        path for path in run_model_dir.glob('best_epoch_*')
        if path.is_dir()
    ]

    assert len(model_directories) == 1

    assert model_directories[0] == (
        PROJECT_ROOT / summary['best_model_path']
    )

    retention_records.append({
        'Run ID': run_id,
        'Retained Artifacts': len(manifest),
        'Selected Models': len(model_directories),
        'Successful Updates': summary['successful_updates'],
        'Skipped Updates': summary['skipped_updates'],
        'Recovery Checkpoints Removed': True,
        'Verification': 'Passed'
    })

    add_verified_artifact(manifest_path)
    add_verified_artifact(retention_path)

    pruning_path = (
        run_dir / 'logs/checkpoint_pruning_history.json'
    )

    if pruning_path.exists():
        add_verified_artifact(pruning_path)

    print(f"{run_id}: 20 retained artifacts verified.")

assert retained_artifacts == 60

print("\nFinal RoBERTa artifact verification")

display(pd.DataFrame(retention_records))


# Verify the consolidated comparison artifacts

comparison_paths = [
    TABLES_DIR / 'transformer_validation_comparison.csv',
    TABLES_DIR / 'transformer_per_label_comparison.csv',
    TABLES_DIR / 'transformer_controlled_ablation.csv',
    TABLES_DIR / 'overall_validation_ranking.csv',
    FIGURES_DIR / 'transformer_consolidated_comparison.png',
    ROBERTA_SELECTION_PATH
]

for path in comparison_paths:

    add_verified_artifact(path)

    print(path.name, ": Verified")


# Verify saved comparison table dimensions

saved_transformer_comparison = pd.read_csv(
    TABLES_DIR / 'transformer_validation_comparison.csv'
)

saved_per_label_comparison = pd.read_csv(
    TABLES_DIR / 'transformer_per_label_comparison.csv'
)

saved_ablation = pd.read_csv(
    TABLES_DIR / 'transformer_controlled_ablation.csv'
)

saved_overall_ranking = pd.read_csv(
    TABLES_DIR / 'overall_validation_ranking.csv'
)

assert len(saved_transformer_comparison) == 6
assert len(saved_per_label_comparison) == 36
assert len(saved_ablation) == 7
assert len(saved_overall_ranking) == 17

assert saved_overall_ranking.iloc[0]['run_id'] == (
    selected_run_id
)

assert saved_transformer_comparison.iloc[0]['Run ID'] == (
    selected_run_id
)

assert np.isclose(
    saved_transformer_comparison.iloc[0]['Macro AP'],
    selected_summary['validation_macro_ap'],
    rtol=0,
    atol=1e-10
)

print("Comparison tables: Verified.")


# Verify shared initialization and configuration artifacts

verify_model_files(
    INITIAL_MODEL_DIR,
    initial_manifest
)

for path in INITIAL_MODEL_DIR.iterdir():

    if path.is_file():
        add_verified_artifact(path)

for path in [
    INITIALIZATION_PATH,
    LOGS_DIR / 'roberta_tokenization_summary.json',
    LOSS_CONFIG_PATH,
    PIPELINE_CONFIG_PATH,
    SMOKE_SUMMARY_PATH,
    RECOVERY_RESOLUTION_PATH
]:

    add_verified_artifact(path)

print("Shared initialization and training evidence: Verified.")


# Verify unchanged development inputs

assert calculate_sha256(TRAIN_PATH) == (
    EXPECTED_TRAINING_HASH
)

assert calculate_sha256(VALIDATION_PATH) == (
    EXPECTED_VALIDATION_HASH
)

assert calculate_sha256(
    SRC_DIR / 'evaluation_utils.py'
) == evaluation_hash

assert calculate_sha256(REGISTRY_PATH) == (
    registry_hash_before
)


# Prepare new methodological decisions

new_decisions = [
    {
        'decision_id': 'D061',
        'component': 'RoBERTa Tokenization',
        'decision': (
            'Use the pinned RoBERTa tokenizer with '
            '256-token truncation and dynamic padding.'
        ),
        'justification': (
            'Preserves the original development partitions '
            'and matched sequence-length budget.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D062',
        'component': 'RoBERTa Loss Ablation',
        'decision': (
            'Compare standard BCE, square-root weighted BCE '
            'and focal loss with gamma 2.'
        ),
        'justification': (
            'Matches DistilBERT loss configurations and '
            'isolates loss-function effects within RoBERTa.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D063',
        'component': 'RoBERTa Training',
        'decision': (
            'Retain the matched transformer training budget '
            'and identical within-family initialization.'
        ),
        'justification': (
            'All three runs completed 5625 successful updates '
            'without AMP-skipped updates.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D064',
        'component': 'Checkpoint Reproducibility',
        'decision': (
            'Require exact saved-state restoration and '
            'numerically verified continuation.'
        ),
        'justification': (
            'Initial bitwise continuation comparison failed. '
            'A shared-prefix follow-up passed all documented '
            'numerical tolerances. Both results are preserved. '
            'Bitwise training reproducibility is not established.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D065',
        'component': 'Checkpoint Retention',
        'decision': (
            'Use local recovery checkpoints every 500 updates '
            'and persistent Drive checkpoints at epoch boundaries.'
        ),
        'justification': (
            'Reduces persistent storage and checkpoint-writing '
            'overhead. Local checkpoints can be lost on runtime reset. '
            'Superseded models and completed optimizer states '
            'were removed after verification.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D066',
        'component': 'Training Interruption',
        'decision': (
            'Retain the recovered T05 Weighted BCE experiment.'
        ),
        'justification': (
            'The observed notebook execution log reports recovery '
            'from local checkpoint update 4000, Epoch 3, row 8000. '
            'The run completed all scheduled updates and passed '
            'selected-model reload and artifact verification.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D067',
        'component': 'RoBERTa Model Selection',
        'decision': (
            'Select T06_FOCAL_S42 as the leading '
            'representative-subset RoBERTa candidate.'
        ),
        'justification': (
            'Highest validation Macro AP across the 17 completed '
            'configurations. Selection uses validation only. '
            'Statistical superiority is not established.'
        ),
        'status': 'Implemented'
    },
    {
        'decision_id': 'D068',
        'component': 'Secondary Model Candidate',
        'decision': (
            'Retain T05_WBCE_S42 and T03_FOCAL_S42 '
            'as relevant alternative candidates.'
        ),
        'justification': (
            'RoBERTa weighted BCE achieves the highest optimized '
            'Macro F1. DistilBERT focal offers a useful '
            'performance-efficiency trade-off.'
        ),
        'status': 'Implemented'
    }
]


# Append decisions without duplicating existing records

decision_columns = [
    'decision_id',
    'component',
    'decision',
    'justification',
    'status'
]

decisions = pd.read_csv(
    DECISION_LOG_PATH,
    dtype=str,
    keep_default_na=False
)

assert list(decisions.columns) == decision_columns
assert decisions['decision_id'].is_unique

assert set(
    f'D{index:03d}' for index in range(1, 61)
).issubset(decisions['decision_id'])

decisions_to_append = []

for decision in new_decisions:

    existing = decisions.loc[
        decisions['decision_id'].eq(
            decision['decision_id']
        )
    ]

    if existing.empty:

        decisions_to_append.append(decision)

    else:

        assert existing.iloc[0].to_dict() == decision, (
            f"Decision conflict: {decision['decision_id']}"
        )

if decisions_to_append:

    decisions = pd.concat([
        decisions,
        pd.DataFrame(decisions_to_append)[decision_columns]
    ], ignore_index=True)

    assert decisions['decision_id'].is_unique

    save_csv_atomic(
        DECISION_LOG_PATH,
        decisions
    )

final_decisions = pd.read_csv(
    DECISION_LOG_PATH,
    dtype=str,
    keep_default_na=False
)

assert final_decisions['decision_id'].is_unique

assert set(
    f'D{index:03d}' for index in range(1, 69)
).issubset(final_decisions['decision_id'])

print("\nNew decisions recorded:", len(decisions_to_append))
print("Total methodological decisions:", len(final_decisions))


# Write the notebook research summary

summary_path = RESULTS_DIR / 'summary.md'

summary_text = f"""# Notebook 05 - RoBERTa

## Objective

Evaluate RoBERTa-base under standard BCE, weighted BCE
and focal loss using the representative training subset
and unchanged validation partition.

## Experimental design

- Training comments: {TRAINING_SAMPLES}
- Validation comments: {VALIDATION_SAMPLES}
- Maximum sequence length: {MAX_LENGTH}
- Seed: {SEED}
- Effective batch size: {EFFECTIVE_BATCH_SIZE}
- Maximum epochs: {MAX_EPOCHS}
- Primary selection metric: Validation Macro AP
- Threshold optimization: Per-label validation F1

## Results

| Run | Macro AP | Optimized Macro F1 |
|---|---:|---:|
"""

for _, row in saved_transformer_comparison.iterrows():

    summary_text += (
        f"| {row['Run ID']} | "
        f"{row['Macro AP']:.6f} | "
        f"{row['Optimized Macro F1']:.6f} |\n"
    )

summary_text += f"""

## Selected representative-subset candidate

Selected run: {selected_run_id}

Validation Macro AP:
{selected_summary['validation_macro_ap']:.6f}

Optimized Macro F1:
{selected_summary['validation_optimized_macro_f1']:.6f}

Macro ROC-AUC:
{selected_summary['validation_macro_roc_auc']:.6f}

Training time:
{selected_summary['training_time_seconds'] / 60:.2f} minutes

RoBERTa focal loss achieved the highest validation Macro AP.
RoBERTa weighted BCE achieved the highest optimized Macro F1.
DistilBERT focal loss remains a computationally efficient
alternative.

Both imbalance-aware loss functions improved Macro AP over
standard BCE within each transformer architecture.

## Checkpoint recovery and reproducibility

Exact checkpoint restoration was verified.
The initial bitwise continuation test did not pass.
A shared-prefix follow-up established numerical continuation
within documented engineering tolerances.

Bitwise training reproducibility has not been established.

The T05 execution log reported successful recovery from
a local checkpoint at update 4000.

## Storage and integrity

All 60 retained RoBERTa run artifacts were verified.
Cell 10 verified 120 retained transformer artifacts and
reconstructed metrics for all six transformer experiments.

Superseded best models and completed optimizer checkpoints
were removed after verification.

Exact optimizer continuation of completed runs is no longer
available.

## Limitations

- Results are based on the representative training subset.
- Only one seed was used for each configuration.
- Statistical superiority has not been established.
- Validation-optimized F1 is not an independent test estimate.
- Probability calibration has not been established.
- Official test evaluation has not been performed.
- The final project model has not been frozen.

## Next stage

Notebook 06 - Model Optimization.

Review the remaining planned optimization experiments and
available resources before final model selection.
"""

temporary_summary_path = summary_path.with_suffix('.md.tmp')

temporary_summary_path.write_text(
    summary_text,
    encoding='utf-8'
)

temporary_summary_path.replace(summary_path)

add_verified_artifact(summary_path)


# Save the notebook completion record

project_size = sum(
    path.stat().st_size
    for path in PROJECT_ROOT.rglob('*')
    if path.is_file()
) / 1024**3

ROBERTA_COMPLETION_PATH = (
    LOGS_DIR / 'roberta_completion_summary.json'
)

completion_summary = {
    'notebook': '05_roberta',
    'status': 'Completed',
    'completed_at': datetime.now(timezone.utc).isoformat(),

    'classical_experiments': 11,
    'distilbert_experiments': 3,
    'roberta_experiments': 3,
    'total_registered_experiments': 17,

    'methodological_decisions': len(final_decisions),

    'selected_roberta_run': selected_run_id,

    'secondary_roberta_candidate':
        roberta_selection['runner_up_run_id'],

    'best_optimized_macro_f1_run':
        roberta_selection['best_optimized_macro_f1_run_id'],

    'selected_validation_macro_ap':
        selected_summary['validation_macro_ap'],

    'selected_optimized_macro_f1':
        selected_summary['validation_optimized_macro_f1'],

    'macro_ap_gap_to_runner_up':
        roberta_selection['macro_ap_gap_to_runner_up'],

    'controlled_experiment_settings_verified': True,

    'metrics_reconstructed_from_predictions': True,

    'retained_artifacts_verified': True,

    'retained_roberta_run_artifacts_verified':
        retained_artifacts,

    'retained_transformer_artifacts_verified_in_cell10':
        120,

    'exact_checkpoint_restoration_verified': True,

    'numerical_checkpoint_continuation_verified': True,

    'bitwise_reproducibility_established': False,

    'original_smoke_failure_preserved': True,

    't05_resume_event_reported_in_execution_log': True,

    'completed_optimizer_checkpoints_pruned': True,

    'statistical_significance_established': False,

    'probability_calibration_established': False,

    'additional_model_training_performed': False,

    'additional_figures_generated': 0,

    'official_test_used': False,

    'final_project_model_frozen': False,

    'active_project_storage_gib': project_size,

    'next_notebook': '06_model_optimization',

    'artifact_inventory': [
        artifact_inventory[key]
        for key in sorted(artifact_inventory)
    ]
}

save_json_atomic(
    ROBERTA_COMPLETION_PATH,
    completion_summary
)

assert read_json(
    ROBERTA_COMPLETION_PATH
) == completion_summary


# Verify the registry remains unchanged

assert calculate_sha256(REGISTRY_PATH) == (
    registry_hash_before
)

assert calculate_sha256(TRAIN_PATH) == (
    EXPECTED_TRAINING_HASH
)

assert calculate_sha256(VALIDATION_PATH) == (
    EXPECTED_VALIDATION_HASH
)


# Display the final notebook completion summary

print("\nRoBERTa notebook completion summary")

print("Classical ML experiments: 11")
print("DistilBERT experiments: 3")
print("RoBERTa experiments: 3")
print("Total completed experiments: 17")

print("\nMethodological decisions:", len(final_decisions))

print("RoBERTa retained run artifacts verified:", retained_artifacts)

print("Transformer metrics reconstructed: 6/6")

print("Transformer artifacts verified in Cell 10: 120")

print(
    "Completion inventory artifacts:",
    len(artifact_inventory)
)

print("\nSelected RoBERTa candidate:", selected_run_id)

print(
    "Validation Macro AP:",
    round(selected_summary['validation_macro_ap'], 6)
)

print(
    "Optimized Macro F1:",
    round(selected_summary['validation_optimized_macro_f1'], 6)
)

print(
    "Macro ROC-AUC:",
    round(selected_summary['validation_macro_roc_auc'], 6)
)

print("\nOriginal recovery investigation: Preserved")
print("Completed optimizer checkpoints: Removed")
print("Shared initialization: Preserved")
print("Comparison artifacts: Verified")
print("Decision log: Updated")
print("Notebook summary: Saved")
print("Completion record: Saved")
print("Experiment registry: Unchanged")

print(
    "\nActive project storage (GiB):",
    round(project_size, 4)
)

print("\nAdditional model training performed: No")
print("Additional figures generated: 0")
print("Official test dataset: Not loaded")
print("Final project model frozen: No")

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nNotebook 05 - RoBERTa completed successfully.")
print("Next notebook: 06_model_optimization")